# KVO_06 v1.1.1 — U-Net-guided SAM3 Musanga crown extraction on Baego

**Notebook version: `KVO_06 v1.1.1`**

**Primary observational basis:** six exhaustively digitized crown-validation areas in `ROI_Crowns_Kvo.shp`, with **all visible crowns** digitized in `Crown_Kristof.shp`.

The notebook now separates two tasks:

1. **Observational validation:** within the six crown ROIs, use *all* U-Net candidates passing the fixed probability/coherence criteria, run SAM3, and compare automatic objects against the exhaustive visible-crown reference.
2. **Baego production pilot:** outside this validation logic, the existing ranked top-500 workflow remains available as an optional production-scale pilot.

This makes the six ROIs the observational basis for evaluating the fully automatic **U-Net detection → SAM3 crown delineation** workflow. KVO_03 remains the independent species-classification validation and KVO_05 the prompt-conditioned SAM3 boundary benchmark.


# Table of contents

Colab's **Table of contents** sidebar also lists these Markdown headers automatically. Select any heading there to jump to its block.

- [0R — Resume a new Colab session](#0r-resume-a-new-colab-session)
- [0S — Restore runner without running original SAM3 inference](#0s-restore-runner-without-running-original-sam3-inference)
- [0 — Host environment and configuration](#0-host-environment-and-configuration)
- [1 — Extract coherent U-Net candidates](#1-extract-coherent-u-net-candidates)
- [2 — Isolated official SAM3 environment](#2-isolated-official-sam3-environment)
- [3 — Hugging Face authentication](#3-hugging-face-authentication)
- [4 — SAM3 delineation of U-Net candidates](#4-sam3-delineation-of-u-net-candidates)
- [5 — Convert SAM3 masks to georeferenced candidate crown polygons](#5-convert-sam3-masks-to-georeferenced-candidate-crown-polygons)
- [6 — Add object-level U-Net probability summaries](#6-add-object-level-u-net-probability-summaries)
- [7 — Primary observational validation against Crown_Kristof](#7-primary-observational-validation-against-crown_kristof)
- [8 — Diagnostics and production summary](#8-diagnostics-and-production-summary)
- [9 — Interpretation / next step](#9-interpretation-next-step)
- [10 — Experiment A: adaptive U-Net-guided SAM3 prompts](#10-experiment-a-adaptive-u-net-guided-sam3-prompts)
- [11 — Adaptive crown agreement and reproducibility](#11-adaptive-crown-agreement-and-reproducibility)
- [12 — Experiment B: species-independent RGB crown candidates](#12-experiment-b-species-independent-rgb-crown-candidates)
- [13 — Generic RGB-guided SAM3 and U-Net attribution](#13-generic-rgb-guided-sam3-and-u-net-attribution)
- [14 — Revised all-crown reference and Baego DSM audit](#14-revised-all-crown-reference-and-baego-dsm-audit)
- [15 — Multi-scale DSM candidate detection and ranked RGB fusion](#15-multi-scale-dsm-candidate-detection-and-ranked-rgb-fusion)
- [16 — Sensitivity and crown-size-specific coverage](#16-sensitivity-and-crown-size-specific-coverage)
- [17–19 — New RGB/DSM and SAM3 crown-discovery experiments](#1719-new-rgbdsm-and-sam3-crown-discovery-experiments)
- [20 — Multiscale SAM3 pilot (40 m RGB tiles)](#20-multiscale-sam3-pilot-40-m-rgb-tiles)
- [20B — Actual multiscale RGB/SAM3 visual QA](#20b-actual-multiscale-rgbsam3-visual-qa)
- [21 — DSM-assisted SAM3 crown quality assessment](#21-dsm-assisted-sam3-crown-quality-assessment)
- [22 — Full ROI RGB-only 8 m SAM3 and full manual-outline validation](#22-full-roi-rgb-only-8-m-sam3-and-full-manual-outline-validation)
- [22V — Statistical validation (no GPU)](#22v-statistical-validation-no-gpu)
- [23 — Statistical proposal-selection benchmark (CPU only)](#23-statistical-proposal-selection-benchmark-cpu-only)
- [24 — GIS topology and crown geometry optimization (CPU only)](#24-gis-topology-and-crown-geometry-optimization-cpu-only)
- [25 — Learn to rank SAM3 crown proposals (CPU only)](#25-learn-to-rank-sam3-crown-proposals-cpu-only)
- [26 — Proposal-budget curves and overlap-aware selection (CPU only)](#26-proposal-budget-curves-and-overlap-aware-selection-cpu-only)
- [27 — Stricter one-to-one training labels and model comparison (CPU only)](#27-stricter-one-to-one-training-labels-and-model-comparison-cpu-only)
- [28 — Multiregion SAM3 crown discovery (same zero-shot RGB configuration)](#28-multiregion-sam3-crown-discovery-same-zero-shot-rgb-configuration)
- [29 — Six-ROI crown proposal classifiers: RGB versus RGB + DSM (CPU only)](#29-six-roi-crown-proposal-classifiers-rgb-versus-rgb--dsm-cpu-only)
- [30 — SAM3 crown fine-tuning pilot: export masks and verify official training setup](#30-sam3-crown-fine-tuning-pilot-export-masks-and-verify-official-training-setup)
- [30B — SAM3 fine-tuning smoke test](#30b-sam3-fine-tuning-smoke-test-coco-qa-and-short-gpu-run)


## 0R — Resume a new Colab session

Run **0R**, then **Block 2 only if the SAM3 environment is missing**, then **0S**, then **Blocks 10–13**. Skip Blocks 1 and 3–9. Saved Drive masks are preserved.


In [ ]:
# BLOCK 0R — Fresh runtime resume (no original inference)
import os,sys,subprocess,shutil,warnings
from pathlib import Path
import numpy as np,pandas as pd,geopandas as gpd,rasterio
from rasterio.windows import Window
from rasterio.features import shapes
from shapely.geometry import Point,shape
from PIL import Image
from tqdm.auto import tqdm
from google.colab import drive,userdata
ROOT=Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
if not ROOT.exists():drive.mount("/content/drive")
GEO=ROOT/"data"/"Geo_Ref";AD=GEO/"Annotations"
OUT=ROOT/"KVO_work"/"KVO_06_UNet_guided_SAM3";RAW=OUT/"sam3_raw"
RASTER=GEO/"Baego_georef.tif";PROB=ROOT/"KVO_work"/"KVO_04_final_multisite"/"probability_maps"/"Baego_musanga_probability_uint16.tif"
CROWN_REF=AD/"Crown_Kristof.shp";CROWN_ROI=AD/"ROI_Crowns_Kvo.shp"
CHIP_SIZE_M=24.;MIN_PRED_AREA_M2=2.;MAX_PRED_AREA_M2=500.
manifest=pd.read_csv(OUT/"candidate_manifest.csv")
roi=gpd.read_file(CROWN_ROI)
if roi.crs is None:raise ValueError("ROI missing CRS")
with rasterio.open(RASTER) as src:
    if roi.crs!=src.crs:roi=roi.to_crs(src.crs)
roi_union=roi.geometry.union_all()
SAMPY="/content/sam3env/bin/python";RUNNER=Path("/content/run_sam3_kvo06.py")
env=os.environ.copy();token=userdata.get("HF_TOKEN")
if token:env["HF_TOKEN"]=token
print("Saved candidates:",len(manifest),"existing SAM3 masks:",len(list(RAW.glob("cand_*.npz"))))
print("SAM3 env:",Path(SAMPY).exists())


## 0S — Restore runner without running original SAM3 inference

Recreates only the Python runner script that was previously embedded in Block 4.


In [ ]:
# BLOCK 0S — Runner definition only
RUNNER=Path("/content/run_sam3_kvo06.py")
RUNNER.write_text(r'''import sys
from pathlib import Path
import numpy as np,pandas as pd,torch
from PIL import Image
from tqdm import tqdm
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

manifest=Path(sys.argv[1]); outdir=Path(sys.argv[2]); box_m=float(sys.argv[3]); force=sys.argv[4]=="1"
outdir.mkdir(parents=True,exist_ok=True); df=pd.read_csv(manifest)
if not torch.cuda.is_available(): raise RuntimeError("KVO_06 requires CUDA")
torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True
torch.autocast("cuda",dtype=torch.bfloat16).__enter__(); torch.inference_mode().__enter__()
bpe="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz"
model=build_sam3_image_model(bpe_path=bpe); processor=Sam3Processor(model)
print("SAM3 ready; candidates=",len(df),flush=True)
for r in tqdm(df.itertuples(),total=len(df),desc="DELINEATE"):
    op=outdir/f"cand_{int(r.candidate_id):05d}.npz"
    if op.exists() and not force: continue
    img=Image.open(r.path).convert("RGB")
    state=processor.set_image(img)
    # Normalized centre-x, centre-y, width, height; candidate is chip centre.
    frac=min(0.95,box_m/(r.width*r.gsd))
    box=np.array([[0.5,0.5,frac,frac]],dtype=np.float32)
    out=processor.add_geometric_prompt(box=box,label=np.array([1],dtype=np.int32),state=state)
    masks=out["masks"].detach().cpu().numpy().astype(np.uint8)
    if masks.ndim==4: masks=masks[:,0]
    scores=out["scores"].detach().float().cpu().numpy()
    np.savez_compressed(op,masks=masks,scores=scores)
print("DONE",flush=True)
''')

print('SAM3 runner restored:',RUNNER)


## 0 — Host environment and configuration

Use a **GPU runtime**. KVO_06 reuses the same isolated official SAM3 setup as KVO_05.


In [ ]:
import sys, subprocess, os, json, random, warnings, gc, shutil, math
from pathlib import Path
print("Host Python:",sys.version)
subprocess.run(["nvidia-smi"],check=False)
!pip -q install -U "numpy>=2,<2.6" "huggingface_hub<2" geopandas rasterio pyogrio shapely scikit-learn matplotlib tqdm pillow


In [ ]:
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
import rasterio
from rasterio.windows import Window
from rasterio.features import shapes
from shapely.geometry import shape, Point
from tqdm.auto import tqdm
from PIL import Image
from google.colab import drive
warnings.filterwarnings("ignore")

SEED=42
random.seed(SEED); np.random.seed(SEED)
DRIVE_ROOT=Path("/content/drive"); MYDRIVE=DRIVE_ROOT/"MyDrive"
def healthy():
    try: return MYDRIVE.exists() and any(MYDRIVE.iterdir())
    except OSError: return False
if not healthy(): drive.mount(str(DRIVE_ROOT))

ROOT=MYDRIVE/"Colab Notebooks"/"Musanga"; GEO=ROOT/"data"/"Geo_Ref"; AD=GEO/"Annotations"
OUT=ROOT/"KVO_work"/"KVO_06_UNet_guided_SAM3"
CHIPS=OUT/"chips"; RAW=OUT/"sam3_raw"; VIS=OUT/"examples"
for p in [OUT,CHIPS,RAW,VIS]: p.mkdir(parents=True,exist_ok=True)

RASTER=GEO/"Baego_georef.tif"
PROB=ROOT/"KVO_work"/"KVO_04_final_multisite"/"probability_maps"/"Baego_musanga_probability_uint16.tif"
CROWN_REF=AD/"Crown_Kristof.shp"
CROWN_ROI=AD/"ROI_Crowns_Kvo.shp"
assert RASTER.exists(),RASTER
assert PROB.exists(),PROB
assert CROWN_REF.exists(),CROWN_REF
assert CROWN_ROI.exists(),CROWN_ROI

# Candidate extraction. This is intentionally conservative and computationally cheap.
PROB_THRESHOLD=0.90
SCAN_WINDOW_PX=2048
PEAK_MIN_DISTANCE_M=10.0
CONTEXT_RADIUS_M=4.0       # score support around each peak
MIN_CONTEXT_MEAN=0.60      # reject isolated hot pixels
VALIDATION_MODE=True        # primary KVO_06 observational experiment
MAX_CANDIDATES=500         # used only if VALIDATION_MODE=False
CHIP_SIZE_M=24.0          # local RGB context around each U-Net candidate
BOX_SIZE_M=10.0           # SAM3 geometric prompt centered on probability peak
MIN_PRED_AREA_M2=2.0
MAX_PRED_AREA_M2=500.0
FORCE_CHIPS=False
FORCE_SAM=False
print("OUT:",OUT)


## 1 — Extract coherent U-Net candidates

Candidate detection: probability ≥0.90, local-context support, then 10 m spatial NMS.

**Validation mode (default):** retain every qualifying candidate whose point lies within the six polygons of `ROI_Crowns_Kvo.shp`. There is no top-N confidence cap inside the observational areas. RGB chips and SAM3 are generated only for those candidates.

**v0.8:** the complete Block 1 is resume-safe: candidate tuples are converted to the downstream dataframe/manifest format within the same block, and the ROI shapefile is read with SHX restoration enabled for robust Drive-mounted shapefiles.


In [ ]:
# v0.8 — coherent candidate extraction + ROI-filtered chip/manifest preparation.
# Pass 1: large-window scan; retain the strongest pixel per 10-m metric grid cell.
with rasterio.open(PROB) as ps:
    pcrs=ps.crs; ptr=ps.transform
    gsd=float((abs(ptr.a)+abs(ptr.e))/2)
    grid_m=float(PEAK_MIN_DISTANCE_M)
    nx=math.ceil(ps.width/SCAN_WINDOW_PX); ny=math.ceil(ps.height/SCAN_WINDOW_PX)
    nwin=nx*ny; best={}
    thr=int(round(PROB_THRESHOLD*10000))
    for r0 in tqdm(range(0,ps.height,SCAN_WINDOW_PX),total=ny,desc="U-NET SCAN"):
        for c0 in range(0,ps.width,SCAN_WINDOW_PX):
            hh=min(SCAN_WINDOW_PX,ps.height-r0); ww=min(SCAN_WINDOW_PX,ps.width-c0)
            a=ps.read(1,window=Window(c0,r0,ww,hh))
            rr,cc=np.where((a!=65535) & (a>=thr))
            if len(rr)==0: continue
            vals=a[rr,cc].astype(np.float32)/10000.0
            gr=rr+r0; gc=cc+c0
            xs,ys=rasterio.transform.xy(ptr,gr,gc,offset="center")
            xs=np.asarray(xs); ys=np.asarray(ys)
            gx=np.floor(xs/grid_m).astype(np.int64); gy=np.floor(ys/grid_m).astype(np.int64)
            order=np.argsort(vals)[::-1]; seen=set()
            for j in order:
                key=(int(gx[j]),int(gy[j]))
                if key in seen: continue
                seen.add(key)
                rec=(float(vals[j]),int(gr[j]),int(gc[j]),float(xs[j]),float(ys[j]))
                old=best.get(key)
                if old is None or rec[0]>old[0]: best[key]=rec

    print("Large raster windows scanned:",nwin)
    print("Strong coarse candidate cells:",len(best))

    # Pass 2: measure local support around each strong peak directly from probability map.
    rad=max(1,int(round(CONTEXT_RADIUS_M/gsd)))
    coherent=[]
    for peak,row,col,x,y in tqdm(best.values(),desc="CONTEXT SCORE"):
        w=Window(col-rad,row-rad,2*rad+1,2*rad+1)
        a=ps.read(1,window=w,boundless=True,fill_value=65535)
        v=a[a!=65535].astype(np.float32)/10000.0
        context_mean=float(v.mean()) if len(v) else 0.0
        context_high=float((v>=PROB_THRESHOLD).mean()) if len(v) else 0.0
        if context_mean < MIN_CONTEXT_MEAN: continue
        # Ranking rewards both a strong peak and broad local support.
        score=0.40*peak + 0.45*context_mean + 0.15*context_high
        coherent.append((score,peak,context_mean,context_high,row,col,x,y))

print("Coherent candidates before spatial NMS:",len(coherent))

# Pass 3: exact 10-m NMS using a spatial hash, highest coherent score first.
coherent.sort(key=lambda z:z[0],reverse=True)
accepted=[]; occupied={}
for rec in tqdm(coherent,desc="CANDIDATE NMS"):
    score,peak,cmean,chigh,row,col,x,y=rec
    kx=int(math.floor(x/grid_m)); ky=int(math.floor(y/grid_m)); ok=True
    for dx in (-1,0,1):
        for dy in (-1,0,1):
            for qx,qy in occupied.get((kx+dx,ky+dy),[]):
                if (x-qx)**2+(y-qy)**2 < PEAK_MIN_DISTANCE_M**2:
                    ok=False; break
            if not ok: break
        if not ok: break
    if ok:
        accepted.append(rec)
        occupied.setdefault((kx,ky),[]).append((x,y))

cand=pd.DataFrame(accepted,columns=["candidate_score","peak_prob","context_mean","context_high_frac",
                                    "row","col","x","y"])
cand=cand.sort_values("candidate_score",ascending=False).reset_index(drop=True)
n_before=len(cand)
cg=gpd.GeoDataFrame(cand,geometry=gpd.points_from_xy(cand.x,cand.y),crs=pcrs)

if VALIDATION_MODE:
    # Robust to a temporarily missing/stale .shx view on mounted Google Drive.
    with rasterio.Env(SHAPE_RESTORE_SHX='YES'):
        roi=gpd.read_file(CROWN_ROI)
    if roi.crs is None: raise ValueError("ROI_Crowns_Kvo.shp has no CRS.")
    if roi.crs!=pcrs: roi=roi.to_crs(pcrs)
    roi=roi[roi.geometry.notna() & ~roi.geometry.is_empty].copy()
    roi["geometry"]=roi.geometry.make_valid()
    roi_union=roi.geometry.union_all()
    cg=cg[cg.geometry.intersects(roi_union)].copy()
    print("Validation ROIs:",len(roi))
    print("Qualifying U-Net candidates inside validation ROIs:",len(cg))
else:
    if MAX_CANDIDATES is not None: cg=cg.head(MAX_CANDIDATES).copy()

cg=cg.reset_index(drop=True)
cg["candidate_id"]=np.arange(len(cg),dtype=int)
cg.to_file(OUT/"Baego_UNet_candidates.gpkg",layer="candidates",driver="GPKG")
print("Peak threshold:",PROB_THRESHOLD)
print("Minimum context mean:",MIN_CONTEXT_MEAN)
print("Minimum separation m:",PEAK_MIN_DISTANCE_M)
print("Candidates after NMS:",n_before)
print("Candidates retained for SAM3:",len(cg),"(validation mode)" if VALIDATION_MODE else "(production pilot)")
display(cg[["candidate_id","candidate_score","peak_prob","context_mean","context_high_frac"]].head(10))

# Prepare one compact RGB chip per retained candidate only.
manifest=[]
with rasterio.open(RASTER) as src:
    if src.crs!=pcrs: raise ValueError("RGB and probability raster CRS differ")
    px=float((abs(src.transform.a)+abs(src.transform.e))/2)
    chip_px=max(128,int(round(CHIP_SIZE_M/px)))
    if chip_px%2: chip_px+=1
    for r in tqdm(cg.itertuples(),total=len(cg),desc="RGB CHIPS"):
        row,col=src.index(r.x,r.y)
        w=Window(col-chip_px//2,row-chip_px//2,chip_px,chip_px)
        op=CHIPS/f"cand_{r.candidate_id:05d}.jpg"
        if FORCE_CHIPS or not op.exists():
            rgb=np.moveaxis(src.read([1,2,3],window=w,boundless=True,fill_value=0),0,-1).astype(np.uint8)
            Image.fromarray(rgb).save(op,quality=95)
        manifest.append(dict(candidate_id=r.candidate_id,path=str(op),x=r.x,y=r.y,
                             candidate_score=r.candidate_score,peak_prob=r.peak_prob,
                             context_mean=r.context_mean,context_high_frac=r.context_high_frac,
                             row_off=int(w.row_off),col_off=int(w.col_off),
                             height=chip_px,width=chip_px,gsd=px))
manifest=pd.DataFrame(manifest)
manifest.to_csv(OUT/"candidate_manifest.csv",index=False)
print("RGB chips:",len(manifest)," chip pixels:",chip_px)


## 2 — Isolated official SAM3 environment

In [ ]:
# SAM3 needs numpy<2, while current Colab geospatial packages need numpy>=2.
# Use uv to create a real isolated Python 3.12 environment; do NOT modify host Python.
UV=Path("/content/uv")
SAMENV=Path("/content/sam3env")
SAMREPO=Path("/content/sam3")

if not UV.exists():
    subprocess.run(["bash","-lc","curl -LsSf https://astral.sh/uv/install.sh | env UV_INSTALL_DIR=/content sh"],check=True)
    # uv installer may name the binary directly under /content.
    candidates=[Path("/content/uv"),Path("/content/uv/uv"),Path.home()/".local/bin/uv"]
    UV=next((p for p in candidates if p.exists() and p.is_file()),None)
    assert UV is not None,"uv installation failed"
else:
    UV=Path("/content/uv")
print("uv:",subprocess.check_output([str(UV),"--version"],text=True).strip())

# Recreate only the isolated SAM environment if a previous failed venv is present.
if SAMENV.exists() and not (SAMENV/"bin/python").exists():
    shutil.rmtree(SAMENV)
if not (SAMENV/"bin/python").exists():
    subprocess.run([str(UV),"venv","--python","3.12",str(SAMENV)],check=True)

SAMPY=str(SAMENV/"bin/python")
# Reuse Colab's CUDA-enabled torch wheel via explicit install in the isolated env.
subprocess.run([str(UV),"pip","install","--python",SAMPY,
                "torch==2.10.0","torchvision",
                "--index-url","https://download.pytorch.org/whl/cu128"],check=True)

if not SAMREPO.exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/facebookresearch/sam3.git",str(SAMREPO)],check=True)
else:
    subprocess.run(["git","-C",str(SAMREPO),"pull","--ff-only"],check=True)

# Official SAM3 currently requires numpy>=1.26,<2. Keep that constraint inside this env only.
subprocess.run([str(UV),"pip","install","--python",SAMPY,"setuptools<82"],check=True)
# The image builder imports modules that use packages listed by Meta under
# notebook/dev extras (not only the minimal core dependencies). Install the
# official extras rather than discovering them one-by-one at runtime.
subprocess.run([str(UV),"pip","install","--python",SAMPY,
                "-e",str(SAMREPO)+"[notebooks,dev]"],check=True)

subprocess.run([SAMPY,"-c",
    "import sys,numpy,torch,sam3; print('SAM Python',sys.version.split()[0],'numpy',numpy.__version__,'torch',torch.__version__,'CUDA',torch.cuda.is_available())"],
    check=True)
print("HOST unchanged: Python",sys.version.split()[0],"numpy",np.__version__,"rasterio",rasterio.__version__)


## 3 — Hugging Face authentication

In [ ]:
from google.colab import userdata
try: HF_TOKEN=userdata.get("HF_TOKEN")
except Exception: HF_TOKEN=None
assert HF_TOKEN,"Add your Hugging Face read token to Colab Secrets as HF_TOKEN."
print("HF_TOKEN found (value not displayed).")


## 4 — SAM3 delineation of U-Net candidates

SAM3 is now used as a **local delineator**, not a wall-to-wall detector. Each U-Net candidate supplies a small approximate box centred on the probability peak. SAM3 receives the local RGB chip plus that geometric prompt and returns a crown mask.

This is the computational advantage of the revised workflow: SAM3 is run only where the U-Net already provides evidence for Musanga.


In [ ]:
RUNNER=Path("/content/run_sam3_kvo06.py")
RUNNER.write_text(r'''import sys
from pathlib import Path
import numpy as np,pandas as pd,torch
from PIL import Image
from tqdm import tqdm
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

manifest=Path(sys.argv[1]); outdir=Path(sys.argv[2]); box_m=float(sys.argv[3]); force=sys.argv[4]=="1"
outdir.mkdir(parents=True,exist_ok=True); df=pd.read_csv(manifest)
if not torch.cuda.is_available(): raise RuntimeError("KVO_06 requires CUDA")
torch.backends.cuda.matmul.allow_tf32=True; torch.backends.cudnn.allow_tf32=True
torch.autocast("cuda",dtype=torch.bfloat16).__enter__(); torch.inference_mode().__enter__()
bpe="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz"
model=build_sam3_image_model(bpe_path=bpe); processor=Sam3Processor(model)
print("SAM3 ready; candidates=",len(df),flush=True)
for r in tqdm(df.itertuples(),total=len(df),desc="DELINEATE"):
    op=outdir/f"cand_{int(r.candidate_id):05d}.npz"
    if op.exists() and not force: continue
    img=Image.open(r.path).convert("RGB")
    state=processor.set_image(img)
    # Normalized centre-x, centre-y, width, height; candidate is chip centre.
    frac=min(0.95,box_m/(r.width*r.gsd))
    box=np.array([[0.5,0.5,frac,frac]],dtype=np.float32)
    out=processor.add_geometric_prompt(box=box,label=np.array([1],dtype=np.int32),state=state)
    masks=out["masks"].detach().cpu().numpy().astype(np.uint8)
    if masks.ndim==4: masks=masks[:,0]
    scores=out["scores"].detach().float().cpu().numpy()
    np.savez_compressed(op,masks=masks,scores=scores)
print("DONE",flush=True)
''')
env=os.environ.copy(); env["HF_TOKEN"]=HF_TOKEN
cmd=[SAMPY,"-u",str(RUNNER),str(OUT/"candidate_manifest.csv"),str(RAW),str(BOX_SIZE_M),"1" if FORCE_SAM else "0"]
print("Starting targeted SAM3 delineation with live output...")
p=subprocess.run(cmd,env=env)
print("Return code:",p.returncode)
if p.returncode!=0: raise RuntimeError("SAM3 delineation subprocess failed; error printed above.")


## 5 — Convert SAM3 masks to georeferenced candidate crown polygons

For each candidate we retain the SAM3 mask component containing the U-Net peak (the chip centre), convert it to map coordinates, and attach the original U-Net peak probability. Implausibly small/large objects are discarded.


In [ ]:
pred=[]
with rasterio.open(RASTER) as src:
    for r in tqdm(manifest.itertuples(),total=len(manifest),desc="POLYGONIZE"):
        z=np.load(RAW/f"cand_{int(r.candidate_id):05d}.npz")
        masks=z["masks"]; scores=z["scores"]
        if len(masks)==0: continue
        # Prefer highest-score mask containing chip centre.
        order=np.argsort(scores)[::-1]; chosen=None
        for j in order:
            m=masks[j].astype(np.uint8)
            if m.shape!=(r.height,r.width):
                m=np.array(Image.fromarray(m).resize((r.width,r.height),resample=Image.Resampling.NEAREST))
            if m[r.height//2,r.width//2]>0: chosen=(m,float(scores[j])); break
        if chosen is None: continue
        m,score=chosen
        wt=rasterio.windows.transform(Window(r.col_off,r.row_off,r.width,r.height),src.transform)
        geoms=[]
        for gj,val in shapes(m,mask=m.astype(bool),transform=wt):
            if val==1: geoms.append(shape(gj))
        if not geoms: continue
        pt=Point(r.x,r.y)
        geom=max((q for q in geoms if q.contains(pt) or q.touches(pt)),key=lambda q:q.area,default=None)
        if geom is None: continue
        area=geom.area
        if not (MIN_PRED_AREA_M2<=area<=MAX_PRED_AREA_M2): continue
        pred.append(dict(candidate_id=int(r.candidate_id),peak_prob=float(r.peak_prob),
                         sam_score=score,crown_area_m2=area,geometry=geom))
crowns=gpd.GeoDataFrame(pred,crs=src.crs)
crowns.to_file(OUT/"Baego_UNet_guided_SAM3_crowns.gpkg",layer="crowns",driver="GPKG")
print("Candidates:",len(manifest))
print("Valid SAM3 crown objects:",len(crowns))
display(crowns.drop(columns="geometry").head())


## 6 — Add object-level U-Net probability summaries

The peak probability detects the candidate; here we also summarize the KVO_04 probability raster inside each SAM3 crown. These are useful object-level attributes for later filtering and satellite aggregation.


In [ ]:
from rasterio.mask import mask as rio_mask
rows=[]
with rasterio.open(PROB) as ps:
    for r in tqdm(crowns.itertuples(),total=len(crowns),desc="OBJECT PROBABILITY"):
        arr,_=rio_mask(ps,[r.geometry],crop=True,filled=False,indexes=1)
        vals=np.asarray(arr.compressed(),dtype=np.float32)
        vals=vals[vals!=65535]/10000.0
        rows.append(dict(candidate_id=r.candidate_id,
                         mean_prob=float(np.mean(vals)) if len(vals) else np.nan,
                         median_prob=float(np.median(vals)) if len(vals) else np.nan,
                         p90_prob=float(np.quantile(vals,.9)) if len(vals) else np.nan))
stats=pd.DataFrame(rows)
crowns=crowns.merge(stats,on="candidate_id",how="left")
crowns.to_file(OUT/"Baego_UNet_guided_SAM3_crowns.gpkg",layer="crowns",driver="GPKG")
print(crowns[["peak_prob","mean_prob","median_prob","crown_area_m2"]].describe())


## 7 — Primary observational validation against Crown_Kristof

`Crown_Kristof.shp` contains **all visible crowns digitized within the six areas of `ROI_Crowns_Kvo.shp`**. This is therefore the primary observational basis for KVO_06.

The analysis is restricted to the six ROIs and reports both directions:

- **Reference recovery:** for every digitized visible crown, the best overlapping automatic U-Net→SAM3 object and its IoU.
- **Automatic-object matching:** for every automatic crown, the best overlapping digitized crown and its IoU.

This allows unmatched automatic objects *inside these exhaustively digitized visible-crown areas* to be identified explicitly, while keeping the geometric match statistics separate from the U-Net species-probability evidence.


In [ ]:
from shapely.validation import make_valid

def safe_iou(a,b):
    try:
        a=make_valid(a); b=make_valid(b)
        inter=a.intersection(b).area
        if inter<=0: return 0.0
        union=a.union(b).area
        return float(inter/union) if union>0 else 0.0
    except Exception:
        return 0.0

ref=gpd.read_file(CROWN_REF)
roi=gpd.read_file(CROWN_ROI)
if ref.crs is None:
    print("Assigning EPSG:32635 to Crown_Kristof (Baego digitizing CRS)")
    ref=ref.set_crs("EPSG:32635")
if roi.crs is None: raise ValueError("ROI_Crowns_Kvo missing CRS")
if ref.crs!=crowns.crs: ref=ref.to_crs(crowns.crs)
if roi.crs!=crowns.crs: roi=roi.to_crs(crowns.crs)
ref=ref[ref.geometry.notna() & ~ref.geometry.is_empty].copy()
roi=roi[roi.geometry.notna() & ~roi.geometry.is_empty].copy()
ref["geometry"]=ref.geometry.apply(make_valid)
roi["geometry"]=roi.geometry.apply(make_valid)
roi_union=roi.geometry.union_all()

# Enforce the observational footprint for both datasets.
ref=ref[ref.geometry.intersects(roi_union)].copy().reset_index(drop=True)
auto=crowns[crowns.geometry.intersects(roi_union)].copy().reset_index(drop=True)
print("Validation ROIs:",len(roi))
print("Digitized visible reference crowns:",len(ref))
print("Automatic SAM3 crowns in ROIs:",len(auto))

# Automatic object -> best reference crown.
ref_sidx=ref.sindex
auto_rows=[]
for r in tqdm(auto.itertuples(),total=len(auto),desc="AUTO -> REFERENCE"):
    ids=list(ref_sidx.query(r.geometry,predicate="intersects"))
    pairs=[(int(i),safe_iou(r.geometry,ref.iloc[i].geometry)) for i in ids]
    bi,bv=max(pairs,key=lambda z:z[1],default=(-1,0.0))
    auto_rows.append(dict(candidate_id=int(r.candidate_id),best_ref_index=bi,best_ref_iou=bv))
auto_qa=pd.DataFrame(auto_rows)
auto_qa.to_csv(OUT/"validation_auto_to_reference.csv",index=False)

# Reference crown -> best automatic object.
auto_sidx=auto.sindex
ref_rows=[]
for i,r in tqdm(ref.iterrows(),total=len(ref),desc="REFERENCE -> AUTO"):
    ids=list(auto_sidx.query(r.geometry,predicate="intersects"))
    pairs=[(int(j),safe_iou(r.geometry,auto.iloc[j].geometry)) for j in ids]
    bj,bv=max(pairs,key=lambda z:z[1],default=(-1,0.0))
    cid=int(auto.iloc[bj].candidate_id) if bj>=0 else -1
    ref_rows.append(dict(ref_index=int(i),best_candidate_id=cid,best_auto_iou=bv))
ref_qa=pd.DataFrame(ref_rows)
ref_qa.to_csv(OUT/"validation_reference_to_auto.csv",index=False)

nref=len(ref); nauto=len(auto)
print("\nREFERENCE RECOVERY")
print("Any overlap:",int((ref_qa.best_auto_iou>0).sum()),"/",nref)
print("IoU >= 0.5:",int((ref_qa.best_auto_iou>=0.5).sum()),"/",nref)
print("IoU >= 0.7:",int((ref_qa.best_auto_iou>=0.7).sum()),"/",nref)
if (ref_qa.best_auto_iou>0).any():
    print("Median IoU among overlapping reference crowns:",
          float(ref_qa.loc[ref_qa.best_auto_iou>0,"best_auto_iou"].median()))

print("\nAUTOMATIC OBJECT MATCHING")
print("Any reference overlap:",int((auto_qa.best_ref_iou>0).sum()),"/",nauto)
print("IoU >= 0.5:",int((auto_qa.best_ref_iou>=0.5).sum()),"/",nauto)
print("IoU >= 0.7:",int((auto_qa.best_ref_iou>=0.7).sum()),"/",nauto)
if (auto_qa.best_ref_iou>0).any():
    print("Median IoU among overlapping automatic crowns:",
          float(auto_qa.loc[auto_qa.best_ref_iou>0,"best_ref_iou"].median()))


## 8 — Diagnostics and production summary

In [ ]:
fig,ax=plt.subplots(figsize=(7,5))
if len(crowns):
    ax.scatter(crowns.peak_prob,crowns.mean_prob,s=12,alpha=.6)
ax.set_xlabel("U-Net candidate peak probability"); ax.set_ylabel("Mean U-Net probability inside SAM3 crown")
ax.set_title("Baego: detector → delineator consistency")
plt.show()

summary={
    "probability_threshold":PROB_THRESHOLD,
    "minimum_candidate_separation_m":PEAK_MIN_DISTANCE_M,
    "n_candidates":int(len(manifest)),
    "n_valid_sam3_crowns":int(len(crowns)),
    "median_crown_area_m2":float(crowns.crown_area_m2.median()) if len(crowns) else None,
    "median_object_mean_probability":float(crowns.mean_prob.median()) if len(crowns) else None,
}
(OUT/"summary.json").write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))


## 9 — Interpretation / next step

KVO_06 now uses the six exhaustively digitized crown areas as its **observational basis**.

- **KVO_03:** independent leave-one-site-out evidence for U-Net species classification.
- **KVO_04:** final continuous Musanga probability maps.
- **KVO_05:** prompt-conditioned SAM3 crown-boundary benchmark.
- **KVO_06:** fully automatic U-Net → SAM3 workflow evaluated inside six areas where all visible crowns were digitized.

The production-scale Baego inventory should only be expanded after examining recovery, matching IoU, and unmatched automatic objects in these six validation areas.


## 10 — Experiment A: adaptive U-Net-guided SAM3 prompts

Uses local 0.60 probability components for approximate size and four prompt bins (10/14/18/22m). The original 0.90 candidate threshold is retained for seeds. This is a first sensitivity experiment, not final calibration.


In [ ]:

# BLOCK 10 — Adaptive U-Net box prompts; reuses existing chips and SAM3 runner.
from scipy.ndimage import label as ndi_label
from shapely.validation import make_valid
EXP=OUT/"experiments_v09"; EXP.mkdir(exist_ok=True,parents=True)
MAN=pd.read_csv(OUT/"candidate_manifest.csv")
BOX_VALUES=[10.,14.,18.,22.]
def box_from_probability(r,threshold=.60,margin=1.25):
    with rasterio.open(PROB) as ds:
        rr,cc=ds.index(float(r.x),float(r.y))
        rad=int(round(15/abs(ds.transform.a)))
        a=ds.read(1,window=Window(cc-rad,rr-rad,2*rad+1,2*rad+1),boundless=True,fill_value=65535)
        good=(a!=65535)&(a>=int(threshold*10000))
        labs,num=ndi_label(good)
        k=labs[rad,rad]
        if k==0:return 14.
        ys,xs=np.where(labs==k)
        extent=max((xs.max()-xs.min()+1)*abs(ds.transform.a),(ys.max()-ys.min()+1)*abs(ds.transform.e))
        return float(np.clip(extent*margin,8,22))
MAN["adaptive_box_m"]=[box_from_probability(r) for r in MAN.itertuples()]
MAN.to_csv(EXP/"adaptive_boxes.csv",index=False)
print(MAN.adaptive_box_m.describe())
# Existing runner accepts one box size per call; group adaptive boxes into 4 bins,
# retaining original candidate_id and output paths for reproducibility.
MAN["box_group"]=MAN.adaptive_box_m.apply(lambda x:min(BOX_VALUES,key=lambda b:abs(b-x)))
adaptive_raw=EXP/"adaptive_raw";adaptive_raw.mkdir(exist_ok=True)
for b,g in MAN.groupby("box_group"):
    mf=EXP/f"adaptive_manifest_{int(b)}m.csv";g.to_csv(mf,index=False)
    p=subprocess.run([SAMPY,"-u",str(RUNNER),str(mf),str(adaptive_raw),str(b),"0"],env=env)
    if p.returncode:raise RuntimeError(f"SAM3 failed for {b} m")
print("Adaptive masks ready:",len(list(adaptive_raw.glob("cand_*.npz"))))


## 11 — Adaptive crown agreement and reproducibility

Compare against manual interpreted boundaries; no species-level validation claim.


In [ ]:

# BLOCK 11 — Shared polygonization and manual–automatic agreement.
from rasterio.features import shapes
from shapely.geometry import Point,shape
from PIL import Image
def polygons_from_masks(manifest_df,raw_dir):
    records=[]
    with rasterio.open(RASTER) as src:
        for r in tqdm(manifest_df.itertuples(),total=len(manifest_df),desc="POLYGONIZE"):
            p=raw_dir/f"cand_{int(r.candidate_id):05d}.npz"
            if not p.exists():continue
            z=np.load(p);masks=z["masks"];scores=z["scores"]
            if not len(masks):continue
            for j in np.argsort(scores)[::-1]:
                m=masks[j].astype("uint8")
                if m.ndim==3:m=m[0]
                if m.shape!=(r.height,r.width):
                    m=np.array(Image.fromarray(m).resize((r.width,r.height),Image.Resampling.NEAREST))
                if not m[r.height//2,r.width//2]:continue
                wt=rasterio.windows.transform(Window(r.col_off,r.row_off,r.width,r.height),src.transform)
                pt=Point(r.x,r.y)
                geoms=[shape(g) for g,v in shapes(m,mask=m.astype(bool),transform=wt) if v==1]
                geom=max((g for g in geoms if g.intersects(pt)),key=lambda g:g.area,default=None)
                if geom is not None and MIN_PRED_AREA_M2<=geom.area<=MAX_PRED_AREA_M2:
                    records.append(dict(candidate_id=int(r.candidate_id),area_m2=geom.area,geometry=make_valid(geom)))
                break
    return gpd.GeoDataFrame(records,geometry="geometry",crs=rasterio.open(RASTER).crs)
def agreement(auto,reference):
    if auto.crs!=reference.crs:auto=auto.to_crs(reference.crs)
    idx=reference.sindex;rows=[]
    for a in auto.itertuples():
        best=(-1,0.,0.,np.nan)
        for j in idx.query(a.geometry,predicate="intersects"):
            g=reference.iloc[j].geometry
            inter=a.geometry.intersection(g).area
            union=a.geometry.union(g).area
            iou=inter/union if union else 0.
            if iou>best[1]:best=(int(j),iou,inter,g.area)
        j,iou,inter,refarea=best
        rows.append(dict(candidate_id=a.candidate_id,manual_index=j,iou=iou,area_ratio=a.geometry.area/refarea if j>=0 else np.nan,manual_coverage=inter/refarea if j>=0 else np.nan,sam_inside_manual=inter/a.geometry.area if a.geometry.area else np.nan))
    return pd.DataFrame(rows)
reference=gpd.read_file(CROWN_REF)
if reference.crs is None:reference=reference.set_crs("EPSG:32635")
reference=reference[reference.geometry.intersects(roi_union)].copy()
adaptive_crowns=polygons_from_masks(MAN,adaptive_raw)
adaptive_qa=agreement(adaptive_crowns,reference)
adaptive_qa.to_csv(EXP/"adaptive_agreement.csv",index=False)
adaptive_crowns.to_file(EXP/"adaptive_crowns.gpkg",driver="GPKG")
print("Adaptive valid crowns:",len(adaptive_crowns),"overlap:",(adaptive_qa.iou>0).sum(),"median matched IoU:",adaptive_qa.loc[adaptive_qa.iou>0,"iou"].median())
# Reproducibility: compare the original 10m mask files against the sensitivity-run 10m files.
original_ids={int(p.stem.split("_")[-1]) for p in RAW.glob("cand_*.npz")}
print("Original 10m raw masks:",len(original_ids),"adaptive masks:",len(list(adaptive_raw.glob("cand_*.npz"))))


## 12 — Experiment B: species-independent RGB crown candidates

**Pilot approximation:** RGB local-extrema proposals followed by SAM3 geometric prompts. This is not native SAM3 automatic discovery and may miss crowns. The 846 manual crowns are observational comparisons only.


In [ ]:

# BLOCK 12 — Species-independent crown proposals within the six ROIs.
# Generic RGB candidate generation via local canopy brightness/texture extrema.
# This is a pilot heuristic, NOT native automatic SAM3 discovery; compare empirically.
from scipy.ndimage import gaussian_filter,maximum_filter
from rasterio.mask import mask as rio_mask
from shapely.geometry import mapping
generic=[]
with rasterio.open(RASTER) as src:
    for k,region in enumerate(roi.geometry):
        arr,tr=rio_mask(src,[mapping(region)],crop=True,indexes=[1,2,3],filled=True,nodata=0)
        rgb=arr.astype("float32").mean(axis=0)
        # Local RGB texture peaks, spaced ~5m; exclude masked background.
        px=abs(src.transform.a); sep=max(3,int(round(5/px)))
        smooth=gaussian_filter(rgb,sigma=max(2,sep/5))
        peak=(smooth==maximum_filter(smooth,size=2*sep+1))&(rgb>20)
        ys,xs=np.where(peak)
        for y,x in zip(ys,xs):
            xx,yy=rasterio.transform.xy(tr,int(y),int(x))
            if region.covers(Point(xx,yy)):generic.append((k,xx,yy))
print("Generic RGB candidates:",len(generic))
# Build independent chips, same manifest contract as Block 4.
grows=[]
with rasterio.open(RASTER) as src:
    px=abs(src.transform.a);size=max(128,int(round(CHIP_SIZE_M/px)));size+=size%2
    for i,(region,x,y) in enumerate(tqdm(generic,desc="GENERIC RGB CHIPS")):
        row,col=src.index(x,y);w=Window(col-size//2,row-size//2,size,size)
        out=src.read([1,2,3],window=w,boundless=True,fill_value=0).transpose(1,2,0)
        path=EXP/f"generic_{i:05d}.png"
        if not path.exists():Image.fromarray(out.astype("uint8")).save(path)
        grows.append(dict(candidate_id=i,region=region,x=x,y=y,row_off=int(w.row_off),col_off=int(w.col_off),width=size,height=size,gsd=px,path=str(path)))
generic_manifest=pd.DataFrame(grows)
generic_manifest.to_csv(EXP/"generic_manifest.csv",index=False)
print("Generic chips:",len(generic_manifest))


## 13 — Generic RGB-guided SAM3 and U-Net attribution


In [ ]:

# BLOCK 13 — Species-independent SAM3 segmentation + agreement.
# Uses 14m generic prompts; independent of U-Net seed threshold.
generic_raw=EXP/"generic_raw";generic_raw.mkdir(exist_ok=True)
p=subprocess.run([SAMPY,"-u",str(RUNNER),str(EXP/"generic_manifest.csv"),str(generic_raw),"14","0"],env=env)
if p.returncode:raise RuntimeError("Generic SAM3 failed")
generic_crowns=polygons_from_masks(generic_manifest,generic_raw)
generic_qa=agreement(generic_crowns,reference)
generic_qa.to_csv(EXP/"generic_agreement.csv",index=False)
generic_crowns.to_file(EXP/"generic_crowns.gpkg",driver="GPKG")
print("Generic valid:",len(generic_crowns),"overlapping manual:",int((generic_qa.iou>0).sum()),"median matched IoU:",generic_qa.loc[generic_qa.iou>0,"iou"].median())
# Overlay species probability only AFTER delineation.
from rasterio.mask import mask as rio_mask
with rasterio.open(PROB) as ps:
    vals=[]
    for g in generic_crowns.geometry:
        a,_=rio_mask(ps,[mapping(g)],crop=True,indexes=1,filled=False)
        v=np.asarray(a.compressed(),dtype="float32");v=v[v!=65535]/10000
        vals.append(float(v.mean()) if len(v) else np.nan)
generic_crowns["musanga_mean_prob"]=vals
generic_crowns.to_file(EXP/"generic_crowns_with_probability.gpkg",driver="GPKG")
display(pd.DataFrame([{"method":"adaptive","valid":len(adaptive_crowns),"matched":int((adaptive_qa.iou>0).sum()),"median_iou":adaptive_qa.loc[adaptive_qa.iou>0,"iou"].median()},{"method":"generic_rgb","valid":len(generic_crowns),"matched":int((generic_qa.iou>0).sum()),"median_iou":generic_qa.loc[generic_qa.iou>0,"iou"].median()}]))


## 14 — Revised all-crown reference and Baego DSM audit

Run **0R** first in a new session; **Blocks 14–16 do not need SAM3, GPU, or Blocks 1–13**. This reads the revised Crown_Kristof shapefile afresh and checks DSM/RGB alignment.


In [ ]:
# BLOCK 14 — Audit revised all-crown reference and DSM (no SAM3/GPU)
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.mask import mask as rio_mask
from shapely.geometry import mapping
from scipy.ndimage import gaussian_filter,maximum_filter,minimum_filter
from shapely.validation import make_valid
DSM=GEO/"Baego_DSM.tif"
assert DSM.exists(),f"DSM not found: {DSM}"
reference_v2=gpd.read_file(CROWN_REF)
if reference_v2.crs is None:
    print("Crown_Kristof CRS absent: assigning EPSG:32635 (verify digitizing CRS)")
    reference_v2=reference_v2.set_crs("EPSG:32635")
with rasterio.open(RASTER) as rgb,rasterio.open(DSM) as dsm:
    reference_v2=reference_v2.to_crs(rgb.crs)
    roi_v2=gpd.read_file(CROWN_ROI).to_crs(rgb.crs)
    print("RGB:",rgb.crs,rgb.res,"DSM:",dsm.crs,dsm.res)
    dx=(dsm.transform.c-rgb.transform.c)/rgb.res[0]
    dy=(dsm.transform.f-rgb.transform.f)/rgb.res[1]
    print("DSM pixel-grid compatible:",rgb.crs==dsm.crs and np.allclose(rgb.res,dsm.res,atol=1e-7,rtol=0) and np.allclose([dx,dy],np.round([dx,dy]),atol=1e-5),"offset pixels:",(dx,dy))
reference_v2=reference_v2[reference_v2.geometry.notna()&(~reference_v2.geometry.is_empty)].copy()
reference_v2["geometry"]=reference_v2.geometry.map(make_valid)
ref_roi_union=roi_v2.geometry.union_all()
reference_v2=reference_v2[reference_v2.geometry.intersects(ref_roi_union)].copy()
print("Revised crowns:",len(reference_v2),"ROIs:",len(roi_v2),"median area m2:",round(reference_v2.geometry.area.median(),1))
print("Invalid after repair:",(~reference_v2.is_valid).sum())
print("Per-ROI counts:",[int(reference_v2.intersects(g).sum()) for g in roi_v2.geometry])


## 15 — Multi-scale DSM candidate detection and ranked RGB fusion

Uses aligned **Baego_DSM.tif**, elevations in metres. Detects canopy maxima at 3 smoothing scales, ranks by local relief, tests 1.5–4m separation, preserves RGB candidates. CPU-only; no SAM3.


In [ ]:
# BLOCK 15 v2 — Multi-scale DSM maxima + ranked fusion; CPU only
# Reads aligned metre-valued DSM, preserving original RGB candidates.
from scipy.ndimage import gaussian_filter, maximum_filter, minimum_filter, label
from scipy.spatial import cKDTree
from rasterio.mask import mask as rio_mask
from shapely.geometry import mapping, Point
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from pathlib import Path

DSM = GEO/"Baego_DSM.tif"
if not DSM.exists():
    raise FileNotFoundError(f"DSM not found: {DSM}")
EXP15=OUT/"experiments_v15_dsm";EXP15.mkdir(parents=True,exist_ok=True)
if "generic_manifest" not in globals():
    options=list(OUT.rglob("*generic*manifest*.csv"))
    if not options: raise FileNotFoundError("Run Block 12 or provide the saved generic_manifest CSV.")
    generic_manifest=pd.read_csv(options[0])
    print("Restored RGB candidates from:",options[0])
rgb_points=generic_manifest[["x","y"]].dropna().drop_duplicates().copy()
rgb_points["source"]="RGB"
rgb_points["local_relief_m"]=np.nan
print("RGB candidates:",len(rgb_points))

# Use ROI crops, not full-resolution DSM. Check actual grid match and overlap.
peak_rows=[]
with rasterio.open(RASTER) as rgb, rasterio.open(DSM) as dsm:
    dx=(dsm.transform.c-rgb.transform.c)/rgb.res[0]
    dy=(dsm.transform.f-rgb.transform.f)/rgb.res[1]
    same=(rgb.crs==dsm.crs and np.allclose(rgb.res,dsm.res,atol=1e-7,rtol=0) and np.allclose([dx,dy],np.round([dx,dy]),atol=1e-5))
    print("DSM/RGB pixel-grid compatible:",same,"offset:",(dx,dy),"DSM units: metres (user confirmed)")
    if not same: raise ValueError("DSM pixel grids differ; check Snap Raster and cell size in ArcGIS.")
    if not (dsm.bounds.left<=rgb.bounds.right and dsm.bounds.right>=rgb.bounds.left and dsm.bounds.bottom<=rgb.bounds.top and dsm.bounds.top>=rgb.bounds.bottom):
        raise ValueError("DSM and RGB do not overlap.")
    gsd=abs(rgb.transform.a)
    for roi_id,geom in enumerate(roi_v2.geometry):
        masked,tr=rio_mask(dsm,[mapping(geom)],crop=True,indexes=1,filled=False)
        valid=(~np.ma.getmaskarray(masked)) & np.isfinite(masked.data)
        if not valid.any(): continue
        arr=masked.astype("float32").filled(np.nan)
        baseline=float(np.nanmedian(arr[valid]))
        arr=np.where(valid,arr,baseline)
        # Multi-scale smoothing; local relief removes slow topographic gradients.
        for smooth_m in (0.35,0.70,1.20):
            sigma=max(0.5,smooth_m/gsd)
            sm=gaussian_filter(arr,sigma=sigma)
            base=minimum_filter(sm,size=2*max(1,round(10/gsd))+1)
            relief=sm-base
            # 1.5m neighborhood to generate candidates; separation tuned downstream.
            r=max(1,round(1.5/gsd))
            peaks=(sm>=maximum_filter(sm,size=2*r+1)-1e-5)&valid&(relief>=0.35)
            # Collapse plateaus into one location rather than hundreds of equal-value pixels.
            labs,nlab=label(peaks)
            for labid in range(1,nlab+1):
                yy,xx=np.where(labs==labid)
                k=np.argmax(relief[yy,xx]); y=int(yy[k]);x=int(xx[k])
                east,north=rasterio.transform.xy(tr,y,x)
                if geom.covers(Point(east,north)):
                    peak_rows.append((roi_id,east,north,float(relief[y,x]),float(sm[y,x]),smooth_m))
        print(f"ROI {roi_id+1}: raw multiscale peaks so far {len(peak_rows)}")
peaks_df=pd.DataFrame(peak_rows,columns=["roi","x","y","local_relief_m","surface_elevation_m","smoothing_m"])
peaks_df.to_csv(EXP15/"dsm_multiscale_raw_peaks.csv",index=False)
print("Multi-scale peaks:",len(peaks_df))

# Rank strong local-relief peaks first; avoid duplicate maxima from different scales.
def greedy_spatial_nms(frame,sep_m,initial=None):
    if frame.empty:return frame.copy()
    f=frame.sort_values(["local_relief_m","smoothing_m"],ascending=[False,False]).reset_index(drop=True)
    coords=f[["x","y"]].to_numpy()
    tree=cKDTree(coords)
    blocked=np.zeros(len(f),bool)
    if initial is not None and len(initial):
        near=cKDTree(initial[["x","y"]].to_numpy()).query(coords)[0]
        blocked |= near<sep_m
    chosen=[]
    for i in range(len(f)):
        if blocked[i]:continue
        chosen.append(i)
        blocked[tree.query_ball_point(coords[i],r=sep_m)]=True
    return f.iloc[chosen].copy().reset_index(drop=True)

# Compare alternative minimum distances; preserve all RGB candidates.
fusions={}
for spacing in (1.5,2.0,2.5,3.0,4.0):
    dsm_keep=greedy_spatial_nms(peaks_df,spacing,initial=rgb_points)
    fusion=pd.concat([rgb_points,dsm_keep.assign(source="DSM")],ignore_index=True)
    fusion["candidate_id"]=np.arange(len(fusion))
    fusions[spacing]=fusion
    print(f"Spacing {spacing:.1f}m: DSM added {len(dsm_keep)}, fused {len(fusion)}")
    fusion.to_csv(EXP15/f"fused_spacing_{str(spacing).replace('.','p')}m.csv",index=False)
# Default 2m, chosen provisionally; Block 16 compares all variants.
accepted=fusions[2.0]
dsm_points=peaks_df
print("Default fused candidates:",len(accepted))


## 16 — Sensitivity and crown-size-specific coverage

Compares all candidate-spacing variants against the revised full-coverage crown annotations. Saves CSV reports; no SAM3.


In [ ]:
# BLOCK 16 v2 — Evaluate multiscale DSM fusion against revised all-crown reference
import geopandas as gpd
from scipy.spatial import cKDTree
def coverage_v2(points,ref):
    pts=gpd.GeoDataFrame(points.copy(),geometry=gpd.points_from_xy(points.x,points.y),crs=ref.crs)
    hits=gpd.sjoin(pts,ref[["geometry"]].reset_index(drop=True),how="left",predicate="within")
    idx=hits.index_right.dropna().astype(int)
    inside=hits.loc[hits.index_right.notna()].index.nunique()
    return dict(candidates=len(pts),crowns=len(ref),crowns_covered=idx.nunique(),coverage_pct=round(100*idx.nunique()/len(ref),1),candidates_inside=int(inside),candidates_outside=int(len(pts)-inside)),set(idx.tolist())
results=[];covered_sets={}
for name,pts in [("RGB",rgb_points)]+[(f"RGB+DSM {spacing:g}m",pts) for spacing,pts in fusions.items()]:
    stats,covered=coverage_v2(pts,reference_v2)
    results.append(dict(method=name,**stats))
    covered_sets[name]=covered
table15=pd.DataFrame(results)
display(table15)
table15.to_csv(EXP15/"candidate_coverage_sensitivity.csv",index=False)
# Crown-size stratification for RGB and default DSM fusion.
areas=reference_v2.reset_index(drop=True).geometry.area
bands=pd.cut(areas,bins=[0,20,40,80,float("inf")],labels=["<20","20-40","40-80",">80"],include_lowest=True)
size_rows=[]
for method in ("RGB","RGB+DSM 2m"):
    covered=covered_sets[method]
    for band in bands.cat.categories:
        ids=set(np.flatnonzero((bands==band).to_numpy()))
        size_rows.append(dict(method=method,crown_area_m2=str(band),crowns=len(ids),covered=len(ids&covered),coverage_pct=round(100*len(ids&covered)/len(ids),1) if ids else np.nan))
display(pd.DataFrame(size_rows))
print("Choose spacing based on coverage versus number of candidates; do not run SAM3 yet.")


## 17–19 — New RGB/DSM and SAM3 crown-discovery experiments

**Current recommended workflow: Block 19** runs SAM3 directly on a dense geometric-prompt grid over one complete ROI, with no treetop-candidate bottleneck. It uses 24 m overlapping RGB tiles, 6 m prompt spacing and 10 m boxes. It caches every SAM3 prediction on Drive, conservatively removes duplicates, and evaluates against the revised manually digitized crowns.

**Run in a fresh Colab runtime:** **0R → 2 (only if /content/sam3env is missing) → 14 → 19**. Block 0S and Blocks 15–18 are **not required** for Block 19. Use a GPU runtime for Block 19. Set `MAX_TILES=4` for a smoke test or `None` for the full ROI. Smoke-test and full-run outputs are isolated.

Earlier Blocks 17/18/18B were developed interactively; their saved results remain on Google Drive, and the full-ROI successor is consolidated below.


In [ ]:
# BLOCK 19 — Full ROI, candidate-free SAM3 crown discovery (resumable)
# Run after 0R, 2 (if environment absent), 14. No need for 0S or 15–18.
import os, sys, subprocess, json, math
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import Window
from rasterio.features import shapes
from rasterio.enums import Resampling
from shapely.geometry import box, Point, shape
from scipy.optimize import linear_sum_assignment
from PIL import Image
import matplotlib.pyplot as plt

ROI_INDEX = 0
TILE_M = 24.0
OVERLAP_M = 6.0
GRID_M = 6.0
BOX_M = 10.0
MAX_TILES = None       # None = complete ROI; 4 = technical smoke test
MIN_AREA_M2, MAX_AREA_M2 = 2.0, 500.0
EDGE_MARGIN_M = 0.4
DUP_IOU = 0.65
NESTED_CONTAINMENT = 0.95
SIMILAR_AREA_RATIO = 0.65

assert "roi_v2" in globals() and "reference_v2" in globals(), "Run Block 14"
assert Path(SAMPY).exists(), "Run Block 2 to install isolated SAM3 environment"
B19 = OUT / f"experiments_v19_full_roi_{ROI_INDEX}"
TILES, RAW19 = B19/"tiles", B19/"raw"
for d in (B19,TILES,RAW19): d.mkdir(parents=True,exist_ok=True)

with rasterio.open(RASTER) as src:
    crs, gsd = src.crs, abs(src.transform.a)
    roi19 = roi_v2.to_crs(crs).geometry.iloc[ROI_INDEX]
    n = int(round(TILE_M/gsd))
    stride = max(1,int(round((TILE_M-OVERLAP_M)/gsd)))
    minx,miny,maxx,maxy = roi19.bounds
    r0,c0 = src.index(minx,maxy)
    r1,c1 = src.index(maxx,miny)
    def starts(lo,hi):
        if hi-lo <= n: return [int(round((lo+hi-n)/2))]
        seq=list(range(int(lo),int(hi)-n+1,stride))
        last=int(hi)-n
        if not seq or seq[-1]!=last: seq.append(last)
        return sorted(set(seq))
    tiles=[]
    for rr in starts(r0,r1+1):
        for cc in starts(c0,c1+1):
            win=Window(cc,rr,n,n)
            b=rasterio.windows.bounds(win,src.transform)
            if not box(*b).intersects(roi19): continue
            tid=len(tiles); path=TILES/f"tile_{tid:04d}.png"
            tiles.append(dict(tile_id=tid,row_off=rr,col_off=cc,width=n,height=n,
                              xmin=b[0],ymin=b[1],xmax=b[2],ymax=b[3],path=str(path)))
    if MAX_TILES is not None: tiles=tiles[:MAX_TILES]
    assert tiles,"No ROI tiles"
    for t in tiles:
        p=Path(t["path"])
        if p.exists(): continue
        win=Window(t["col_off"],t["row_off"],n,n)
        rgb=src.read([1,2,3],window=win,boundless=True,fill_value=0).transpose(1,2,0)
        Image.fromarray(rgb.astype(np.uint8)).save(p)
tm=pd.DataFrame(tiles)
# Scope outputs by the selected tile IDs so smoke tests never masquerade as full ROI.
run_tag="full" if MAX_TILES is None else f"first_{MAX_TILES}_tiles"
work=B19/run_tag; raw=work/"raw"; raw.mkdir(parents=True,exist_ok=True)
rows=[]
for t in tm.itertuples():
    for dy in np.arange(GRID_M/2,TILE_M,GRID_M):
        for dx in np.arange(GRID_M/2,TILE_M,GRID_M):
            x,y=t.xmin+dx,t.ymax-dy
            if roi19.covers(Point(x,y)):
                rows.append(dict(tile_id=int(t.tile_id),path=t.path,cx=dx/TILE_M,cy=dy/TILE_M,
                                 box_m=BOX_M,width=n,height=n,gsd=gsd,
                                 row_off=int(t.row_off),col_off=int(t.col_off)))
pm=pd.DataFrame(rows)
assert len(pm),"No grid prompts inside ROI"
pm.insert(0,"prompt_id",np.arange(len(pm)))
pm.to_csv(work/"prompt_manifest.csv",index=False)
print(f"ROI {ROI_INDEX}: {roi19.area/10000:.3f} ha | tiles={len(tm)} | prompts={len(pm)} | mode={run_tag}")
print("Existing masks:",sum((raw/f"prompt_{i:06d}.npz").exists() for i in pm.prompt_id))

runner=Path("/content/run_sam3_kvo19.py")
runner.write_text(r'''
import sys
from pathlib import Path
import numpy as np,pandas as pd,torch
from PIL import Image
from tqdm import tqdm
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
df=pd.read_csv(sys.argv[1]); outdir=Path(sys.argv[2]);outdir.mkdir(parents=True,exist_ok=True)
if not torch.cuda.is_available(): raise RuntimeError("CUDA GPU required")
torch.backends.cuda.matmul.allow_tf32=True
bpe="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz"
with torch.inference_mode(),torch.autocast("cuda",dtype=torch.bfloat16):
    model=build_sam3_image_model(bpe_path=bpe)
    processor=Sam3Processor(model)
    for tid,g in df.groupby("tile_id",sort=True):
        pending=[r for r in g.itertuples() if not (outdir/f"prompt_{int(r.prompt_id):06d}.npz").exists()]
        if not pending:continue
        img=Image.open(pending[0].path).convert("RGB")
        for r in tqdm(pending,desc=f"Tile {tid}",leave=False):
            # Fresh prompt state per box, to avoid accumulated prompts.
            state=processor.set_image(img)
            frac=min(.95,float(r.box_m)/(float(r.width)*float(r.gsd)))
            out=processor.add_geometric_prompt(
                box=np.array([[float(r.cx),float(r.cy),frac,frac]],dtype=np.float32),
                label=np.array([1],dtype=np.int32),state=state)
            masks=out["masks"].detach().cpu().numpy()
            scores=out["scores"].detach().float().cpu().numpy()
            if masks.ndim==4:masks=masks[:,0]
            if masks.ndim==2:masks=masks[None]
            np.savez_compressed(outdir/f"prompt_{int(r.prompt_id):06d}.npz",
                                masks=(masks>0.5).astype(np.uint8),scores=scores)
        print(f"Tile {tid} complete",flush=True)
print("Inference complete",flush=True)
''')
missing=sum(not (raw/f"prompt_{i:06d}.npz").exists() for i in pm.prompt_id)
if missing:
    run_env=os.environ.copy()
    if "env" in globals(): run_env.update(env)
    print("Running SAM3 for",missing,"unsaved prompts")
    subprocess.run([SAMPY,"-u",str(runner),str(work/"prompt_manifest.csv"),str(raw)],
                   env=run_env,check=True)
else: print("All masks cached; skipping SAM3")

# Convert every returned instance, not just the highest-scoring mask.
features=[]
with rasterio.open(RASTER) as src:
    for r in pm.itertuples():
        p=raw/f"prompt_{int(r.prompt_id):06d}.npz"
        if not p.exists():continue
        with np.load(p) as z:
            masks=z["masks"]; scores=np.asarray(z["scores"]).reshape(-1)
        if masks.ndim==4:masks=masks[:,0]
        if masks.ndim==2:masks=masks[None]
        win=Window(int(r.col_off),int(r.row_off),int(r.width),int(r.height))
        transform=src.window_transform(win)
        for k,m in enumerate(masks):
            m=np.asarray(m)>0
            if m.shape!=(n,n) or not m.any():continue
            margin=max(1,int(round(EDGE_MARGIN_M/gsd)))
            if m[:margin].any() or m[-margin:].any() or m[:,:margin].any() or m[:,-margin:].any():
                continue
            geoms=[shape(g) for g,v in shapes(m.astype("uint8"),mask=m,transform=transform) if v==1]
            if not geoms:continue
            geom=max(geoms,key=lambda g:g.area)
            if not (MIN_AREA_M2<=geom.area<=MAX_AREA_M2):continue
            if not roi19.covers(geom.representative_point()):continue
            features.append(dict(prompt_id=int(r.prompt_id),tile_id=int(r.tile_id),
                                 instance=int(k),score=float(scores[min(k,len(scores)-1)]),
                                 area_m2=float(geom.area),geometry=geom))
print("Valid raw proposals:",len(features))
assert features,"No polygons survived filtering; inspect tile-edge masks"
allpred=gpd.GeoDataFrame(features,geometry="geometry",crs=crs)
allpred.to_file(work/"all_proposals.gpkg",driver="GPKG")

# Spatial-index-assisted conservative duplicate removal.
ordered=allpred.sort_values("score",ascending=False).reset_index(drop=True)
keep=[]; removed=[]
for i,r in ordered.iterrows():
    g=r.geometry; duplicate=False
    for j in keep:
        h=ordered.geometry.iloc[j]
        if not g.intersects(h):continue
        inter=g.intersection(h).area
        if inter<=0:continue
        iou=inter/(g.area+h.area-inter)
        containment=inter/min(g.area,h.area)
        ratio=min(g.area,h.area)/max(g.area,h.area)
        if iou>=DUP_IOU or (containment>=NESTED_CONTAINMENT and ratio>=SIMILAR_AREA_RATIO):
            duplicate=True;removed.append(dict(prompt_id=int(r.prompt_id),against=int(j),iou=iou,
                                                containment=containment,area_ratio=ratio))
            break
    if not duplicate:keep.append(i)
clean=ordered.iloc[keep].copy().reset_index(drop=True)
clean["crown_id"]=np.arange(1,len(clean)+1)
clean.to_file(work/"clean_crowns.gpkg",driver="GPKG")
pd.DataFrame(removed).to_csv(work/"removed_duplicates.csv",index=False)
print(f"Clean predictions: {len(clean)} | removed duplicates: {len(removed)}")

# Evaluate only reference crowns fully contained in the processed tile union
# (and ROI). This prevents partial smoke-test coverage being labelled full-ROI.
footprints=[box(t.xmin,t.ymin,t.xmax,t.ymax) for t in tm.itertuples()]
from shapely.ops import unary_union
covered=unary_union(footprints).intersection(roi19)
reference=reference_v2.to_crs(crs).copy()
reference=reference[reference.geometry.apply(covered.covers)].copy().reset_index(drop=True)
mat=np.zeros((len(reference),len(clean)),dtype=np.float32)
idx=clean.sindex
for i,g in enumerate(reference.geometry):
    for j in idx.query(g,predicate="intersects"):
        h=clean.geometry.iloc[j]; inter=g.intersection(h).area
        union=g.area+h.area-inter
        if union>0:mat[i,j]=inter/union
matches=[]
if mat.size:
    utility=(mat>=.5).astype(float)*1000+mat
    ii,jj=linear_sum_assignment(utility,maximize=True)
    matches=[(int(i),int(j),float(mat[i,j])) for i,j in zip(ii,jj) if mat[i,j]>=.5]
matched_ref={i for i,j,v in matches}
reference["area_m2"]=reference.geometry.area
reference["matched"]=[i in matched_ref for i in range(len(reference))]
reference["best_iou"]=mat.max(axis=1) if len(clean) else 0
reference["size_class"]=pd.cut(reference.area_m2,[0,20,40,80,np.inf],
                               labels=["<20","20–40","40–80",">80"])
size=reference.groupby("size_class",observed=True).agg(
    n=("matched","size"),matched=("matched","sum"),recall=("matched","mean"),
    median_best_iou=("best_iou","median"))
size.to_csv(work/"recall_by_size.csv")
display(size)
recall=len(matches)/len(reference) if len(reference) else float("nan")
precision=len(matches)/len(clean) if len(clean) else float("nan")
summary=dict(roi_index=ROI_INDEX,mode=run_tag,roi_area_ha=roi19.area/10000,
             tiles=len(tm),prompts=len(pm),manual_crowns=len(reference),
             raw_proposals=len(allpred),clean_predictions=len(clean),
             matched_iou_05=len(matches),recall_05=recall,precision_05=precision,
             median_matched_iou=float(np.median([v for _,_,v in matches])) if matches else None)
(work/"summary.json").write_text(json.dumps(summary,indent=2))
print("RESULTS:",json.dumps(summary,indent=2))

# Full ROI visualization, downsampled to avoid loading the giant raster.
with rasterio.open(RASTER) as src:
    from rasterio.windows import from_bounds
    win=from_bounds(*roi19.bounds,transform=src.transform).round_offsets().round_lengths()
    scale=max(1,int(math.ceil(max(win.width,win.height)/1600)))
    hh=max(1,int(win.height/scale));ww=max(1,int(win.width/scale))
    img=src.read([1,2,3],window=win,out_shape=(3,hh,ww),
                 resampling=Resampling.average).transpose(1,2,0)
    bb=rasterio.windows.bounds(win,src.transform)
fig,ax=plt.subplots(figsize=(12,12))
ax.imshow(img,extent=(bb[0],bb[2],bb[1],bb[3]),origin="upper")
gpd.GeoSeries([roi19],crs=crs).boundary.plot(ax=ax,color="white",linewidth=2)
if len(reference):reference.boundary.plot(ax=ax,color="yellow",linewidth=1)
clean.boundary.plot(ax=ax,color="cyan",linewidth=.8)
ax.set_title(f"ROI {ROI_INDEX} | {run_tag} | manual {len(reference)} | SAM3 {len(clean)} | recall@0.5 {recall:.1%}")
ax.set_aspect("equal");plt.tight_layout()
fig.savefig(work/"visual_QA.png",dpi=180,bbox_inches="tight")
plt.show()
print("BLOCK 19 COMPLETE:",work)


## 20 — Multiscale SAM3 pilot (40 m RGB tiles)

**Run:** 0R → Block 2 only if SAM3 environment missing → Block 14 → **Block 20**. Requires the saved **Block 19B** diagnostic CSV and Block 19 clean predictions. No need to rerun Blocks 19/19B. GPU for inference; cached masks allow resuming. Four representative pilot areas (selected using diagnostic reference results, not used as SAM3 prompt shapes), fixed 6 m grid, box sizes 8/12/16/20 m. Compare each scale and combined scales with the Block 19 baseline on the **same fully contained manual crowns**. Outputs saved to `experiments_v20_multiscale_roi_0/`.

**Scientific note:** pilot areas are deliberately selected, so results are diagnostic rather than unbiased site-wide accuracy estimates.

In [ ]:
# BLOCK 20 — Multiscale SAM3 pilot: 40m RGB context, 8/12/16/20m boxes
# Run 0R -> 2 (if needed) -> 14 -> 20. Does not rerun Block 19.
import os, sys, json, subprocess, math
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import Window
from rasterio.features import shapes
from rasterio.enums import Resampling
from shapely.geometry import Point, box, shape
from scipy.optimize import linear_sum_assignment
from PIL import Image
import matplotlib.pyplot as plt

ROI_INDEX=0
TILE_M=40.0
GRID_M=6.0
BOX_SIZES_M=(8.,12.,16.,20.)
N_AREAS=4
EDGE_MARGIN_M=0.4
MIN_AREA_M2,MAX_AREA_M2=2.,700.
DUP_IOU=.65
NESTED_CONTAINMENT=.95
SIMILAR_AREA_RATIO=.65
assert "reference_v2" in globals() and "roi_v2" in globals(), "Run Block 14"
assert Path(SAMPY).exists(), "Run Block 2 to install SAM3"
B19=OUT/f"experiments_v19_full_roi_{ROI_INDEX}"/"full"
DIAG=B19/"diagnostics_19B"/"crown_diagnostics_clean.csv"
assert DIAG.exists(), "Run Block 19B first; diagnostic CSV missing"
assert (B19/"clean_crowns.gpkg").exists(), "Block 19 clean predictions missing"
B20=OUT/f"experiments_v20_multiscale_roi_{ROI_INDEX}"
TILES=B20/"tiles";RAW20=B20/"raw"
for p in (B20,TILES,RAW20):p.mkdir(parents=True,exist_ok=True)

with rasterio.open(RASTER) as src:
    crs=src.crs;gsd=abs(src.transform.a)
    roi20=roi_v2.to_crs(crs).geometry.iloc[ROI_INDEX]
    ref=reference_v2.to_crs(crs)
    ref=ref[ref.geometry.apply(roi20.covers)].copy().reset_index(drop=True)
    d=pd.read_csv(DIAG)
    assert len(d)==len(ref),"Block 19B reference order/count differs; rerun 19B after Block 14"
    # Choose two large poorly recovered crowns, one medium low-IoU, one well-matched control.
    d=d.copy()
    d["group"]=np.select([
        (d.crown_area_m2>80)&(d.best_iou<.5),
        (d.crown_area_m2.between(20,80))&(d.best_iou<.5),
        d.best_iou>=.5],
        ["large_missed","medium_low_iou","matched_control"],default="other")
    targets=[]
    for group,quota in [("large_missed",2),("medium_low_iou",1),("matched_control",1)]:
        subset=d[d.group==group].sort_values(
            ["combined_coverage","crown_area_m2"],ascending=[True,False])
        for rr in subset.itertuples():
            p=ref.geometry.iloc[int(rr.crown_id)].representative_point()
            if all(p.distance(q["point"])>=TILE_M*.7 for q in targets):
                targets.append(dict(point=p,group=group,ref_id=int(rr.crown_id)))
                if sum(q["group"]==group for q in targets)>=quota:break
    if len(targets)<N_AREAS:
        for rr in d.sort_values("crown_area_m2",ascending=False).itertuples():
            p=ref.geometry.iloc[int(rr.crown_id)].representative_point()
            if all(p.distance(q["point"])>=TILE_M*.5 for q in targets):
                targets.append(dict(point=p,group="additional",ref_id=int(rr.crown_id)))
                if len(targets)>=N_AREAS:break
    targets=targets[:N_AREAS]
    assert targets,"No pilot target areas found"
    n=int(round(TILE_M/gsd))
    tiles=[]
    for i,t in enumerate(targets):
        rr,cc=src.index(t["point"].x,t["point"].y)
        win=Window(int(cc-n//2),int(rr-n//2),n,n)
        b=rasterio.windows.bounds(win,src.transform)
        png=TILES/f"area_{i:02d}.png"
        if not png.exists():
            rgb=src.read([1,2,3],window=win,boundless=True,fill_value=0).transpose(1,2,0)
            Image.fromarray(rgb.astype(np.uint8)).save(png)
        tiles.append(dict(area_id=i,group=t["group"],reference_id=t["ref_id"],
                          path=str(png),row_off=int(win.row_off),col_off=int(win.col_off),
                          xmin=b[0],ymin=b[1],xmax=b[2],ymax=b[3],n=n))
tm=pd.DataFrame(tiles);tm.to_csv(B20/"pilot_areas.csv",index=False)
print("Pilot areas:");display(tm[["area_id","group","reference_id"]])
# Same centres for all box sizes. 40m tiles: 6m grid positions 3,9,...,39m.
# Keep centres >= BOX_MAX/2 from tile edges so large boxes remain in the image.
offsets=np.arange(GRID_M/2,TILE_M,GRID_M)
offsets=[v for v in offsets if max(BOX_SIZES_M)/2<=v<=TILE_M-max(BOX_SIZES_M)/2]
rows=[]
for t in tm.itertuples():
    for dy in offsets:
        for dx in offsets:
            x,y=t.xmin+dx,t.ymax-dy
            if not roi20.covers(Point(x,y)):continue
            for sz in BOX_SIZES_M:
                rows.append(dict(area_id=int(t.area_id),path=t.path,row_off=int(t.row_off),
                                 col_off=int(t.col_off),n=n,gsd=gsd,
                                 cx=dx/TILE_M,cy=dy/TILE_M,box_m=sz))
pm=pd.DataFrame(rows)
assert len(pm),"No prompt centres inside selected areas"
pm.insert(0,"prompt_id",np.arange(len(pm)))
pm.to_csv(B20/"prompt_manifest.csv",index=False)
print(f"Prompts: {len(pm)} across {len(tm)} areas, {len(BOX_SIZES_M)} scales")
print("Existing masks:",sum((RAW20/f"prompt_{i:06d}.npz").exists() for i in pm.prompt_id))

runner=Path("/content/run_sam3_kvo20.py")
runner.write_text(r'''
import sys
from pathlib import Path
import numpy as np,pandas as pd,torch
from PIL import Image
from tqdm import tqdm
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
df=pd.read_csv(sys.argv[1]);dest=Path(sys.argv[2]);dest.mkdir(parents=True,exist_ok=True)
assert torch.cuda.is_available(),"SAM3 requires GPU"
torch.backends.cuda.matmul.allow_tf32=True
with torch.inference_mode(),torch.autocast("cuda",dtype=torch.bfloat16):
    model=build_sam3_image_model(bpe_path="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz")
    processor=Sam3Processor(model)
    for area,g in df.groupby("area_id",sort=True):
        todo=[r for r in g.itertuples() if not (dest/f"prompt_{int(r.prompt_id):06d}.npz").exists()]
        if not todo:continue
        img=Image.open(todo[0].path).convert("RGB")
        for r in tqdm(todo,desc=f"Area {area}",leave=False):
            state=processor.set_image(img)
            frac=float(r.box_m)/(float(r.n)*float(r.gsd))
            out=processor.add_geometric_prompt(
                box=np.array([[float(r.cx),float(r.cy),frac,frac]],dtype=np.float32),
                label=np.array([1],dtype=np.int32),state=state)
            masks=out["masks"].detach().cpu().numpy()
            scores=out["scores"].detach().float().cpu().numpy()
            if masks.ndim==4:masks=masks[:,0]
            if masks.ndim==2:masks=masks[None]
            np.savez_compressed(dest/f"prompt_{int(r.prompt_id):06d}.npz",
                                masks=(masks>0.5).astype(np.uint8),scores=scores)
        print(f"Area {area} complete",flush=True)
print("All done",flush=True)
''')
missing=sum(not (RAW20/f"prompt_{i:06d}.npz").exists() for i in pm.prompt_id)
if missing:
    run_env=os.environ.copy()
    if "env" in globals():run_env.update(env)
    subprocess.run([SAMPY,"-u",str(runner),str(B20/"prompt_manifest.csv"),str(RAW20)],
                   env=run_env,check=True)
else:print("All inference cached")

# Polygonize masks for every scale; keep all returned instances.
records=[]
with rasterio.open(RASTER) as src:
    for r in pm.itertuples():
        p=RAW20/f"prompt_{int(r.prompt_id):06d}.npz"
        if not p.exists():continue
        with np.load(p) as z:
            masks=z["masks"];scores=np.asarray(z["scores"]).reshape(-1)
        if masks.ndim==4:masks=masks[:,0]
        if masks.ndim==2:masks=masks[None]
        transform=src.window_transform(Window(int(r.col_off),int(r.row_off),int(r.n),int(r.n)))
        for k,m in enumerate(masks):
            m=np.asarray(m)>0
            if m.shape!=(int(r.n),int(r.n)) or not m.any():continue
            margin=max(1,int(round(EDGE_MARGIN_M/gsd)))
            if m[:margin].any() or m[-margin:].any() or m[:,:margin].any() or m[:,-margin:].any():continue
            geoms=[shape(g) for g,v in shapes(m.astype("uint8"),mask=m,transform=transform) if v==1]
            if not geoms:continue
            geom=max(geoms,key=lambda g:g.area)
            if not MIN_AREA_M2<=geom.area<=MAX_AREA_M2:continue
            if not roi20.covers(geom.representative_point()):continue
            records.append(dict(prompt_id=int(r.prompt_id),area_id=int(r.area_id),
                                box_m=float(r.box_m),instance=int(k),
                                score=float(scores[min(k,len(scores)-1)]),geometry=geom))
assert records,"No valid masks; inspect raw outputs"
pred=gpd.GeoDataFrame(records,geometry="geometry",crs=crs)
pred.to_file(B20/"all_multiscale_proposals.gpkg",driver="GPKG")

def dedup(gdf):
    ordered=gdf.sort_values("score",ascending=False).reset_index(drop=True)
    keep=[]
    for i,r in ordered.iterrows():
        g=r.geometry;dup=False
        for j in keep:
            h=ordered.geometry.iloc[j]
            if not g.intersects(h):continue
            inter=g.intersection(h).area
            if inter<=0:continue
            iou=inter/(g.area+h.area-inter)
            contain=inter/min(g.area,h.area)
            ratio=min(g.area,h.area)/max(g.area,h.area)
            if iou>=DUP_IOU or (contain>=NESTED_CONTAINMENT and ratio>=SIMILAR_AREA_RATIO):
                dup=True;break
        if not dup:keep.append(i)
    return ordered.iloc[keep].copy().reset_index(drop=True)

# Evaluation population: crowns fully contained in the ROI AND in at least
# one of the four 40m windows. Identical population for all scales + baseline.
pilot_footprints=[box(t.xmin,t.ymin,t.xmax,t.ymax) for t in tm.itertuples()]
refpilot=ref[ref.geometry.apply(lambda g:any(w.covers(g) for w in pilot_footprints))].copy().reset_index(drop=True)
print("Manual crowns fully inside pilot tiles:",len(refpilot))

def match_metrics(reference,predictions):
    nref,npred=len(reference),len(predictions)
    mat=np.zeros((nref,npred),dtype=np.float32)
    if nref and npred:
        idx=predictions.sindex
        for i,g in enumerate(reference.geometry):
            for j in idx.query(g,predicate="intersects"):
                h=predictions.geometry.iloc[j];a=g.intersection(h).area
                if a>0:mat[i,j]=a/(g.area+h.area-a)
        utility=(mat>=.5).astype(float)*1000+mat
        ii,jj=linear_sum_assignment(utility,maximize=True)
        matches=[float(mat[i,j]) for i,j in zip(ii,jj) if mat[i,j]>=.5]
    else:matches=[]
    return dict(manual=nref,predictions=npred,matched=len(matches),
                recall=len(matches)/nref if nref else np.nan,
                precision=len(matches)/npred if npred else np.nan,
                median_matched_iou=float(np.median(matches)) if matches else np.nan,
                large_recall=float((mat[refpilot.geometry.area.to_numpy()>80].max(axis=1)>=.5).mean())
                  if npred and (refpilot.geometry.area>80).any() else np.nan)

# Keep only proposals whose representative points fall inside pilot windows.
# Same geographic scope for baseline and new multiscale proposals.
def in_pilot(g):
    return any(w.covers(g.representative_point()) for w in pilot_footprints)

baseline=gpd.read_file(B19/"clean_crowns.gpkg").to_crs(crs)
baseline=baseline[baseline.geometry.apply(in_pilot)].copy().reset_index(drop=True)
results=[dict(scale="Block19 baseline 10m/24m",**match_metrics(refpilot,baseline))]
for sz in BOX_SIZES_M:
    subset=pred[np.isclose(pred.box_m,sz)].copy()
    clean=dedup(subset)
    clean.to_file(B20/f"clean_{int(sz)}m.gpkg",driver="GPKG")
    results.append(dict(scale=f"{int(sz)}m",**match_metrics(refpilot,clean)))
combined=dedup(pred)
combined.to_file(B20/"clean_multiscale_combined.gpkg",driver="GPKG")
results.append(dict(scale="all scales combined",**match_metrics(refpilot,combined)))
result=pd.DataFrame(results)
result.to_csv(B20/"scale_comparison.csv",index=False)
display(result.round(3))
fig,ax=plt.subplots(figsize=(10,5))
ax.bar(result.scale,result.recall)
ax.set_ylabel("Recall at IoU ≥ 0.5");ax.set_ylim(0,1)
ax.tick_params(axis="x",rotation=35);plt.tight_layout()
fig.savefig(B20/"scale_recall.png",dpi=160,bbox_inches="tight")
plt.show()
print("BLOCK 20 COMPLETE:",B20)


## 20B — Actual multiscale RGB/SAM3 visual QA

**Run 0R → 14 → 20B.** CPU only. Plots actual orthomosaic pixels, manual crown boundaries and the saved Block 19/20 predictions at identical extents. Produces four comparison figures in `experiments_v20_multiscale_roi_0/visual_QA_20B/`. No model inference.


In [ ]:
# BLOCK 20B — TRUE RGB / SAM3 multiscale visual QA (CPU only)
# Run 0R -> 14 -> 20B. Does NOT run SAM3.
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from shapely.geometry import box
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ROI_INDEX=0
B19=OUT/f"experiments_v19_full_roi_{ROI_INDEX}"/"full"
B20=OUT/f"experiments_v20_multiscale_roi_{ROI_INDEX}"
VIS=B20/"visual_QA_20B";VIS.mkdir(parents=True,exist_ok=True)
areas=pd.read_csv(B20/"pilot_areas.csv")
assert len(areas),"No pilot areas"
paths=[
    ("Block 19 (10 m, 24 m tiles)",B19/"clean_crowns.gpkg"),
    ("8 m",B20/"clean_8m.gpkg"),
    ("12 m",B20/"clean_12m.gpkg"),
    ("16 m",B20/"clean_16m.gpkg"),
    ("20 m",B20/"clean_20m.gpkg"),
    ("Combined",B20/"clean_multiscale_combined.gpkg"),
]
for label,path in paths:
    assert path.exists(),f"Missing {label}: {path}. Run Block 20."
with rasterio.open(RASTER) as src:
    crs=src.crs
    preds=[(label,gpd.read_file(path).to_crs(crs)) for label,path in paths]
    reference=reference_v2.to_crs(crs)
    roi20=roi_v2.to_crs(crs).geometry.iloc[ROI_INDEX]
    reference=reference[reference.geometry.intersects(roi20)].copy()
    records=[]
    for area in areas.itertuples():
        footprint=box(float(area.xmin),float(area.ymin),float(area.xmax),float(area.ymax))
        # Same exact raster window for every column in a row.
        win=Window(int(area.col_off),int(area.row_off),int(area.n),int(area.n))
        rgb=src.read([1,2,3],window=win,out_shape=(3,800,800),
                     resampling=Resampling.average,boundless=True,fill_value=0)
        rgb=np.moveaxis(rgb,0,-1).astype(np.uint8)
        extent=(area.xmin,area.xmax,area.ymin,area.ymax)
        ref_area=reference[reference.geometry.intersects(footprint)]
        fig,axes=plt.subplots(1,6,figsize=(25,4.8),sharex=True,sharey=True)
        for ax,(label,gdf) in zip(axes,preds):
            subset=gdf[gdf.geometry.intersects(footprint)]
            ax.imshow(rgb,extent=extent,origin="upper")
            if len(ref_area):
                ref_area.boundary.plot(ax=ax,color="yellow",linewidth=1.0,zorder=3)
            if len(subset):
                subset.boundary.plot(ax=ax,color="cyan",linewidth=.8,zorder=4)
            ax.set_xlim(area.xmin,area.xmax);ax.set_ylim(area.ymin,area.ymax)
            ax.set_aspect("equal");ax.set_title(f"{label}\n{len(subset)} proposals",fontsize=10)
            ax.set_xticks([]);ax.set_yticks([])
            records.append(dict(area_id=int(area.area_id),group=area.group,
                                scale=label,visible_predictions=len(subset),
                                visible_reference=len(ref_area)))
        fig.suptitle(f"Baego ROI {ROI_INDEX} | Pilot area {area.area_id}: {area.group} "
                     f"| reference ID {area.reference_id}",fontsize=13)
        fig.legend(handles=[
            Line2D([0],[0],color="yellow",lw=2,label="Manual crown boundaries"),
            Line2D([0],[0],color="cyan",lw=2,label="SAM3 crown proposals")
        ],loc="lower center",ncol=2,bbox_to_anchor=(.5,-.02))
        fig.tight_layout(rect=(0,.06,1,.94))
        p=VIS/f"area_{int(area.area_id):02d}_real_RGB_comparison.png"
        fig.savefig(p,dpi=190,bbox_inches="tight")
        plt.show()
        plt.close(fig)
        print("Saved:",p)
pd.DataFrame(records).to_csv(VIS/"visible_polygon_counts.csv",index=False)
print("\nBLOCK 20B COMPLETE — real RGB and actual SAM3 polygons")
print("Figures:",VIS)


## 21 — DSM-assisted SAM3 crown quality assessment

**Run 0R → 14 → 21.** CPU only. Uses saved Block 19/20 polygons and DSM, aligns DSM to RGB with WarpedVRT, calculates geometry, DSM local contrast, nesting and overlap, and evaluates cautious quality filters against fully contained manual reference crowns in the same pilot areas. No SAM3 rerun. Outputs in `experiments_v21_DSM_quality_roi_0/`. DSM is surface elevation, not a terrain-normalized canopy height model. Flags are exploratory review priorities, not validated quality probabilities.


In [ ]:
# BLOCK 21 — DSM-assisted crown quality assessment (CPU only)
# Run 0R -> 14 -> 21. No SAM3 rerun. No modifications to source polygons.
from pathlib import Path
import math, json
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.vrt import WarpedVRT
from rasterio.windows import from_bounds
from rasterio.features import geometry_mask
from rasterio.enums import Resampling
from shapely.geometry import box
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ROI_INDEX=0
B19=OUT/f"experiments_v19_full_roi_{ROI_INDEX}"/"full"
B20=OUT/f"experiments_v20_multiscale_roi_{ROI_INDEX}"
B21=OUT/f"experiments_v21_DSM_quality_roi_{ROI_INDEX}"
B21.mkdir(parents=True,exist_ok=True)
DSM=GEO/"Baego_DSM.tif"
assert DSM.exists(),"Baego DSM missing"
assert "roi_v2" in globals() and "reference_v2" in globals(),"Run Block 14"
sources={
    "baseline_19":B19/"clean_crowns.gpkg",
    "scale_8m":B20/"clean_8m.gpkg",
    "multiscale":B20/"clean_multiscale_combined.gpkg",
}
for key,path in sources.items():assert path.exists(),f"Missing {key}: {path}"
areas=pd.read_csv(B20/"pilot_areas.csv")
assert len(areas),"Block 20 pilot areas missing"

# Quality metrics calculated from polygon geometry and aligned DSM.
# DSM is SURFACE ELEVATION, not canopy height; local contrasts are proxies only.
# We sample a local 2m exterior ring for relative elevation.
def dsm_metrics(g, vrt, transform, pixel_size, buffer_m=2.0):
    minx,miny,maxx,maxy=g.buffer(buffer_m).bounds
    win=from_bounds(minx,miny,maxx,maxy,transform=transform)
    win=win.round_offsets().round_lengths()
    if win.width<2 or win.height<2:return np.nan,np.nan,np.nan,np.nan,0
    # Prevent pathological reads for very large polygons.
    if win.width*win.height>2_000_000:return np.nan,np.nan,np.nan,np.nan,0
    z=vrt.read(1,window=win,masked=True)
    wt=rasterio.windows.transform(win,transform)
    inside=geometry_mask([g],out_shape=z.shape,transform=wt,invert=True)
    outer=geometry_mask([g.buffer(buffer_m)],out_shape=z.shape,transform=wt,invert=True)
    arr=np.asarray(z,dtype=float)
    valid=np.isfinite(arr)&(~np.ma.getmaskarray(z))&(arr>0)
    a=arr[inside&valid];b=arr[outer&~inside&valid]
    if len(a)<5:return np.nan,np.nan,np.nan,np.nan,len(a)
    med=float(np.median(a))
    spread=float(np.percentile(a,90)-np.percentile(a,10))
    contrast=med-float(np.median(b)) if len(b)>=5 else np.nan
    rough=float(np.std(a))
    return med,spread,contrast,rough,len(a)

def quality_dataset(gdf, vrt, transform, pixel_size):
    gdf=gdf.copy().reset_index(drop=True)
    gdf["area_m2"]=gdf.geometry.area
    gdf["perimeter_m"]=gdf.geometry.length
    gdf["compactness"]=4*np.pi*gdf.area_m2/(gdf.perimeter_m**2).clip(lower=1e-6)
    gdf["equiv_diameter_m"]=2*np.sqrt(gdf.area_m2/np.pi)
    metrics=[dsm_metrics(g,vrt,transform,pixel_size) for g in gdf.geometry]
    gdf[["dsm_median","dsm_p90_p10","dsm_local_contrast",
         "dsm_std","dsm_valid_pixels"]]=pd.DataFrame(metrics,index=gdf.index)
    # Pairwise spatial overlap and containment (not just IoU).
    si=gdf.sindex
    overlap=np.zeros(len(gdf),dtype=int)
    nested=np.zeros(len(gdf),dtype=int)
    max_iou=np.zeros(len(gdf),dtype=float)
    for i,g in enumerate(gdf.geometry):
        for j in si.query(g,predicate="intersects"):
            j=int(j)
            if j<=i:continue
            h=gdf.geometry.iloc[j]
            inter=g.intersection(h).area
            if inter<=0:continue
            union=g.area+h.area-inter
            iou=inter/union
            overlap[i]+=1;overlap[j]+=1
            max_iou[i]=max(max_iou[i],iou)
            max_iou[j]=max(max_iou[j],iou)
            if inter/min(g.area,h.area)>=.95:
                nested[i]+=1;nested[j]+=1
    gdf["overlap_count"]=overlap
    gdf["nested_count"]=nested
    gdf["max_pair_iou"]=max_iou
    # Interpretable, deliberately cautious flags; no reference labels used.
    # No absolute elevation/height threshold: terrain may vary.
    area_flag=(gdf.area_m2<5)
    shape_flag=(gdf.compactness<.12)
    redundancy_flag=(gdf.nested_count>=2)|(gdf.max_pair_iou>=.55)
    low_contrast=(gdf.dsm_local_contrast.notna())&(gdf.dsm_local_contrast<-.75)
    gdf["flag_tiny"]=area_flag.astype(int)
    gdf["flag_irregular"]=shape_flag.astype(int)
    gdf["flag_redundant"]=redundancy_flag.astype(int)
    gdf["flag_low_relative_DSM"]=low_contrast.astype(int)
    gdf["flag_count"]=gdf[["flag_tiny","flag_irregular",
                           "flag_redundant","flag_low_relative_DSM"]].sum(axis=1)
    gdf["quality_class"]=np.select(
        [gdf.flag_count>=2,gdf.flag_count==1],
        ["likely_artifact_review","uncertain"],default="higher_priority")
    # Note: these are review priorities, not classifier probabilities.
    return gdf

def evaluate(ref,pred):
    mat=np.zeros((len(ref),len(pred)),dtype=np.float32)
    if len(ref) and len(pred):
        ix=pred.sindex
        for i,g in enumerate(ref.geometry):
            for j in ix.query(g,predicate="intersects"):
                h=pred.geometry.iloc[j];a=g.intersection(h).area
                if a>0:mat[i,j]=a/(g.area+h.area-a)
        utility=(mat>=.5).astype(float)*1000+mat
        ii,jj=linear_sum_assignment(utility,maximize=True)
        matched=sum(mat[i,j]>=.5 for i,j in zip(ii,jj))
    else:matched=0
    return dict(manual=len(ref),predictions=len(pred),matched=int(matched),
                recall=matched/len(ref) if len(ref) else np.nan,
                precision=matched/len(pred) if len(pred) else np.nan)

with rasterio.open(RASTER) as rgb, rasterio.open(DSM) as dsm:
    crs=rgb.crs;roi21=roi_v2.to_crs(crs).geometry.iloc[ROI_INDEX]
    ref=reference_v2.to_crs(crs)
    # Evaluation on the exact same 40m pilot tiles as Block 20.
    footprints=[box(float(t.xmin),float(t.ymin),float(t.xmax),float(t.ymax))
                for t in areas.itertuples()]
    ref=ref[ref.geometry.apply(
        lambda g:roi21.covers(g) and any(w.covers(g) for w in footprints)
    )].copy().reset_index(drop=True)
    print("Pilot reference crowns:",len(ref))
    # Align DSM to RGB pixels using a VRT (accounts for one-pixel shift).
    with WarpedVRT(dsm,crs=crs,transform=rgb.transform,
                   width=rgb.width,height=rgb.height,
                   resampling=Resampling.nearest) as vrt:
        datasets={}
        results=[]
        for name,path in sources.items():
            original=gpd.read_file(path).to_crs(crs)
            # Same pilot geography for all three sources.
            subset=original[original.geometry.apply(
                lambda g:any(w.covers(g.representative_point()) for w in footprints)
            )].copy().reset_index(drop=True)
            scored=quality_dataset(subset,vrt,rgb.transform,abs(rgb.transform.a))
            scored.to_file(B21/f"{name}_quality.gpkg",driver="GPKG")
            scored.drop(columns="geometry").to_csv(B21/f"{name}_quality.csv",index=False)
            datasets[name]=scored
            for filter_name,mask in [
                ("unfiltered",np.ones(len(scored),dtype=bool)),
                ("exclude_2plus_flags",scored.flag_count<2),
                ("higher_priority_only",scored.flag_count==0)]:
                selected=scored.loc[mask].copy().reset_index(drop=True)
                results.append(dict(source=name,filter=filter_name,
                                    **evaluate(ref,selected)))
        comparison=pd.DataFrame(results)
        comparison.to_csv(B21/"filter_sensitivity.csv",index=False)
        display(comparison.round(3))
        for area in areas.itertuples():
            win=rasterio.windows.Window(int(area.col_off),int(area.row_off),
                                        int(area.n),int(area.n))
            chip=rgb.read([1,2,3],window=win,out_shape=(3,850,850),
                          resampling=Resampling.average,boundless=True,fill_value=0)
            chip=np.moveaxis(chip,0,-1).astype(np.uint8)
            ext=(area.xmin,area.xmax,area.ymin,area.ymax)
            foot=box(area.xmin,area.ymin,area.xmax,area.ymax)
            fig,axes=plt.subplots(1,3,figsize=(16,5.6),sharex=True,sharey=True)
            for ax,(name,df) in zip(axes,datasets.items()):
                ax.imshow(chip,extent=ext,origin="upper")
                rr=ref[ref.geometry.intersects(foot)]
                if len(rr):rr.boundary.plot(ax=ax,color="yellow",linewidth=1.1,zorder=4)
                sub=df[df.geometry.intersects(foot)]
                for cls,color in [("higher_priority","lime"),
                                  ("uncertain","cyan"),
                                  ("likely_artifact_review","red")]:
                    layer=sub[sub.quality_class==cls]
                    if len(layer):layer.boundary.plot(ax=ax,color=color,linewidth=.9,zorder=5)
                ax.set_title(f"{name}\n{len(sub)} visible polygons",fontsize=11)
                ax.set_xlim(area.xmin,area.xmax);ax.set_ylim(area.ymin,area.ymax)
                ax.set_xticks([]);ax.set_yticks([]);ax.set_aspect("equal")
            fig.legend(handles=[
                Line2D([0],[0],color="yellow",lw=2,label="Manual reference"),
                Line2D([0],[0],color="lime",lw=2,label="Higher priority"),
                Line2D([0],[0],color="cyan",lw=2,label="Uncertain"),
                Line2D([0],[0],color="red",lw=2,label="Likely artifact — review")
            ],loc="lower center",ncol=4)
            fig.suptitle(f"Block 21 — area {area.area_id} ({area.group})")
            fig.tight_layout(rect=(0,.07,1,.94))
            fig.savefig(B21/f"area_{int(area.area_id):02d}_quality.png",
                        dpi=170,bbox_inches="tight")
            plt.show();plt.close(fig)
print("BLOCK 21 COMPLETE:",B21)
print("Review filter_sensitivity.csv and quality figures before discarding any polygons.")


## 22 — Full ROI RGB-only 8 m SAM3 and full manual-outline validation

**Inference:** GPU Colab, run `0R → 2 (if needed) → 14 → 22`. Resumable, 40 m RGB tiles, 8 m box prompts, 6 m grid, 10 m overlap. Saves masks and polygon outputs separately from Block 19. **Validation:** CPU, run `0R → 14 → 22V` after full inference. Manual `Crown_Kristof.shp` outlines are used only for evaluation, never for prompts. ROI from `ROI_Crowns_Kvo.shp`. Both methods evaluated against the identical fully-contained manual crowns; no DSM. Compare recall at IoU 0.3/0.5/0.7, matched crown area bias, fragmentation and size classes. Unmatched predictions are not automatically false positives.


In [ ]:
# BLOCK 22 — Full ROI RGB SAM3, 8m boxes / 40m tiles (resumable)
# Run after 0R, 2 (if environment absent), 14. No need for 0S or 15–18.
import os, sys, subprocess, json, math
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import Window
from rasterio.features import shapes
from rasterio.enums import Resampling
from shapely.geometry import box, Point, shape
from scipy.optimize import linear_sum_assignment
from PIL import Image
import matplotlib.pyplot as plt

ROI_INDEX = 0
TILE_M = 40.0
OVERLAP_M = 10.0
GRID_M = 6.0
BOX_M = 8.0
MAX_TILES = None       # None = complete ROI; 4 = technical smoke test
MIN_AREA_M2, MAX_AREA_M2 = 2.0, 700.0
EDGE_MARGIN_M = 0.4
DUP_IOU = 0.65
NESTED_CONTAINMENT = 0.95
SIMILAR_AREA_RATIO = 0.65

assert "roi_v2" in globals() and "reference_v2" in globals(), "Run Block 14"
assert Path(SAMPY).exists(), "Run Block 2 to install isolated SAM3 environment"
B19 = OUT / f"experiments_v22_full_roi_8m_{ROI_INDEX}"
TILES, RAW19 = B19/"tiles", B19/"raw"
for d in (B19,TILES,RAW19): d.mkdir(parents=True,exist_ok=True)

with rasterio.open(RASTER) as src:
    crs, gsd = src.crs, abs(src.transform.a)
    roi19 = roi_v2.to_crs(crs).geometry.iloc[ROI_INDEX]
    n = int(round(TILE_M/gsd))
    stride = max(1,int(round((TILE_M-OVERLAP_M)/gsd)))
    minx,miny,maxx,maxy = roi19.bounds
    r0,c0 = src.index(minx,maxy)
    r1,c1 = src.index(maxx,miny)
    def starts(lo,hi):
        if hi-lo <= n: return [int(round((lo+hi-n)/2))]
        seq=list(range(int(lo),int(hi)-n+1,stride))
        last=int(hi)-n
        if not seq or seq[-1]!=last: seq.append(last)
        return sorted(set(seq))
    tiles=[]
    for rr in starts(r0,r1+1):
        for cc in starts(c0,c1+1):
            win=Window(cc,rr,n,n)
            b=rasterio.windows.bounds(win,src.transform)
            if not box(*b).intersects(roi19): continue
            tid=len(tiles); path=TILES/f"tile_{tid:04d}.png"
            tiles.append(dict(tile_id=tid,row_off=rr,col_off=cc,width=n,height=n,
                              xmin=b[0],ymin=b[1],xmax=b[2],ymax=b[3],path=str(path)))
    if MAX_TILES is not None: tiles=tiles[:MAX_TILES]
    assert tiles,"No ROI tiles"
    for t in tiles:
        p=Path(t["path"])
        if p.exists(): continue
        win=Window(t["col_off"],t["row_off"],n,n)
        rgb=src.read([1,2,3],window=win,boundless=True,fill_value=0).transpose(1,2,0)
        Image.fromarray(rgb.astype(np.uint8)).save(p)
tm=pd.DataFrame(tiles)
# Scope outputs by the selected tile IDs so smoke tests never masquerade as full ROI.
run_tag="full" if MAX_TILES is None else f"first_{MAX_TILES}_tiles"
work=B19/run_tag; raw=work/"raw"; raw.mkdir(parents=True,exist_ok=True)
rows=[]
for t in tm.itertuples():
    for dy in np.arange(BOX_M/2,TILE_M-BOX_M/2+1e-6,GRID_M):
        for dx in np.arange(GRID_M/2,TILE_M,GRID_M):
            x,y=t.xmin+dx,t.ymax-dy
            if roi19.covers(Point(x,y)):
                rows.append(dict(tile_id=int(t.tile_id),path=t.path,cx=dx/TILE_M,cy=dy/TILE_M,
                                 box_m=BOX_M,width=n,height=n,gsd=gsd,
                                 row_off=int(t.row_off),col_off=int(t.col_off)))
pm=pd.DataFrame(rows)
assert len(pm),"No grid prompts inside ROI"
pm.insert(0,"prompt_id",np.arange(len(pm)))
pm.to_csv(work/"prompt_manifest.csv",index=False)
print(f"ROI {ROI_INDEX}: {roi19.area/10000:.3f} ha | tiles={len(tm)} | prompts={len(pm)} | mode={run_tag}")
print("Existing masks:",sum((raw/f"prompt_{i:06d}.npz").exists() for i in pm.prompt_id))

runner=Path("/content/run_sam3_kvo22.py")
runner.write_text(r'''
import sys
from pathlib import Path
import numpy as np,pandas as pd,torch
from PIL import Image
from tqdm import tqdm
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
df=pd.read_csv(sys.argv[1]); outdir=Path(sys.argv[2]);outdir.mkdir(parents=True,exist_ok=True)
if not torch.cuda.is_available(): raise RuntimeError("CUDA GPU required")
torch.backends.cuda.matmul.allow_tf32=True
bpe="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz"
with torch.inference_mode(),torch.autocast("cuda",dtype=torch.bfloat16):
    model=build_sam3_image_model(bpe_path=bpe)
    processor=Sam3Processor(model)
    for tid,g in df.groupby("tile_id",sort=True):
        pending=[r for r in g.itertuples() if not (outdir/f"prompt_{int(r.prompt_id):06d}.npz").exists()]
        if not pending:continue
        img=Image.open(pending[0].path).convert("RGB")
        for r in tqdm(pending,desc=f"Tile {tid}",leave=False):
            # Fresh prompt state per box, to avoid accumulated prompts.
            state=processor.set_image(img)
            frac=min(.95,float(r.box_m)/(float(r.width)*float(r.gsd)))
            out=processor.add_geometric_prompt(
                box=np.array([[float(r.cx),float(r.cy),frac,frac]],dtype=np.float32),
                label=np.array([1],dtype=np.int32),state=state)
            masks=out["masks"].detach().cpu().numpy()
            scores=out["scores"].detach().float().cpu().numpy()
            if masks.ndim==4:masks=masks[:,0]
            if masks.ndim==2:masks=masks[None]
            np.savez_compressed(outdir/f"prompt_{int(r.prompt_id):06d}.npz",
                                masks=(masks>0.5).astype(np.uint8),scores=scores)
        print(f"Tile {tid} complete",flush=True)
print("Inference complete",flush=True)
''')
missing=sum(not (raw/f"prompt_{i:06d}.npz").exists() for i in pm.prompt_id)
if missing:
    run_env=os.environ.copy()
    if "env" in globals(): run_env.update(env)
    print("Running SAM3 for",missing,"unsaved prompts")
    subprocess.run([SAMPY,"-u",str(runner),str(work/"prompt_manifest.csv"),str(raw)],
                   env=run_env,check=True)
else: print("All masks cached; skipping SAM3")

# Convert every returned instance, not just the highest-scoring mask.
features=[]
with rasterio.open(RASTER) as src:
    for r in pm.itertuples():
        p=raw/f"prompt_{int(r.prompt_id):06d}.npz"
        if not p.exists():continue
        with np.load(p) as z:
            masks=z["masks"]; scores=np.asarray(z["scores"]).reshape(-1)
        if masks.ndim==4:masks=masks[:,0]
        if masks.ndim==2:masks=masks[None]
        win=Window(int(r.col_off),int(r.row_off),int(r.width),int(r.height))
        transform=src.window_transform(win)
        for k,m in enumerate(masks):
            m=np.asarray(m)>0
            if m.shape!=(n,n) or not m.any():continue
            margin=max(1,int(round(EDGE_MARGIN_M/gsd)))
            if m[:margin].any() or m[-margin:].any() or m[:,:margin].any() or m[:,-margin:].any():
                continue
            geoms=[shape(g) for g,v in shapes(m.astype("uint8"),mask=m,transform=transform) if v==1]
            if not geoms:continue
            geom=max(geoms,key=lambda g:g.area)
            if not (MIN_AREA_M2<=geom.area<=MAX_AREA_M2):continue
            if not roi19.covers(geom.representative_point()):continue
            features.append(dict(prompt_id=int(r.prompt_id),tile_id=int(r.tile_id),
                                 instance=int(k),score=float(scores[min(k,len(scores)-1)]),
                                 area_m2=float(geom.area),geometry=geom))
print("Valid raw proposals:",len(features))
assert features,"No polygons survived filtering; inspect tile-edge masks"
allpred=gpd.GeoDataFrame(features,geometry="geometry",crs=crs)
allpred.to_file(work/"all_proposals.gpkg",driver="GPKG")

# Spatial-index-assisted conservative duplicate removal.
ordered=allpred.sort_values("score",ascending=False).reset_index(drop=True)
keep=[]; removed=[]
for i,r in ordered.iterrows():
    g=r.geometry; duplicate=False
    for j in keep:
        h=ordered.geometry.iloc[j]
        if not g.intersects(h):continue
        inter=g.intersection(h).area
        if inter<=0:continue
        iou=inter/(g.area+h.area-inter)
        containment=inter/min(g.area,h.area)
        ratio=min(g.area,h.area)/max(g.area,h.area)
        if iou>=DUP_IOU or (containment>=NESTED_CONTAINMENT and ratio>=SIMILAR_AREA_RATIO):
            duplicate=True;removed.append(dict(prompt_id=int(r.prompt_id),against=int(j),iou=iou,
                                                containment=containment,area_ratio=ratio))
            break
    if not duplicate:keep.append(i)
clean=ordered.iloc[keep].copy().reset_index(drop=True)
clean["crown_id"]=np.arange(1,len(clean)+1)
clean.to_file(work/"clean_crowns.gpkg",driver="GPKG")
pd.DataFrame(removed).to_csv(work/"removed_duplicates.csv",index=False)
print(f"Clean predictions: {len(clean)} | removed duplicates: {len(removed)}")

# Evaluate only reference crowns fully contained in the processed tile union
# (and ROI). This prevents partial smoke-test coverage being labelled full-ROI.
footprints=[box(t.xmin,t.ymin,t.xmax,t.ymax) for t in tm.itertuples()]
from shapely.ops import unary_union
covered=unary_union(footprints).intersection(roi19)
reference=reference_v2.to_crs(crs).copy()
reference=reference[reference.geometry.apply(covered.covers)].copy().reset_index(drop=True)
mat=np.zeros((len(reference),len(clean)),dtype=np.float32)
idx=clean.sindex
for i,g in enumerate(reference.geometry):
    for j in idx.query(g,predicate="intersects"):
        h=clean.geometry.iloc[j]; inter=g.intersection(h).area
        union=g.area+h.area-inter
        if union>0:mat[i,j]=inter/union
matches=[]
if mat.size:
    utility=(mat>=.5).astype(float)*1000+mat
    ii,jj=linear_sum_assignment(utility,maximize=True)
    matches=[(int(i),int(j),float(mat[i,j])) for i,j in zip(ii,jj) if mat[i,j]>=.5]
matched_ref={i for i,j,v in matches}
reference["area_m2"]=reference.geometry.area
reference["matched"]=[i in matched_ref for i in range(len(reference))]
reference["best_iou"]=mat.max(axis=1) if len(clean) else 0
reference["size_class"]=pd.cut(reference.area_m2,[0,20,40,80,np.inf],
                               labels=["<20","20–40","40–80",">80"])
size=reference.groupby("size_class",observed=True).agg(
    n=("matched","size"),matched=("matched","sum"),recall=("matched","mean"),
    median_best_iou=("best_iou","median"))
size.to_csv(work/"recall_by_size.csv")
display(size)
recall=len(matches)/len(reference) if len(reference) else float("nan")
precision=len(matches)/len(clean) if len(clean) else float("nan")
summary=dict(roi_index=ROI_INDEX,mode=run_tag,roi_area_ha=roi19.area/10000,
             tiles=len(tm),prompts=len(pm),manual_crowns=len(reference),
             raw_proposals=len(allpred),clean_predictions=len(clean),
             matched_iou_05=len(matches),recall_05=recall,precision_05=precision,
             median_matched_iou=float(np.median([v for _,_,v in matches])) if matches else None)
(work/"summary.json").write_text(json.dumps(summary,indent=2))
print("RESULTS:",json.dumps(summary,indent=2))

print("BLOCK 22 COMPLETE. Run Block 22V for full statistical validation:",work)


## 22V — Statistical validation (no GPU)

Run after Block 22 completes. No SAM3 rerun; exports CSVs in `experiments_v22_full_roi_8m_0/full/validation/`.


In [ ]:
# BLOCK 22V — Full ROI statistical validation: manual crowns vs SAM3
# CPU only; run 0R -> 14 -> 22V after Block 22. No SAM3 inference.
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd
from scipy.optimize import linear_sum_assignment
from shapely.ops import unary_union
from shapely.geometry import box
import json
ROI_INDEX=0
NEW=OUT/f"experiments_v22_full_roi_8m_{ROI_INDEX}"/"full"
OLD=OUT/f"experiments_v19_full_roi_{ROI_INDEX}"/"full"
V=NEW/"validation";V.mkdir(parents=True,exist_ok=True)
assert (NEW/"clean_crowns.gpkg").exists(),"Run Block 22 full inference first"
assert (OLD/"clean_crowns.gpkg").exists(),"Block 19 baseline missing"
assert "reference_v2" in globals() and "roi_v2" in globals(),"Run Block 14"
with rasterio.open(RASTER) as src:crs=src.crs
roi=roi_v2.to_crs(crs).geometry.iloc[ROI_INDEX]
ref=reference_v2.to_crs(crs)
# Only fully-contained reference crowns, common to both full-ROI runs.
ref=ref[ref.geometry.apply(roi.covers)].copy().reset_index(drop=True)
ref["manual_id"]=np.arange(len(ref))
ref["manual_area_m2"]=ref.geometry.area
ref["size_class"]=pd.cut(ref.manual_area_m2,[0,20,40,80,np.inf],
                           labels=["<20","20-40","40-80",">80"])
thresholds=[.3,.5,.7]
records=[];size_rows=[];manual_rows=[];match_rows=[];pred_rows=[]
for name,source in [("Block19_10m_24m",OLD),("Block22_8m_40m",NEW)]:
    p=gpd.read_file(source/"clean_crowns.gpkg").to_crs(crs).reset_index(drop=True)
    # Filter prediction representative points to ROI consistently.
    p=p[p.geometry.apply(lambda g:roi.covers(g.representative_point()))].reset_index(drop=True)
    ix=p.sindex
    mat=np.zeros((len(ref),len(p)),dtype=np.float32)
    overlap_counts=np.zeros(len(ref),dtype=int)
    covered=np.zeros(len(ref),dtype=float)
    best_pred=np.full(len(ref),-1,dtype=int)
    # Coverage union is calculated per manual crown, not by summing overlaps.
    for i,g in enumerate(ref.geometry):
        js=list(ix.query(g,predicate="intersects"))
        fragments=[]
        for j in js:
            h=p.geometry.iloc[j];inter=g.intersection(h)
            a=inter.area
            if a<=0:continue
            mat[i,j]=a/(g.area+h.area-a)
            if a/g.area>=.05:overlap_counts[i]+=1
            fragments.append(inter)
        if fragments:covered[i]=unary_union(fragments).area/g.area
        if len(p):best_pred[i]=int(np.argmax(mat[i]))
    best=mat.max(axis=1) if len(p) else np.zeros(len(ref))
    for thr in thresholds:
        if mat.size:
            utility=(mat>=thr).astype(float)*1000+mat
            ii,jj=linear_sum_assignment(utility,maximize=True)
            matches=[(int(i),int(j),float(mat[i,j])) for i,j in zip(ii,jj) if mat[i,j]>=thr]
        else:matches=[]
        records.append(dict(model=name,threshold=thr,manual=len(ref),predictions=len(p),
                            matched=len(matches),recall=len(matches)/len(ref) if len(ref) else np.nan,
                            reference_matched_prediction_fraction=len(matches)/len(p) if len(p) else np.nan,
                            median_matched_iou=np.median([v for _,_,v in matches]) if matches else np.nan))
        for cls,group in ref.groupby("size_class",observed=True):
            ids=set(group.index)
            m=sum(i in ids for i,j,v in matches)
            size_rows.append(dict(model=name,threshold=thr,size_class=str(cls),
                                  manual=len(group),matched=m,recall=m/len(group)))
        if thr==.5:
            for i,j,iou in matches:
                a=ref.geometry.iloc[i].area;b=p.geometry.iloc[j].area
                match_rows.append(dict(model=name,manual_id=i,pred_id=j,iou=iou,
                                       manual_area_m2=a,pred_area_m2=b,
                                       area_ratio=b/a,relative_area_error=(b-a)/a))
    for i in range(len(ref)):
        manual_rows.append(dict(model=name,manual_id=i,size_class=str(ref.size_class.iloc[i]),
                                manual_area_m2=float(ref.manual_area_m2.iloc[i]),
                                best_iou=float(best[i]),combined_coverage=float(covered[i]),
                                significant_overlap_count=int(overlap_counts[i]),
                                best_prediction_id=int(best_pred[i])))
    # Flag predictions overlapping no manual crown; not necessarily false positives
    for j,g in enumerate(p.geometry):
        pred_rows.append(dict(model=name,pred_id=j,area_m2=g.area,
                              best_reference_iou=float(mat[:,j].max()) if len(ref) else 0,
                              intersects_reference=bool((mat[:,j]>0).any()) if len(ref) else False))
metrics=pd.DataFrame(records);by_size=pd.DataFrame(size_rows)
diagnostics=pd.DataFrame(manual_rows);matches=pd.DataFrame(match_rows)
pred_diagnostics=pd.DataFrame(pred_rows)
metrics.to_csv(V/"iou_threshold_comparison.csv",index=False)
by_size.to_csv(V/"recall_by_size.csv",index=False)
diagnostics.to_csv(V/"manual_crown_fragmentation.csv",index=False)
matches.to_csv(V/"one_to_one_matches_iou05.csv",index=False)
pred_diagnostics.to_csv(V/"prediction_reference_overlap.csv",index=False)
print("Full-ROI manual crowns:",len(ref))
display(metrics.round(3))
display(by_size[by_size.threshold==.5].round(3))
if len(matches):
    print("Matched crown area bias at IoU 0.5:")
    display(matches.groupby("model").agg(n=("iou","size"),
        median_area_ratio=("area_ratio","median"),
        median_relative_area_error=("relative_area_error","median")).round(3))
print("Saved validation tables:",V)
print("Note: unmatched SAM3 polygons may be genuine unannotated crowns.")


## 23 — Statistical proposal-selection benchmark (CPU only)

Run `0R → 14 → 23`. Uses the existing full-ROI Block 22 SAM3 polygons. No DSM, no new inference. Tunes confidence and overlap/nesting suppression on a western spatial reference subset, locks the parameters, and evaluates them on the eastern subset with a 6 m split buffer. Produces `spatial_holdout_comparison.csv`, `training_parameter_sweep.csv`, `selected_parameters.csv`, and `selected_full_roi.gpkg`. Manual crowns are never used to choose individual polygons. This is a within-ROI holdout, not a separate-site validation.


In [ ]:
# BLOCK 23 — CPU-only proposal selection benchmark, full ROI 0
# Run 0R -> 14 -> 23. No DSM, no SAM3 rerun, no modification of original polygons.
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio, matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from shapely.geometry import box
ROI_INDEX=0
SOURCE=OUT/f"experiments_v22_full_roi_8m_{ROI_INDEX}"/"full"/"clean_crowns.gpkg"
DEST=OUT/f"experiments_v23_proposal_selection_roi_{ROI_INDEX}"
DEST.mkdir(parents=True,exist_ok=True)
assert SOURCE.exists(),"Run Block 22 first"
assert "reference_v2" in globals() and "roi_v2" in globals(),"Run Block 14"
with rasterio.open(RASTER) as src:crs=src.crs
roi=roi_v2.to_crs(crs).geometry.iloc[ROI_INDEX]
pred=gpd.read_file(SOURCE).to_crs(crs).reset_index(drop=True)
ref=reference_v2.to_crs(crs)
ref=ref[ref.geometry.apply(roi.covers)].copy().reset_index(drop=True)
assert len(pred) and len(ref),"Empty input"
pred["selection_id"]=np.arange(len(pred))
pred["area_m2"]=pred.geometry.area
pred["compactness"]=4*np.pi*pred.area_m2/(pred.geometry.length**2).clip(lower=1e-8)
if "score" not in pred.columns:
    raise ValueError("SAM3 score column missing; cannot benchmark confidence filters")
pred["score"]=pd.to_numeric(pred["score"],errors="coerce")
print("SAM3 score distribution:",pred.score.describe().round(3).to_dict())
# Split by reference crown spatial location. Spatial blocks reduce train/test leakage.
# Thresholds are selected only from the west side; east is untouched until final evaluation.
split=float(ref.geometry.centroid.x.median())
train_idx=np.flatnonzero(ref.geometry.centroid.x.to_numpy()<split)
test_idx=np.flatnonzero(ref.geometry.centroid.x.to_numpy()>=split)
assert len(train_idx)>20 and len(test_idx)>20
print(f"Manual reference: train {len(train_idx)}, test {len(test_idx)}; east/west spatial split x={split:.1f}")
# Precompute IoU for ALL predictions. Reference outlines never guide filtering.
mat=np.zeros((len(ref),len(pred)),dtype=np.float32)
pidx=pred.sindex
for i,g in enumerate(ref.geometry):
    for j in pidx.query(g,predicate="intersects"):
        h=pred.geometry.iloc[j];inter=g.intersection(h).area
        if inter>0:mat[i,j]=inter/(g.area+h.area-inter)
# A candidate selected anywhere in ROI may be matched to references in its evaluation subset.
def metric(ref_ids,selected,thr=.5):
    m=mat[np.ix_(ref_ids,selected)]
    if not m.size:return dict(n_ref=len(ref_ids),n_predictions=len(selected),matched=0,recall=0.,matched_fraction=0.)
    utility=(m>=thr).astype(float)*1000+m
    ii,jj=linear_sum_assignment(utility,maximize=True)
    good=np.array([m[i,j]>=thr for i,j in zip(ii,jj)],dtype=bool)
    count=int(good.sum())
    return dict(n_ref=len(ref_ids),n_predictions=len(selected),matched=count,
                recall=count/len(ref_ids),matched_fraction=count/len(selected) if len(selected) else 0.)
# Spatial filtering: only candidate centres within a half-ROI count toward that half's
# candidate count. We still use full prediction polygons for IoU.
px=pred.geometry.centroid.x.to_numpy()
# Use a 6m exclusion buffer at the split for parameter selection and evaluation,
# avoiding nearby training/test reference crowns.
buffer_m=6.
train_idx=np.array([i for i in train_idx if ref.geometry.iloc[i].centroid.x<split-buffer_m])
test_idx=np.array([i for i in test_idx if ref.geometry.iloc[i].centroid.x>split+buffer_m])
assert len(train_idx)>15 and len(test_idx)>15,"Not enough crowns after spatial buffer"
print("After split buffer: train",len(train_idx),"test",len(test_idx))
train_candidates=np.flatnonzero(px<split-buffer_m)
test_candidates=np.flatnonzero(px>split+buffer_m)
# Cache polygon-pair overlap for NMS; no manual reference geometry used.
pair_iou={};pair_contain={}
idx=pred.sindex
for i,g in enumerate(pred.geometry):
    for j in idx.query(g,predicate="intersects"):
        j=int(j)
        if j<=i:continue
        h=pred.geometry.iloc[j];a=g.intersection(h).area
        if a<=0:continue
        pair_iou[i,j]=a/(g.area+h.area-a)
        pair_contain[i,j]=a/min(g.area,h.area)
def select(candidate_ids,score_cut,nms_iou,containment,rank_mode):
    ids=[int(i) for i in candidate_ids if np.isfinite(pred.score.iloc[i]) and pred.score.iloc[i]>=score_cut]
    if rank_mode=="score":
        ids.sort(key=lambda i:(float(pred.score.iloc[i]),float(pred.area_m2.iloc[i])),reverse=True)
    else:
        # Soft preference for plausible-sized, compact proposals, NOT fitted to labels.
        ids.sort(key=lambda i:(float(pred.score.iloc[i])+.08*float(pred.compactness.iloc[i])
                 -.03*abs(np.log(max(float(pred.area_m2.iloc[i]),1)/40))),reverse=True)
    kept=[]
    for i in ids:
        reject=False
        for j in kept:
            k=(min(i,j),max(i,j))
            if pair_iou.get(k,0)>=nms_iou or pair_contain.get(k,0)>=containment:
                reject=True;break
        if not reject:kept.append(i)
    return np.array(kept,dtype=int)
# Score quantiles, not absolute score assumptions.
q=np.unique(np.quantile(pred.score.dropna(),[0,.1,.25,.4,.55,.7]))
settings=[]
for cutoff in q:
    for iou,contain in [(1.01,1.01),(.7,1.01),(.5,1.01),(.5,.95),(.35,.9)]:
        for rank in ["score","score_shape"]:
            settings.append(dict(cutoff=float(cutoff),nms_iou=iou,containment=contain,rank=rank))
rows=[]
for k,cfg in enumerate(settings):
    selected=select(train_candidates,cfg["cutoff"],cfg["nms_iou"],cfg["containment"],cfg["rank"])
    m=metric(train_idx,selected)
    rows.append(dict(setting_id=k,**cfg,**m))
tuning=pd.DataFrame(rows)
tuning.to_csv(DEST/"training_parameter_sweep.csv",index=False)
baseline=metric(train_idx,train_candidates)
# Conservative optimization: maximize recall first, prefer fewer proposals only
# among configurations retaining at least 95% of unfiltered training matches.
target=.95*baseline["matched"]
eligible=tuning[tuning.matched>=target]
if len(eligible):
    chosen=eligible.sort_values(["n_predictions","recall"],ascending=[True,False]).iloc[0]
else:
    chosen=tuning.sort_values(["matched","n_predictions"],ascending=[False,True]).iloc[0]
cfg=dict(cutoff=float(chosen.cutoff),nms_iou=float(chosen.nms_iou),
         containment=float(chosen.containment),rank=str(chosen["rank"]))
print("\nSelected on training region only:",cfg)
print("Unfiltered training:",baseline)
print("Filtered training:",metric(train_idx,select(train_candidates,**dict(
    score_cut=cfg["cutoff"],nms_iou=cfg["nms_iou"],containment=cfg["containment"],
    rank_mode=cfg["rank"]))))
# Locked settings applied to independent east test region.
test_sel=select(test_candidates,score_cut=cfg["cutoff"],nms_iou=cfg["nms_iou"],
                containment=cfg["containment"],rank_mode=cfg["rank"])
train_sel=select(train_candidates,score_cut=cfg["cutoff"],nms_iou=cfg["nms_iou"],
                 containment=cfg["containment"],rank_mode=cfg["rank"])
comparisons=[]
for region,refs,candidates,selected in [
    ("training_west",train_idx,train_candidates,train_sel),
    ("heldout_east",test_idx,test_candidates,test_sel)]:
    for method,ids in [("unfiltered",candidates),("selected",selected)]:
        for thr in [.3,.5,.7]:
            comparisons.append(dict(region=region,method=method,iou_threshold=thr,
                                    **metric(refs,ids,thr)))
comparison=pd.DataFrame(comparisons)
comparison.to_csv(DEST/"spatial_holdout_comparison.csv",index=False)
display(comparison.round(3))
# Save full-ROI application of locked parameters, no use of test labels.
full_ids=select(np.arange(len(pred)),score_cut=cfg["cutoff"],
                nms_iou=cfg["nms_iou"],containment=cfg["containment"],rank_mode=cfg["rank"])
selected_gdf=pred.iloc[full_ids].copy().reset_index(drop=True)
selected_gdf.to_file(DEST/"selected_full_roi.gpkg",driver="GPKG")
pd.DataFrame([cfg]).to_csv(DEST/"selected_parameters.csv",index=False)
print("\nFull ROI:",len(pred),"input proposals ->",len(full_ids),"selected proposals")
print("Outputs:",DEST)
print("Caution: training/test are spatial halves of one ROI, not independent forest sites.")
print("Unmatched polygons are not necessarily false detections.")


## 24 — GIS topology and crown geometry optimization (CPU only)

Run **0R → 14 → 24**. Tests multipart/sliver cleanup, small-candidate area thresholds, morphological opening and nested-polygon removal (retain larger versus smaller). Selects parameters on western manual crowns and reports performance on spatially held-out eastern crowns. **No SAM3 inference, DSM or GPU.** Original Block 22 polygons are unchanged.


In [ ]:
# BLOCK 24 — GIS topology / crown geometry benchmark (CPU only)
# Run 0R -> 14 -> 24. Uses Block 22 ORIGINAL 879 polygons, not Block 23 output.
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from shapely.geometry import Polygon, MultiPolygon
from scipy.optimize import linear_sum_assignment
from shapely.validation import make_valid
from itertools import product
SRC=OUT/"experiments_v22_full_roi_8m_0"/"full"/"clean_crowns.gpkg"
DST=OUT/"experiments_v24_gis_cleanup_roi_0";DST.mkdir(parents=True,exist_ok=True)
assert SRC.exists(),"Run Block 22 first"
assert "reference_v2" in globals() and "roi_v2" in globals(),"Run Block 14"
with rasterio.open(RASTER) as ds:crs=ds.crs
roi=roi_v2.to_crs(crs).geometry.iloc[0]
P=gpd.read_file(SRC).to_crs(crs).reset_index(drop=True)
R=reference_v2.to_crs(crs)
R=R[R.geometry.apply(roi.covers)].reset_index(drop=True)
P["pid"]=np.arange(len(P))
score_col=next((c for c in ["score","confidence","predicted_iou"] if c in P),None)
scores=pd.to_numeric(P[score_col],errors="coerce").fillna(0).to_numpy() if score_col else np.zeros(len(P))
def polys(g):
    if g is None or g.is_empty:return []
    if g.geom_type=="Polygon":return [g]
    if hasattr(g,"geoms"):return [p for x in g.geoms for p in polys(x)]
    return []
def repair(g):
    try:return make_valid(g) if not g.is_valid else g
    except Exception:return g.buffer(0)
orig=[repair(g) for g in P.geometry]
def clean(g,component_min=0,opening=0):
    parts=[p for p in polys(g) if p.area>=component_min]
    if not parts:return None
    # Remove disconnected fragments; never combine separated parts into a single crown.
    g=max(parts,key=lambda x:x.area)
    if opening:
        try:
            t=g.buffer(-opening,join_style=1)
            if t.is_empty:return None
            t=t.buffer(opening,join_style=1)
            parts=polys(repair(t))
            if not parts:return None
            g=max(parts,key=lambda x:x.area)
        except Exception:return None
    return repair(g)
# Precompute a small, explicit grid of geometry-only cleanup choices.
geoms={}
for comp,opening in product([0,.5,1.0],[0,.15,.3]):
    geoms[(comp,opening)]=[clean(g,comp,opening) for g in orig]
# Fixed spatial holdout. Reference outlines never influence which polygons get selected.
mid=float(np.median(R.geometry.centroid.x))
rx=R.geometry.centroid.x.to_numpy()
px=P.geometry.centroid.x.to_numpy()
buffer_m=6
west=np.flatnonzero(rx<mid-buffer_m);east=np.flatnonzero(rx>mid+buffer_m)
west_p=np.flatnonzero(px<mid-buffer_m);east_p=np.flatnonzero(px>mid+buffer_m)
assert min(len(west),len(east))>15
def select(gs,ids,min_area=0,contain=.99,policy="larger"):
    valid=[int(i) for i in ids if gs[i] is not None and gs[i].area>=min_area]
    if contain>1:return valid
    # Keep larger or higher-confidence polygon in highly nested pairs.
    if policy=="larger":order=sorted(valid,key=lambda i:(gs[i].area,scores[i]),reverse=True)
    elif policy=="smaller":order=sorted(valid,key=lambda i:(-gs[i].area,scores[i]),reverse=True)
    else:order=sorted(valid,key=lambda i:(scores[i],gs[i].area),reverse=True)
    kept=[];removed=set()
    for i in order:
        if i in removed:continue
        kept.append(i)
        a=gs[i]
        for j in order:
            if j==i or j in removed or j in kept:continue
            b=gs[j]
            if not a.intersects(b):continue
            overlap=a.intersection(b).area/min(a.area,b.area)
            if overlap>=contain:removed.add(j)
    return kept
def evaluate(gs,ref_ids,ids,thr=.5):
    ids=list(ids);ref_ids=list(ref_ids)
    if not ids:return dict(matched=0,recall=0.,median_iou=np.nan,n_predictions=0)
    sg=gpd.GeoSeries([gs[j] for j in ids],crs=crs)
    si=sg.sindex
    M=np.zeros((len(ref_ids),len(ids)),dtype=np.float32)
    for k,rid in enumerate(ref_ids):
        g=R.geometry.iloc[rid]
        for j in si.query(g,predicate="intersects"):
            h=sg.iloc[j];inter=g.intersection(h).area
            if inter>0:M[k,j]=inter/(g.area+h.area-inter)
    utility=(M>=thr).astype(np.float32)*1000+M
    ii,jj=linear_sum_assignment(utility,maximize=True)
    values=np.array([M[i,j] for i,j in zip(ii,jj)])
    hits=values[values>=thr]
    return dict(matched=len(hits),recall=len(hits)/len(ref_ids),
                median_iou=float(np.median(hits)) if len(hits) else np.nan,n_predictions=len(ids))
# Tune on west: preserve >=95% of baseline matches, minimize number of proposals.
baseline=evaluate(orig,west,west_p)
rows=[]
for (comp,opening),gs in geoms.items():
    for min_area,contain,policy in product([0,1,3],[1.01,.99,.95],["larger","smaller"]):
        ids=select(gs,west_p,min_area,contain,policy)
        m=evaluate(gs,west,ids)
        rows.append(dict(component_min_m2=comp,opening_m=opening,min_area_m2=min_area,
                         containment=contain,policy=policy,**m))
tune=pd.DataFrame(rows)
tune.to_csv(DST/"training_parameter_sweep.csv",index=False)
eligible=tune[tune.matched>=.95*baseline["matched"]]
if eligible.empty:
    best=tune.sort_values(["matched","n_predictions"],ascending=[False,True]).iloc[0]
else:
    best=eligible.sort_values(["n_predictions","matched"],ascending=[True,False]).iloc[0]
cfg={k:best[k] for k in ["component_min_m2","opening_m","min_area_m2","containment","policy"]}
gs=geoms[(float(cfg["component_min_m2"]),float(cfg["opening_m"]))]
def ids_for(region):return select(gs,region,float(cfg["min_area_m2"]),float(cfg["containment"]),str(cfg["policy"]))
rows=[]
for region,refs,candidates in [("training_west",west,west_p),("heldout_east",east,east_p)]:
    for method,gg,ids in [("original",orig,list(candidates)),("GIS_cleaned",gs,ids_for(candidates))]:
        for thr in [.3,.5,.7]:
            rows.append(dict(region=region,method=method,iou_threshold=thr,
                             n_ref=len(refs),**evaluate(gg,refs,ids,thr)))
results=pd.DataFrame(rows)
results.to_csv(DST/"spatial_holdout_comparison.csv",index=False)
pd.DataFrame([cfg]).to_csv(DST/"selected_parameters.csv",index=False)
# Apply fixed settings to full ROI, without consulting east/test labels.
full_ids=ids_for(np.arange(len(P)))
cleaned=P.iloc[full_ids].copy().reset_index(drop=True)
cleaned.geometry=gpd.GeoSeries([gs[i] for i in full_ids],index=cleaned.index,crs=crs)
cleaned=cleaned[cleaned.geometry.notna() & ~cleaned.geometry.is_empty]
cleaned.to_file(DST/"selected_full_roi.gpkg",driver="GPKG")
print("Block 24: input polygons",len(P),"manual crowns",len(R))
print("Train/test crowns",len(west),len(east),"6m spatial buffer")
print("Training baseline IoU>=.5",baseline)
print("Selected settings:",cfg)
display(results.round(3))
print("Full ROI:",len(P),"->",len(cleaned),"polygons")
print("Saved:",DST)
print("Important: candidate selection uses geometry only; unmatched polygons may be real trees.")


## 25 — Learn to rank SAM3 crown proposals (CPU only)

**Fresh Colab:** run **Block 0R → Block 14 → Block 25**. Requires the saved Block 22 `clean_crowns.gpkg`; **do not rerun Block 22 or install SAM3**. Uses geometry, confidence, overlap and RGB polygon statistics. Labels high-IoU candidates as positive and only clearly conflicting proposals as negative; all others remain uncertain. Train on western proposals and report crown recall on held-out eastern crowns. Outputs scored GeoPackages, feature importances and held-out evaluation. Note: candidate discrimination on the labeled subset does not establish precision over all proposals.


In [ ]:
# BLOCK 25 — Crown proposal classifier; CPU only; run 0R -> 14 -> 25
# Uses ORIGINAL Block 22 proposals; never runs SAM3. Test labels are never used for training.
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.features import geometry_mask
from rasterio.windows import from_bounds, Window
from scipy.optimize import linear_sum_assignment
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
import joblib
from shapely.validation import make_valid
import matplotlib.pyplot as plt
SRC=OUT/"experiments_v22_full_roi_8m_0"/"full"/"clean_crowns.gpkg"
DEST=OUT/"experiments_v25_proposal_classifier_roi_0";DEST.mkdir(parents=True,exist_ok=True)
assert SRC.exists(),"Block 22 saved proposals missing"
assert "reference_v2" in globals() and "roi_v2" in globals(),"Run Block 14 first"
with rasterio.open(RASTER) as src: crs=src.crs
P=gpd.read_file(SRC).to_crs(crs).reset_index(drop=True)
R=reference_v2.to_crs(crs)
roi=roi_v2.to_crs(crs).geometry.iloc[0]
R=R[R.geometry.apply(roi.covers)].reset_index(drop=True)
P["proposal_id"]=np.arange(len(P))
assert len(P)>0 and len(R)>0
if "score" not in P.columns:
    print("WARNING: score missing; using constant confidence")
    P["score"]=0.5
P["score"]=pd.to_numeric(P["score"],errors="coerce").fillna(.5)
# 1. Unsupervised proposal geometry and overlap features
areas=P.geometry.area.to_numpy()
perim=P.geometry.length.to_numpy()
features=pd.DataFrame({"proposal_id":P.proposal_id,
    "score":P.score.to_numpy(),"area_m2":areas,"log_area":np.log1p(areas),
    "perimeter_m":perim,"compactness":4*np.pi*areas/np.maximum(perim**2,1e-8),
    "bbox_fill":areas/np.maximum(np.array([g.envelope.area for g in P.geometry]),1e-8)})
n=len(P);max_iou=np.zeros(n);max_contain=np.zeros(n);n_overlap=np.zeros(n,dtype=int)
n_nested=np.zeros(n,dtype=int);higher_score_overlap=np.zeros(n,dtype=int)
si=P.sindex
for i,g in enumerate(P.geometry):
    for j in si.query(g,predicate="intersects"):
        j=int(j)
        if j<=i:continue
        h=P.geometry.iloc[j];inter=g.intersection(h).area
        if inter<=0:continue
        ov=inter/(areas[i]+areas[j]-inter)
        cont=inter/max(min(areas[i],areas[j]),1e-8)
        max_iou[i]=max(max_iou[i],ov);max_iou[j]=max(max_iou[j],ov)
        max_contain[i]=max(max_contain[i],cont);max_contain[j]=max(max_contain[j],cont)
        n_overlap[i]+=1;n_overlap[j]+=1
        if cont>=.9:n_nested[i]+=1;n_nested[j]+=1
        if P.score.iloc[j]>P.score.iloc[i]:higher_score_overlap[i]+=1
        if P.score.iloc[i]>P.score.iloc[j]:higher_score_overlap[j]+=1
features["max_proposal_iou"]=max_iou;features["max_containment"]=max_contain
features["n_overlaps"]=n_overlap;features["n_nested"]=n_nested
features["n_higher_score_overlaps"]=higher_score_overlap
# 2. RGB polygon statistics; raster window per polygon, no DSM.
rgb_mean=np.full((n,3),np.nan);rgb_sd=np.full((n,3),np.nan)
with rasterio.open(RASTER) as src:
    assert src.count>=3,"Expected RGB raster"
    for i,g in enumerate(P.geometry):
        try:
            win=from_bounds(*g.bounds,transform=src.transform).round_offsets().round_lengths()
            win=win.intersection(Window(0,0,src.width,src.height))
            if win.width<1 or win.height<1:continue
            # Downsample large windows to limit CPU and RAM
            step=max(1,int(np.ceil(max(win.width,win.height)/256)))
            arr=src.read([1,2,3],window=win)[:,::step,::step]
            transform=src.window_transform(win)*rasterio.Affine.scale(step,step)
            mask=geometry_mask([g],out_shape=arr.shape[1:],transform=transform,invert=True)
            if mask.sum()<3:continue
            vals=arr[:,mask].astype(np.float32)
            rgb_mean[i]=vals.mean(axis=1);rgb_sd[i]=vals.std(axis=1)
        except Exception as exc:
            if i<3:print("RGB sample warning:",exc)
for k,c in enumerate("rgb"):
    features[f"{c}_mean"]=rgb_mean[:,k]
    features[f"{c}_std"]=rgb_sd[:,k]
# 3. Reference matching for labels ONLY. Conservative negatives are competing
# proposals that substantially occupy a documented reference crown but fit poorly.
# Unmatched polygons elsewhere are UNLABELED, not assumed to be false crowns.
ref_index=R.sindex
best_iou=np.zeros(n);best_ref=np.full(n,-1,dtype=int)
pred_cover=np.zeros(n)
for i,g in enumerate(P.geometry):
    for j in ref_index.query(g,predicate="intersects"):
        j=int(j);h=R.geometry.iloc[j]
        inter=g.intersection(h).area
        if inter<=0:continue
        iou=inter/(g.area+h.area-inter)
        pred_cover[i]=max(pred_cover[i],inter/max(g.area,1e-8))
        if iou>best_iou[i]:best_iou[i]=iou;best_ref[i]=j
# Strict positive: best IoU>=.5; ambiguous .25-.5 omitted;
# negative: IoU<.25 AND >=70% candidate area overlaps an annotated crown.
labels=np.full(n,-1,dtype=int)
labels[best_iou>=.5]=1
labels[(best_iou<.25)&(pred_cover>=.70)]=0
# Split spatially, with a 6m no-training/test buffer
split=float(R.geometry.centroid.x.median());buf=6.
x=P.geometry.centroid.x.to_numpy()
train=(x<split-buf);test=(x>split+buf)
known=(labels>=0)
train_idx=np.flatnonzero(train&known);test_idx=np.flatnonzero(test&known)
print("Candidate labels: positive",int((labels==1).sum()),"negative",int((labels==0).sum()),
      "uncertain/unlabeled",int((labels<0).sum()))
print("Spatial train/test labeled:",len(train_idx),len(test_idx))
assert len(np.unique(labels[train_idx]))==2 and len(np.unique(labels[test_idx]))==2,(
    "Not enough reliable positive/negative examples on both sides; inspect label counts.")
# 4. Fit model on training only, no target-based tuning on held-out region.
cols=[c for c in features if c!="proposal_id"]
X=features[cols].replace([np.inf,-np.inf],np.nan)
fill=X.iloc[train_idx].median().fillna(0)
X=X.fillna(fill)
clf=RandomForestClassifier(n_estimators=250,min_samples_leaf=4,
                            class_weight="balanced_subsample",random_state=42,n_jobs=-1)
clf.fit(X.iloc[train_idx],labels[train_idx])
proba=clf.predict_proba(X)[:,list(clf.classes_).index(1)]
features["crown_probability"]=proba;features["reference_label"]=labels
features["best_reference_iou_diagnostic_only"]=best_iou
features["spatial_partition"]=np.where(train,"train_west",np.where(test,"test_east","buffer"))
# 5. Test classifier discrimination only on reliable labeled held-out candidates.
print("Held-out labeled candidate ROC AUC:",round(roc_auc_score(labels[test_idx],proba[test_idx]),3))
print("Held-out labeled candidate average precision:",
      round(average_precision_score(labels[test_idx],proba[test_idx]),3))
# 6. Proposal selection: train-only threshold, calibrated to preserve 95% of
# positive training candidates. This is NOT full crown recall.
positive_train=proba[train_idx[labels[train_idx]==1]]
cutoff=float(np.quantile(positive_train,.05))
print("Training-derived classifier cutoff:",round(cutoff,3))
# Apply to entire ROI and retain all uncertain candidates in a separate audit layer.
selected=proba>=cutoff
# Compare one-to-one reference crown recall on spatial held-out side.
def recall(ref_ids,candidate_ids,threshold):
    if len(candidate_ids)==0:return 0
    rr=R.iloc[ref_ids];pp=P.iloc[candidate_ids]
    M=np.zeros((len(rr),len(pp)),dtype=np.float32)
    local_idx=pp.sindex
    for i,g in enumerate(rr.geometry):
        for j in local_idx.query(g,predicate="intersects"):
            h=pp.geometry.iloc[j];inter=g.intersection(h).area
            if inter>0:M[i,j]=inter/(g.area+h.area-inter)
    row,col=linear_sum_assignment((M>=threshold).astype(float)*1000+M,maximize=True)
    return int(sum(M[i,j]>=threshold for i,j in zip(row,col)))
rx=R.geometry.centroid.x.to_numpy()
held_ref=np.flatnonzero(rx>split+buf)
held_candidates=np.flatnonzero(test)
held_selected=np.flatnonzero(test&selected)
rows=[]
confidence_order=held_candidates[np.argsort(-P.score.to_numpy()[held_candidates],kind="stable")]
confidence_selected=confidence_order[:len(held_selected)]
for method,ids in [("original",held_candidates),("classifier_selected",held_selected),("confidence_same_count",confidence_selected)]:
    for t in [.3,.5,.7]:
        matched=recall(held_ref,ids,t)
        rows.append(dict(method=method,iou_threshold=t,manual=len(held_ref),
                         predictions=len(ids),matched=matched,recall=matched/len(held_ref)))
report=pd.DataFrame(rows)
display(report.round(3))
report.to_csv(DEST/"heldout_crown_recall.csv",index=False)
features.to_csv(DEST/"proposal_features_and_scores.csv",index=False)
P["crown_probability"]=proba
P["spatial_partition"]=features.spatial_partition.to_numpy()
P["selection_25"]=selected.astype(int)
P.to_file(DEST/"all_scored_proposals.gpkg",driver="GPKG")
P[selected].to_file(DEST/"classifier_selected_full_roi.gpkg",driver="GPKG")
pd.DataFrame({"feature":cols,"importance":clf.feature_importances_}).sort_values(
    "importance",ascending=False).to_csv(DEST/"feature_importance.csv",index=False)
print("Full ROI:",len(P),"->",int(selected.sum()),"selected")
print("Outputs:",DEST)
print("CAVEAT: negatives are partial/competing proposals, NOT all false crowns.")
print("CAVEAT: test split is within one ROI; next validate on an independent ROI.")

# Supplemental evaluation and reusable training artifacts.
report["reference_matched_prediction_fraction"]=report["matched"]/report["predictions"].clip(lower=1)
report["proposal_reduction_pct"]=100*(1-report["predictions"]/max(len(held_candidates),1))
report.to_csv(DEST/"heldout_crown_recall.csv",index=False)
size_rows=[]
for method,ids in [("original",held_candidates),("classifier_selected",held_selected),("confidence_same_count",confidence_selected)]:
    for lo,hi,label in [(0,20,"under_20"),(20,40,"20_to_40"),(40,80,"40_to_80"),(80,float("inf"),"over_80")]:
        group=np.array([i for i in held_ref if lo<=R.geometry.iloc[i].area<hi],dtype=int)
        if not len(group):continue
        for t in (.3,.5,.7):
            matched=recall(group,ids,t)
            size_rows.append(dict(method=method,size_class=label,iou_threshold=t,manual=len(group),matched=matched,recall=matched/len(group)))
pd.DataFrame(size_rows).to_csv(DEST/"heldout_recall_by_crown_size.csv",index=False)
features["label_confidence"]=np.where(labels==1,"reference_iou_ge_05",np.where(labels==0,"reference_conflict","unlabeled"))
features.to_csv(DEST/"proposal_features_and_scores.csv",index=False)
features[["proposal_id","reference_label","label_confidence","best_reference_iou_diagnostic_only","spatial_partition"]].to_csv(DEST/"training_labels_and_confidence.csv",index=False)
joblib.dump(dict(model=clf,feature_columns=cols,fill_values=fill,cutoff=cutoff,split_x=split,buffer_m=buf,random_state=42),DEST/"crown_candidate_rf.joblib")
print("Saved model, label categories, same-count confidence benchmark and size-stratified recall.")
print("Caution: threshold uses in-sample training scores, so may be optimistic.")


## 26 — Proposal-budget curves and overlap-aware selection (CPU only)

**Fresh runtime:** run **0R → 14 → 26**. Reads cached Block 25 scored polygons (`all_scored_proposals.gpkg`) and the original Block 22 candidates. Does **not** rerun SAM3, retrain the classifier, or require Block 2.

Tests fixed retention budgets (100, 90, 80, 70, 60, 50, 40, 30%) with three selection methods: classifier ranking, confidence ranking, and greedy overlap-aware classifier ranking. Overlap-aware ranking suppresses candidates only above a specified pairwise IoU and cannot promise tree-level uniqueness. Evaluation is exclusively on **76-style eastern held-out manual crowns** defined from Block 25's spatial split; report counts are generated from the actual cached geometries, not assumed.

**Scientific guardrails:** fixed method settings are exploratory and not chosen by held-out performance; curves should not be mined for an optimal threshold on this test split. Each reference is matched to at most one candidate through one-to-one bipartite matching; reference-matched prediction fraction is **not** true precision. Training-side candidates are excluded from held-out prediction evaluation. Outputs are new Block 26 CSVs and geopackages, leaving Blocks 22 and 25 unchanged.


In [ ]:
# BLOCK 26 — CPU-only ranked proposal-budget curves; run 0R -> 14 -> 26
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt

SRC25 = OUT / "experiments_v25_proposal_classifier_roi_0" / "all_scored_proposals.gpkg"
SRC22 = OUT / "experiments_v22_full_roi_8m_0" / "full" / "clean_crowns.gpkg"
DEST26 = OUT / "experiments_v26_ranked_selection_roi_0"
DEST26.mkdir(parents=True, exist_ok=True)
assert SRC25.exists(), f"Missing {SRC25}. Run Block 25 once to cache scores."
assert SRC22.exists(), f"Missing {SRC22}"
assert "reference_v2" in globals() and "roi_v2" in globals(), "Run Block 14 first"
with rasterio.open(RASTER) as src:
    crs26 = src.crs
P26 = gpd.read_file(SRC25).to_crs(crs26).reset_index(drop=True)
B26 = gpd.read_file(SRC22).to_crs(crs26).reset_index(drop=True)
assert len(P26) == len(B26), "Block 25 output and Block 22 proposal counts differ."
assert "crown_probability" in P26 and "score" in P26, "Cached model score columns missing."
# Verify stable input ordering before comparing to original proposal geometry.
assert all(a.equals(b) for a,b in zip(P26.geometry, B26.geometry)), (
    "Proposals were reordered or modified. Cannot safely compare with Block 22."
)
roi26 = roi_v2.to_crs(crs26).geometry.iloc[0]
R26 = reference_v2.to_crs(crs26)
R26 = R26[R26.geometry.apply(roi26.covers)].reset_index(drop=True)
# Reconstruct exactly the Block 25 spatial split.
split26 = float(R26.geometry.centroid.x.median())
buffer26 = 6.0
px26 = P26.geometry.centroid.x.to_numpy()
rx26 = R26.geometry.centroid.x.to_numpy()
test26 = np.flatnonzero(px26 > split26 + buffer26)
ref26 = np.flatnonzero(rx26 > split26 + buffer26)
assert len(test26)>0 and len(ref26)>0
# Candidate scores use no held-out reference geometry.
scores26 = pd.to_numeric(P26.crown_probability, errors="coerce").fillna(-1).to_numpy()
conf26 = pd.to_numeric(P26.score, errors="coerce").fillna(-1).to_numpy()
# Make reference-to-prediction IoU matrix once for efficient repeated budget evaluation.
Rtest26 = R26.iloc[ref26].reset_index(drop=True)
Ptest26 = P26.iloc[test26].reset_index(drop=True)
M26 = np.zeros((len(Rtest26), len(Ptest26)), dtype=np.float32)
rindex26 = Rtest26.sindex
for j,g in enumerate(Ptest26.geometry):
    for i in rindex26.query(g, predicate="intersects"):
        i=int(i)
        h=Rtest26.geometry.iloc[i]
        inter=g.intersection(h).area
        if inter>0:
            M26[i,j]=inter/max(g.area+h.area-inter,1e-9)
# Fixed-budget ranking without any reference-derived optimization.
def ranked26(index, values):
    return np.array(sorted(index, key=lambda i:(-float(values[i]),int(i))), dtype=int)

# Greedy NMS: only suppress when pairwise candidate IoU >= 0.5.
# Fill remaining budget with best-ranked previously suppressed candidates.
# This makes budgets comparable while maintaining an overlap preference.
def overlap_rank26(index, values, nms_iou=0.5):
    ordered=ranked26(index,values)
    pool=set(int(i) for i in index)
    si=P26.sindex
    kept=[]; deferred=[]
    for idx in ordered:
        g=P26.geometry.iloc[idx]
        rejected=False
        for j in si.query(g,predicate="intersects"):
            j=int(j)
            if j not in pool or j==idx or j not in kept:
                continue
            h=P26.geometry.iloc[j]
            intersection=g.intersection(h).area
            if intersection>0 and intersection/max(g.area+h.area-intersection,1e-9)>=nms_iou:
                rejected=True
                break
        (deferred if rejected else kept).append(int(idx))
    return np.array(kept+deferred,dtype=int)

rankings26 = {
    "classifier":ranked26(test26,scores26),
    "sam3_confidence":ranked26(test26,conf26),
    "overlap_aware_classifier":overlap_rank26(test26,scores26,nms_iou=0.5),
}
fractions26=[1.0,.9,.8,.7,.6,.5,.4,.3]
thresholds26=[.3,.5,.7]
pos26={int(idx):j for j,idx in enumerate(test26)}
def matched26(ids, threshold, ref_subset=None):
    rows=np.arange(len(Rtest26),dtype=int) if ref_subset is None else np.asarray(ref_subset,dtype=int)
    cols=np.array([pos26[int(i)] for i in ids],dtype=int)
    if not len(rows) or not len(cols):return 0
    sub=M26[np.ix_(rows,cols)]
    # One-to-one matching, prioritizing count above threshold over fine IoU.
    valid=(sub>=threshold).astype(np.float64)
    ii,jj=linear_sum_assignment(valid*2+sub.astype(np.float64),maximize=True)
    return int(np.count_nonzero(sub[ii,jj]>=threshold))

rows26=[]; size_rows26=[]
size_bins26=[(0,20,"under_20"),(20,40,"20_to_40"),(40,80,"40_to_80"),(80,np.inf,"over_80")]
for name,ranking in rankings26.items():
    for fraction in fractions26:
        k=max(1,int(np.ceil(len(test26)*fraction)))
        chosen=ranking[:k]
        for threshold in thresholds26:
            hits=matched26(chosen,threshold)
            rows26.append(dict(method=name,retained_fraction_target=fraction,
                predictions=k,original_predictions=len(test26),reduction_pct=100*(1-k/len(test26)),
                iou_threshold=threshold,reference_crowns=len(ref26),
                matched=hits,recall=hits/len(ref26),
                reference_matched_prediction_fraction=hits/k))
            for lo,hi,label in size_bins26:
                subset=np.flatnonzero((Rtest26.geometry.area.to_numpy()>=lo)&(Rtest26.geometry.area.to_numpy()<hi))
                if len(subset):
                    matched=matched26(chosen,threshold,subset)
                    size_rows26.append(dict(method=name,retained_fraction_target=fraction,
                        size_class=label,iou_threshold=threshold,reference_crowns=len(subset),
                        matched=matched,recall=matched/len(subset)))
curves26=pd.DataFrame(rows26)
sizes26=pd.DataFrame(size_rows26)
curves26.to_csv(DEST26/"heldout_budget_recall_curves.csv",index=False)
sizes26.to_csv(DEST26/"heldout_budget_recall_by_size.csv",index=False)
# Explicit fixed 70%-retention example, not a tuned or recommended selection.
selection26=rankings26["overlap_aware_classifier"][:int(np.ceil(.7*len(test26)))]
P26.iloc[selection26].to_file(DEST26/"example_heldout_overlap_aware_70pct.gpkg",driver="GPKG")
# Save full-heldout rankings as CSV for audit and re-use.
pd.DataFrame({name:pd.Series(rank) for name,rank in rankings26.items()}).to_csv(
    DEST26/"heldout_candidate_rankings.csv",index=False)
fig,ax=plt.subplots(figsize=(8,5))
for name in rankings26:
    sub=curves26[(curves26.method==name)&(curves26.iou_threshold==.5)].sort_values("reduction_pct")
    ax.plot(sub.reduction_pct,sub.recall,marker="o",label=name)
ax.set(xlabel="Held-out proposal reduction (%)",ylabel="One-to-one crown recall (IoU ≥ 0.5)",
       title="Block 26 — held-out proposal-budget trade-off")
ax.grid(alpha=.25);ax.legend();fig.tight_layout()
fig.savefig(DEST26/"heldout_recall_vs_reduction.png",dpi=150)
plt.show()
display(curves26[curves26.iou_threshold==.5].round(3))
print("Saved Block 26 outputs:",DEST26)
print("CAUTION: exploratory fixed-budget comparisons, not threshold selection on held-out references.")
print("CAUTION: baseline reference-matched fraction is not true precision.")


## 27 — Stricter one-to-one training labels and model comparison (CPU only)

**Fresh Colab:** run **Block 0R → Block 14 → Block 27**. Requires cached Block 25 `proposal_features_and_scores.csv` and `all_scored_proposals.gpkg`; no SAM3 environment, GPU, or rerun of Blocks 22–26.

**Scope:** ROI 0 only (not all six annotated ROIs). Train on western, spatially buffered subset; evaluate only eastern references/proposals. Enforce **one-to-one positive labels** on training-side manual crowns via maximum-cardinality IoU matching at 0.5; define conservative negatives as poor-IoU polygons almost entirely covering an annotated training crown; leave all other candidates *unlabeled*. Reference IoU and reference-derived label diagnostics are **not** model features. Compare a compact logistic regression and Random Forest with the previous Block 25 classifier and SAM3 confidence at fixed candidate budgets. Fixed hyperparameters and budgets: do not tune based on eastern results.

**Cautions:** a candidate that is unmatched to a manual crown is not necessarily a false tree; even high-overlap alternative polygons may be plausible trees. The held-out region has already been inspected in previous blocks and is an exploratory benchmark rather than a pristine final test. For a definitive generalization estimate, produce candidate inventories for other ROIs and reserve an independent ROI.


In [ ]:
# BLOCK 27 — Controlled one-to-one proposal-label experiment; CPU only
# Fresh runtime: 0R -> 14 -> 27. Cached Block 25 features required.
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, rasterio, joblib
from scipy.optimize import linear_sum_assignment
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

B25 = OUT/"experiments_v25_proposal_classifier_roi_0"
DEST27 = OUT/"experiments_v27_one_to_one_training_roi_0"
DEST27.mkdir(parents=True,exist_ok=True)
assert (B25/"all_scored_proposals.gpkg").exists(), "Run Block 25 once to save proposal scores."
assert (B25/"proposal_features_and_scores.csv").exists(), "Block 25 feature CSV missing."
assert "roi_v2" in globals() and "reference_v2" in globals(), "Run Block 14."
with rasterio.open(RASTER) as src: crs27=src.crs
P27=gpd.read_file(B25/"all_scored_proposals.gpkg").to_crs(crs27).reset_index(drop=True)
F27=pd.read_csv(B25/"proposal_features_and_scores.csv")
assert len(P27)==len(F27) and "proposal_id" in F27
assert np.array_equal(F27.proposal_id.to_numpy(),np.arange(len(P27))), "Proposal feature order changed."
R27=reference_v2.to_crs(crs27)
roi27=roi_v2.to_crs(crs27).geometry.iloc[0]
R27=R27[R27.geometry.apply(roi27.covers)].reset_index(drop=True)
split27=float(R27.geometry.centroid.x.median())
buf27=6.0
train_p27=np.flatnonzero(P27.geometry.centroid.x.to_numpy()<split27-buf27)
test_p27=np.flatnonzero(P27.geometry.centroid.x.to_numpy()>split27+buf27)
train_r27=np.flatnonzero(R27.geometry.centroid.x.to_numpy()<split27-buf27)
test_r27=np.flatnonzero(R27.geometry.centroid.x.to_numpy()>split27+buf27)
assert min(map(len,[train_p27,test_p27,train_r27,test_r27]))>0

# For training labels, compare ONLY western proposals with western manual crowns.
def iou_matrix27(ref_ids,prop_ids):
    rr=R27.iloc[ref_ids].reset_index(drop=True)
    pp=P27.iloc[prop_ids].reset_index(drop=True)
    mm=np.zeros((len(rr),len(pp)),dtype=np.float32)
    coverage=np.zeros(len(pp),dtype=np.float32)
    si=rr.sindex
    for j,g in enumerate(pp.geometry):
        for i in si.query(g,predicate="intersects"):
            i=int(i);h=rr.geometry.iloc[i]
            area=g.intersection(h).area
            if area<=0:continue
            mm[i,j]=area/max(g.area+h.area-area,1e-9)
            coverage[j]=max(coverage[j],area/max(g.area,1e-9))
    return mm,coverage
TR27,COVER27=iou_matrix27(train_r27,train_p27)
labels27=np.full(len(P27),-1,dtype=np.int8)
# Maximum-cardinality one-to-one positive matching, then prioritize IoU.
ii,jj=linear_sum_assignment((TR27>=.5).astype(float)*2+TR27.astype(float),maximize=True)
positive_local=np.array([j for i,j in zip(ii,jj) if TR27[i,j]>=.5],dtype=int)
labels27[train_p27[positive_local]]=1
# Negatives must be mostly *inside* a reference crown yet poorly shaped to match it.
# Avoid treating unmatched crown candidates elsewhere as negatives.
best_train_iou=TR27.max(axis=0) if len(train_r27) else np.zeros(len(train_p27))
negative_local=np.flatnonzero((best_train_iou<.25)&(COVER27>=.70))
labels27[train_p27[negative_local]]=0
# Exclude ambiguous candidate polygons from fitting.
labeled_train=train_p27[labels27[train_p27]>=0]
n_positive=int((labels27[labeled_train]==1).sum())
n_negative=int((labels27[labeled_train]==0).sum())
print("ROI 0: proposals",len(P27),"reference crowns",len(R27))
print("West train crowns",len(train_r27),"East held-out crowns",len(test_r27))
print("One-to-one training labels: positive",n_positive,"negative",n_negative,
      "unlabeled western",int((labels27[train_p27]<0).sum()))
if min(n_positive,n_negative)<5:
    raise RuntimeError("Insufficient reliable labels for robust classifier; inspect west-side counts.")

# Only predictors that can be measured without knowing reference crowns.
# Explicit allowlist prevents reference-derived columns leaking into training.
feature_columns27=[
    "score","area_m2","log_area","perimeter_m","compactness","bbox_fill",
    "max_proposal_iou","max_containment","n_overlaps","n_nested",
    "n_higher_score_overlaps","r_mean","g_mean","b_mean",
    "r_std","g_std","b_std"
]
missing27=[c for c in feature_columns27 if c not in F27.columns]
assert not missing27,f"Missing cached Block 25 predictors: {missing27}"
X27=F27[feature_columns27].replace([np.inf,-np.inf],np.nan)
y27=labels27[labeled_train]
models27={
    "logistic_one_to_one":make_pipeline(
        SimpleImputer(strategy="median",add_indicator=False),StandardScaler(),
        LogisticRegression(C=1.,class_weight="balanced",max_iter=1000,random_state=42)),
    "rf_one_to_one":make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(n_estimators=250,min_samples_leaf=4,
                               class_weight="balanced_subsample",n_jobs=-1,random_state=42))
}
score_columns27={}
for key,model in models27.items():
    model.fit(X27.iloc[labeled_train],y27)
    score_columns27[key]=model.predict_proba(X27)[:,list(model.classes_).index(1)]
    joblib.dump({"model":model,"predictors":feature_columns27,"train_indices":labeled_train,
                 "split_x":split27,"buffer_m":buf27,"positive_iou":.5},
                DEST27/(key+".joblib"))
score_columns27["block25_classifier"]=pd.to_numeric(P27.crown_probability,errors="coerce").fillna(-1).to_numpy()
score_columns27["sam3_confidence"]=pd.to_numeric(P27.score,errors="coerce").fillna(-1).to_numpy()

# Strict held-out one-to-one evaluation. Held-out references NEVER influence selection.
TE27,_=iou_matrix27(test_r27,test_p27)
test_position27={int(v):j for j,v in enumerate(test_p27)}
def evaluate27(global_candidate_ids,t):
    jj=np.array([test_position27[int(v)] for v in global_candidate_ids],dtype=int)
    if len(jj)==0:return 0
    sub=TE27[:,jj].astype(float)
    rr,cc=linear_sum_assignment((sub>=t).astype(float)*2+sub,maximize=True)
    return int(np.count_nonzero(sub[rr,cc]>=t))
budgets27=[1.,.9,.8,.7,.6,.5,.4,.3]
thresholds27=[.3,.5,.7]
rows27=[]
for method,scores in score_columns27.items():
    ranking=np.array(sorted(test_p27,key=lambda i:(-float(scores[i]),int(i))),dtype=int)
    for budget in budgets27:
        k=max(1,int(np.ceil(budget*len(test_p27))))
        selected=ranking[:k]
        for t in thresholds27:
            matches=evaluate27(selected,t)
            rows27.append(dict(method=method,retained_fraction_target=budget,
                predictions=k,original_predictions=len(test_p27),
                proposal_reduction_pct=100*(1-k/len(test_p27)),
                iou_threshold=t,reference_crowns=len(test_r27),
                matched=matches,recall=matches/len(test_r27),
                reference_matched_prediction_fraction=matches/k))
    # Save rankings, not only a cherry-picked test threshold.
    pd.DataFrame({"proposal_id":ranking,
        "score":scores[ranking],"rank":np.arange(1,len(ranking)+1)}
        ).to_csv(DEST27/(method+"_heldout_ranking.csv"),index=False)
    P27[method+"_score"]=scores
evaluation27=pd.DataFrame(rows27)
evaluation27.to_csv(DEST27/"heldout_budget_comparison.csv",index=False)
labels_table27=pd.DataFrame({"proposal_id":np.arange(len(P27)),"training_label":labels27,
    "partition":np.where(np.isin(np.arange(len(P27)),train_p27),"train_west",
                  np.where(np.isin(np.arange(len(P27)),test_p27),"test_east","buffer")),
    "label_category":np.where(labels27==1,"one_to_one_reference_match",
                   np.where(labels27==0,"poor_fit_reference_overlap","unlabeled"))})
labels_table27.to_csv(DEST27/"training_labels.csv",index=False)
P27.to_file(DEST27/"all_scored_proposals.gpkg",driver="GPKG")
display(evaluation27[(evaluation27.iou_threshold==.5)&
                     (evaluation27.retained_fraction_target.isin([1.,.7,.6,.5]))].round(3))
print("Saved:",DEST27)
print("CAVEAT: ROI 0 only; reference inventory incomplete; eastern split previously inspected.")
print("CAVEAT: models are fixed beforehand; use independent ROIs for definitive model selection.")


## 28 — Multiregion SAM3 crown discovery (same zero-shot RGB configuration)

**Purpose:** extend the *unchanged Block 22 geometric-prompt SAM3 experiment* from ROI 0 to remaining ROIs (indices 1–5). This is GPU inference for ROIs lacking cached masks; unlike Blocks 25–27, **run Block 2** in a new runtime.

**Restart sequence:** **0R → 2 → 14 → 28**. Block 28 embeds the exact Block 22 implementation so it has no dependency on rerunning Block 22 or fetching notebook code. It leaves ROI 0 intact, checks existing completed ROIs, and resumes missing cached SAM3 prompt masks. Each region writes under `experiments_v22_full_roi_8m_<ROI_INDEX>/full/`, with `all_proposals.gpkg`, `clean_crowns.gpkg`, `summary.json`, and per-size recall. If GPU time is limited, edit `ROI_IDS_28` to a subset (e.g. `[1]`); repeated runs skip complete ROIs and resume incomplete ones.

**Fixed settings:** RGB only; 40 m tiles, 10 m overlap, 6 m prompt grid, 8 m boxes; no DSM, CHM, treetop seeds or manual labels in inference. **DSM is reserved for the next supervised classifier comparison:** use DSM-derived height, local relief and crown-to-surroundings height contrast as additional predictors after all ROI proposals have been generated. This avoids conflating RGB-versus-DSM classifier benefits with a change in SAM3 proposal generation.

**Methodological caveat:** Each ROI's recall is evaluated only against manual crowns covered by the tile union. Reference-matched prediction fraction is not true precision. The original Block 22 code's JSON key `precision_05` refers to *reference-matched proposal fraction* and should not be interpreted as detection precision.


In [ ]:
# BLOCK 28 — ROI 1–5 multiregion expansion; exact Block 22 engine, GPU and resumable
# Fresh runtime: 0R -> 2 -> 14 -> 28. Select smaller groups to manage Colab GPU time.
import json
from pathlib import Path
import pandas as pd
assert "roi_v2" in globals() and "reference_v2" in globals(), "Run Block 14 first"
assert "SAMPY" in globals() and Path(SAMPY).exists(), "Run Block 2 for the SAM3 GPU environment"
ROI_IDS_28 = list(range(1, len(roi_v2)))  # Default: all regions other than ROI 0
assert len(roi_v2)>=2, "Expected multiple evaluation ROIs"
# Captured verbatim from committed Block 22; only the ROI_INDEX assignment is changed.
BLOCK22_SOURCE_28 = "# BLOCK 22 — Full ROI RGB SAM3, 8m boxes / 40m tiles (resumable)\n# Run after 0R, 2 (if environment absent), 14. No need for 0S or 15–18.\nimport os, sys, subprocess, json, math\nfrom pathlib import Path\nimport numpy as np, pandas as pd, geopandas as gpd, rasterio\nfrom rasterio.windows import Window\nfrom rasterio.features import shapes\nfrom rasterio.enums import Resampling\nfrom shapely.geometry import box, Point, shape\nfrom scipy.optimize import linear_sum_assignment\nfrom PIL import Image\nimport matplotlib.pyplot as plt\n\nROI_INDEX = 0\nTILE_M = 40.0\nOVERLAP_M = 10.0\nGRID_M = 6.0\nBOX_M = 8.0\nMAX_TILES = None       # None = complete ROI; 4 = technical smoke test\nMIN_AREA_M2, MAX_AREA_M2 = 2.0, 700.0\nEDGE_MARGIN_M = 0.4\nDUP_IOU = 0.65\nNESTED_CONTAINMENT = 0.95\nSIMILAR_AREA_RATIO = 0.65\n\nassert \"roi_v2\" in globals() and \"reference_v2\" in globals(), \"Run Block 14\"\nassert Path(SAMPY).exists(), \"Run Block 2 to install isolated SAM3 environment\"\nB19 = OUT / f\"experiments_v22_full_roi_8m_{ROI_INDEX}\"\nTILES, RAW19 = B19/\"tiles\", B19/\"raw\"\nfor d in (B19,TILES,RAW19): d.mkdir(parents=True,exist_ok=True)\n\nwith rasterio.open(RASTER) as src:\n    crs, gsd = src.crs, abs(src.transform.a)\n    roi19 = roi_v2.to_crs(crs).geometry.iloc[ROI_INDEX]\n    n = int(round(TILE_M/gsd))\n    stride = max(1,int(round((TILE_M-OVERLAP_M)/gsd)))\n    minx,miny,maxx,maxy = roi19.bounds\n    r0,c0 = src.index(minx,maxy)\n    r1,c1 = src.index(maxx,miny)\n    def starts(lo,hi):\n        if hi-lo <= n: return [int(round((lo+hi-n)/2))]\n        seq=list(range(int(lo),int(hi)-n+1,stride))\n        last=int(hi)-n\n        if not seq or seq[-1]!=last: seq.append(last)\n        return sorted(set(seq))\n    tiles=[]\n    for rr in starts(r0,r1+1):\n        for cc in starts(c0,c1+1):\n            win=Window(cc,rr,n,n)\n            b=rasterio.windows.bounds(win,src.transform)\n            if not box(*b).intersects(roi19): continue\n            tid=len(tiles); path=TILES/f\"tile_{tid:04d}.png\"\n            tiles.append(dict(tile_id=tid,row_off=rr,col_off=cc,width=n,height=n,\n                              xmin=b[0],ymin=b[1],xmax=b[2],ymax=b[3],path=str(path)))\n    if MAX_TILES is not None: tiles=tiles[:MAX_TILES]\n    assert tiles,\"No ROI tiles\"\n    for t in tiles:\n        p=Path(t[\"path\"])\n        if p.exists(): continue\n        win=Window(t[\"col_off\"],t[\"row_off\"],n,n)\n        rgb=src.read([1,2,3],window=win,boundless=True,fill_value=0).transpose(1,2,0)\n        Image.fromarray(rgb.astype(np.uint8)).save(p)\ntm=pd.DataFrame(tiles)\n# Scope outputs by the selected tile IDs so smoke tests never masquerade as full ROI.\nrun_tag=\"full\" if MAX_TILES is None else f\"first_{MAX_TILES}_tiles\"\nwork=B19/run_tag; raw=work/\"raw\"; raw.mkdir(parents=True,exist_ok=True)\nrows=[]\nfor t in tm.itertuples():\n    for dy in np.arange(BOX_M/2,TILE_M-BOX_M/2+1e-6,GRID_M):\n        for dx in np.arange(GRID_M/2,TILE_M,GRID_M):\n            x,y=t.xmin+dx,t.ymax-dy\n            if roi19.covers(Point(x,y)):\n                rows.append(dict(tile_id=int(t.tile_id),path=t.path,cx=dx/TILE_M,cy=dy/TILE_M,\n                                 box_m=BOX_M,width=n,height=n,gsd=gsd,\n                                 row_off=int(t.row_off),col_off=int(t.col_off)))\npm=pd.DataFrame(rows)\nassert len(pm),\"No grid prompts inside ROI\"\npm.insert(0,\"prompt_id\",np.arange(len(pm)))\npm.to_csv(work/\"prompt_manifest.csv\",index=False)\nprint(f\"ROI {ROI_INDEX}: {roi19.area/10000:.3f} ha | tiles={len(tm)} | prompts={len(pm)} | mode={run_tag}\")\nprint(\"Existing masks:\",sum((raw/f\"prompt_{i:06d}.npz\").exists() for i in pm.prompt_id))\n\nrunner=Path(\"/content/run_sam3_kvo22.py\")\nrunner.write_text(r'''\nimport sys\nfrom pathlib import Path\nimport numpy as np,pandas as pd,torch\nfrom PIL import Image\nfrom tqdm import tqdm\nfrom sam3.model_builder import build_sam3_image_model\nfrom sam3.model.sam3_image_processor import Sam3Processor\ndf=pd.read_csv(sys.argv[1]); outdir=Path(sys.argv[2]);outdir.mkdir(parents=True,exist_ok=True)\nif not torch.cuda.is_available(): raise RuntimeError(\"CUDA GPU required\")\ntorch.backends.cuda.matmul.allow_tf32=True\nbpe=\"/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz\"\nwith torch.inference_mode(),torch.autocast(\"cuda\",dtype=torch.bfloat16):\n    model=build_sam3_image_model(bpe_path=bpe)\n    processor=Sam3Processor(model)\n    for tid,g in df.groupby(\"tile_id\",sort=True):\n        pending=[r for r in g.itertuples() if not (outdir/f\"prompt_{int(r.prompt_id):06d}.npz\").exists()]\n        if not pending:continue\n        img=Image.open(pending[0].path).convert(\"RGB\")\n        for r in tqdm(pending,desc=f\"Tile {tid}\",leave=False):\n            # Fresh prompt state per box, to avoid accumulated prompts.\n            state=processor.set_image(img)\n            frac=min(.95,float(r.box_m)/(float(r.width)*float(r.gsd)))\n            out=processor.add_geometric_prompt(\n                box=np.array([[float(r.cx),float(r.cy),frac,frac]],dtype=np.float32),\n                label=np.array([1],dtype=np.int32),state=state)\n            masks=out[\"masks\"].detach().cpu().numpy()\n            scores=out[\"scores\"].detach().float().cpu().numpy()\n            if masks.ndim==4:masks=masks[:,0]\n            if masks.ndim==2:masks=masks[None]\n            np.savez_compressed(outdir/f\"prompt_{int(r.prompt_id):06d}.npz\",\n                                masks=(masks>0.5).astype(np.uint8),scores=scores)\n        print(f\"Tile {tid} complete\",flush=True)\nprint(\"Inference complete\",flush=True)\n''')\nmissing=sum(not (raw/f\"prompt_{i:06d}.npz\").exists() for i in pm.prompt_id)\nif missing:\n    run_env=os.environ.copy()\n    if \"env\" in globals(): run_env.update(env)\n    print(\"Running SAM3 for\",missing,\"unsaved prompts\")\n    subprocess.run([SAMPY,\"-u\",str(runner),str(work/\"prompt_manifest.csv\"),str(raw)],\n                   env=run_env,check=True)\nelse: print(\"All masks cached; skipping SAM3\")\n\n# Convert every returned instance, not just the highest-scoring mask.\nfeatures=[]\nwith rasterio.open(RASTER) as src:\n    for r in pm.itertuples():\n        p=raw/f\"prompt_{int(r.prompt_id):06d}.npz\"\n        if not p.exists():continue\n        with np.load(p) as z:\n            masks=z[\"masks\"]; scores=np.asarray(z[\"scores\"]).reshape(-1)\n        if masks.ndim==4:masks=masks[:,0]\n        if masks.ndim==2:masks=masks[None]\n        win=Window(int(r.col_off),int(r.row_off),int(r.width),int(r.height))\n        transform=src.window_transform(win)\n        for k,m in enumerate(masks):\n            m=np.asarray(m)>0\n            if m.shape!=(n,n) or not m.any():continue\n            margin=max(1,int(round(EDGE_MARGIN_M/gsd)))\n            if m[:margin].any() or m[-margin:].any() or m[:,:margin].any() or m[:,-margin:].any():\n                continue\n            geoms=[shape(g) for g,v in shapes(m.astype(\"uint8\"),mask=m,transform=transform) if v==1]\n            if not geoms:continue\n            geom=max(geoms,key=lambda g:g.area)\n            if not (MIN_AREA_M2<=geom.area<=MAX_AREA_M2):continue\n            if not roi19.covers(geom.representative_point()):continue\n            features.append(dict(prompt_id=int(r.prompt_id),tile_id=int(r.tile_id),\n                                 instance=int(k),score=float(scores[min(k,len(scores)-1)]),\n                                 area_m2=float(geom.area),geometry=geom))\nprint(\"Valid raw proposals:\",len(features))\nassert features,\"No polygons survived filtering; inspect tile-edge masks\"\nallpred=gpd.GeoDataFrame(features,geometry=\"geometry\",crs=crs)\nallpred.to_file(work/\"all_proposals.gpkg\",driver=\"GPKG\")\n\n# Spatial-index-assisted conservative duplicate removal.\nordered=allpred.sort_values(\"score\",ascending=False).reset_index(drop=True)\nkeep=[]; removed=[]\nfor i,r in ordered.iterrows():\n    g=r.geometry; duplicate=False\n    for j in keep:\n        h=ordered.geometry.iloc[j]\n        if not g.intersects(h):continue\n        inter=g.intersection(h).area\n        if inter<=0:continue\n        iou=inter/(g.area+h.area-inter)\n        containment=inter/min(g.area,h.area)\n        ratio=min(g.area,h.area)/max(g.area,h.area)\n        if iou>=DUP_IOU or (containment>=NESTED_CONTAINMENT and ratio>=SIMILAR_AREA_RATIO):\n            duplicate=True;removed.append(dict(prompt_id=int(r.prompt_id),against=int(j),iou=iou,\n                                                containment=containment,area_ratio=ratio))\n            break\n    if not duplicate:keep.append(i)\nclean=ordered.iloc[keep].copy().reset_index(drop=True)\nclean[\"crown_id\"]=np.arange(1,len(clean)+1)\nclean.to_file(work/\"clean_crowns.gpkg\",driver=\"GPKG\")\npd.DataFrame(removed).to_csv(work/\"removed_duplicates.csv\",index=False)\nprint(f\"Clean predictions: {len(clean)} | removed duplicates: {len(removed)}\")\n\n# Evaluate only reference crowns fully contained in the processed tile union\n# (and ROI). This prevents partial smoke-test coverage being labelled full-ROI.\nfootprints=[box(t.xmin,t.ymin,t.xmax,t.ymax) for t in tm.itertuples()]\nfrom shapely.ops import unary_union\ncovered=unary_union(footprints).intersection(roi19)\nreference=reference_v2.to_crs(crs).copy()\nreference=reference[reference.geometry.apply(covered.covers)].copy().reset_index(drop=True)\nmat=np.zeros((len(reference),len(clean)),dtype=np.float32)\nidx=clean.sindex\nfor i,g in enumerate(reference.geometry):\n    for j in idx.query(g,predicate=\"intersects\"):\n        h=clean.geometry.iloc[j]; inter=g.intersection(h).area\n        union=g.area+h.area-inter\n        if union>0:mat[i,j]=inter/union\nmatches=[]\nif mat.size:\n    utility=(mat>=.5).astype(float)*1000+mat\n    ii,jj=linear_sum_assignment(utility,maximize=True)\n    matches=[(int(i),int(j),float(mat[i,j])) for i,j in zip(ii,jj) if mat[i,j]>=.5]\nmatched_ref={i for i,j,v in matches}\nreference[\"area_m2\"]=reference.geometry.area\nreference[\"matched\"]=[i in matched_ref for i in range(len(reference))]\nreference[\"best_iou\"]=mat.max(axis=1) if len(clean) else 0\nreference[\"size_class\"]=pd.cut(reference.area_m2,[0,20,40,80,np.inf],\n                               labels=[\"<20\",\"20–40\",\"40–80\",\">80\"])\nsize=reference.groupby(\"size_class\",observed=True).agg(\n    n=(\"matched\",\"size\"),matched=(\"matched\",\"sum\"),recall=(\"matched\",\"mean\"),\n    median_best_iou=(\"best_iou\",\"median\"))\nsize.to_csv(work/\"recall_by_size.csv\")\ndisplay(size)\nrecall=len(matches)/len(reference) if len(reference) else float(\"nan\")\nprecision=len(matches)/len(clean) if len(clean) else float(\"nan\")\nsummary=dict(roi_index=ROI_INDEX,mode=run_tag,roi_area_ha=roi19.area/10000,\n             tiles=len(tm),prompts=len(pm),manual_crowns=len(reference),\n             raw_proposals=len(allpred),clean_predictions=len(clean),\n             matched_iou_05=len(matches),recall_05=recall,precision_05=precision,\n             median_matched_iou=float(np.median([v for _,_,v in matches])) if matches else None)\n(work/\"summary.json\").write_text(json.dumps(summary,indent=2))\nprint(\"RESULTS:\",json.dumps(summary,indent=2))\n\nprint(\"BLOCK 22 COMPLETE. Run Block 22V for full statistical validation:\",work)\n"
assert BLOCK22_SOURCE_28.count("ROI_INDEX = 0")==1
results28=[]
for target_roi28 in ROI_IDS_28:
    assert 0<=target_roi28<len(roi_v2), f"Unknown ROI index {target_roi28}"
    region_dir28=OUT/f"experiments_v22_full_roi_8m_{target_roi28}"/"full"
    summary_path28=region_dir28/"summary.json"
    cleaned_path28=region_dir28/"clean_crowns.gpkg"
    proposals_path28=region_dir28/"all_proposals.gpkg"
    if summary_path28.exists() and cleaned_path28.exists() and proposals_path28.exists():
        summary28=json.loads(summary_path28.read_text())
        if summary28.get("mode")=="full" and summary28.get("roi_index")==target_roi28:
            print(f"ROI {target_roi28}: full outputs already available, skipping GPU")
            results28.append(summary28)
            continue
    print(f"\\n=== Running or resuming ROI {target_roi28} ===",flush=True)
    engine28=BLOCK22_SOURCE_28.replace("ROI_INDEX = 0",f"ROI_INDEX = {target_roi28}",1)
    # Shared notebook namespace is intentional: mirrors running Block 22 for each ROI.
    exec(compile(engine28,f"<Block22_ROI_{target_roi28}>","exec"),globals())
    assert summary_path28.exists() and cleaned_path28.exists(), f"Missing completed outputs for ROI {target_roi28}"
    results28.append(json.loads(summary_path28.read_text()))
# Aggregate every completed ROI, including the original ROI 0 baseline.
all28=[]
for idx28 in range(len(roi_v2)):
    d28=OUT/f"experiments_v22_full_roi_8m_{idx28}"/"full"
    s28=d28/"summary.json"
    if s28.exists() and (d28/"clean_crowns.gpkg").exists():
        v28=json.loads(s28.read_text())
        if v28.get("mode")=="full" and v28.get("roi_index")==idx28:
            v28["reference_matched_prediction_fraction_05"]=v28.get("matched_iou_05",0)/max(v28.get("clean_predictions",0),1)
            all28.append(v28)
summary_table28=pd.DataFrame(all28).sort_values("roi_index") if all28 else pd.DataFrame()
if len(summary_table28):
    aggregate_dir28=OUT/"experiments_v28_multiregion_summary"
    aggregate_dir28.mkdir(parents=True,exist_ok=True)
    summary_table28.to_csv(aggregate_dir28/"roi_summary.csv",index=False)
    display(summary_table28[[c for c in ["roi_index","roi_area_ha","tiles","prompts","manual_crowns","raw_proposals","clean_predictions","matched_iou_05","recall_05","reference_matched_prediction_fraction_05"] if c in summary_table28.columns]].round(3))
    print("Complete ROI inventories:",len(summary_table28),"/",len(roi_v2))
    print("Saved aggregate summary:",aggregate_dir28/"roi_summary.csv")
else:
    print("No complete full-ROI inventories found.")


## 29 — Six-ROI crown proposal classifiers: RGB versus RGB + DSM (CPU only)

**Fresh Colab:** **Block 0R → Block 14 → Block 29**. Uses all six cached Block 28 `clean_crowns.gpkg` inventories, RGB orthomosaic and `Baego_DSM.tif`. No SAM3 GPU inference, no Block 2. Existing predictions and previous experiments are never overwritten.

**Experiment:** same Random Forest and identical splits/parameters for (A) confidence + polygon geometry + spatial overlap + RGB and (B) all A features + DSM statistics (within-crown height variability, local crown-to-surroundings contrast, local relative height). DSM elevations alone should not be interpreted as tree height without a terrain model (DTM); features emphasize **local differences**. Extracted from geographically aligned raster windows with `WarpedVRT` when needed.

**Validation:** leave-one-ROI-out for each of the six ROIs. On the other five ROIs, create one-to-one reference-matched positive labels (IoU ≥ 0.5), conservative poor-fit canopy negatives, and high-ground-fraction negatives; keep ambiguous candidates unlabeled. The held-out ROI's manual crowns are used **only for evaluation**, never for training labels, feature engineering, thresholds or model choice. Pre-specified fixed retention budgets: 100%, 90%, 80%, 70%, 60%, 50%, 40%, 30%; IoU thresholds 0.3/0.5/0.7. Include a SAM3-confidence baseline; compare recall **at equal proposal counts**, including crown-size classes and ROI 4. All reported reference-matched prediction fractions are distinct from genuine detection precision.

**Caveats:** exhaustive visible-crown coverage allows more informative negative examples, but no claim of perfect annotations; mis-segmented canopy objects and unlabeled proposals should not be blindly treated as background. The six ROIs may be spatially close and not fully independent forest sites. This experiment evaluates **selection**, not whether DSM improves SAM3 segmentation boundaries or generates additional crowns. Reference-validation association remains based on the original complete manual polygons.


In [ ]:
# BLOCK 29 — All six ROIs; RGB vs RGB+DSM proposal ranking; CPU ONLY
# Run 0R -> 14 -> 29. No SAM3 rerun. Writes only experiments_v29_*.
from pathlib import Path
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from rasterio.features import geometry_mask
from rasterio.windows import from_bounds, Window
from shapely.geometry import mapping
from shapely.ops import unary_union
from scipy.optimize import linear_sum_assignment
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
import joblib

DEST29=OUT/"experiments_v29_rgb_vs_dsm_six_roi"
DEST29.mkdir(parents=True,exist_ok=True)
assert "roi_v2" in globals() and "reference_v2" in globals(), "Run Block 14 first"
assert len(roi_v2)==6,"Expected exactly 6 ROIs"
DSM29=GEO/"Baego_DSM.tif"
assert DSM29.exists(),f"DSM missing: {DSM29}"
with rasterio.open(RASTER) as rgb:
    crs29=rgb.crs
roi29=roi_v2.to_crs(crs29).reset_index(drop=True)
references29=reference_v2.to_crs(crs29)
candidates29=[];ref_by_roi29={}
for roi_id in range(6):
    path=OUT/f"experiments_v22_full_roi_8m_{roi_id}"/"full"/"clean_crowns.gpkg"
    assert path.exists(),f"Missing Block 28 clean proposals: {path}"
    p=gpd.read_file(path).to_crs(crs29)
    p=p[p.geometry.notna()&~p.geometry.is_empty].copy().reset_index(drop=True)
    p["roi_id"]=roi_id
    p["local_proposal_id"]=np.arange(len(p))
    candidates29.append(p)
    roi_shape=roi29.geometry.iloc[roi_id]
    ref=references29[references29.geometry.apply(roi_shape.covers)].copy().reset_index(drop=True)
    assert len(ref),f"No fully contained reference crowns ROI {roi_id}"
    ref_by_roi29[roi_id]=ref
P29=gpd.GeoDataFrame(pd.concat(candidates29,ignore_index=True),geometry="geometry",crs=crs29)
P29["proposal_id"]=np.arange(len(P29))
assert len(P29)>=5000,f"Unexpected number of SAM3 proposals: {len(P29)}"
print("Loaded",len(P29),"candidates and",sum(map(len,ref_by_roi29.values())),"references")

# Geometry and overlap features computed strictly within each ROI (no reference labels).
area29=P29.geometry.area.to_numpy()
per29=P29.geometry.length.to_numpy()
base29=pd.DataFrame(dict(proposal_id=P29.proposal_id.to_numpy(),
    roi_id=P29.roi_id.to_numpy(),
    sam3_score=pd.to_numeric(P29["score"],errors="coerce").to_numpy(),
    area_m2=area29,log_area=np.log1p(area29),perimeter_m=per29,
    compactness=4*np.pi*area29/np.maximum(per29**2,1e-9),
    bbox_fill=area29/np.maximum(np.array([g.envelope.area for g in P29.geometry]),1e-9)))
for key in ("max_overlap_iou","overlap_count","containment_count","higher_score_overlap"):
    base29[key]=0.0
for roi_id in range(6):
    ids=np.flatnonzero(P29.roi_id.to_numpy()==roi_id)
    grp=P29.iloc[ids].reset_index(drop=True); idx=grp.sindex
    local_area=grp.geometry.area.to_numpy()
    scores=pd.to_numeric(grp.score,errors="coerce").fillna(-1).to_numpy()
    for i,g in enumerate(grp.geometry):
        for j in idx.query(g,predicate="intersects"):
            j=int(j)
            if j<=i:continue
            h=grp.geometry.iloc[j];inter=g.intersection(h).area
            if inter<=0:continue
            overlap=inter/max(local_area[i]+local_area[j]-inter,1e-9)
            containment=inter/max(min(local_area[i],local_area[j]),1e-9)
            a,b=int(ids[i]),int(ids[j])
            base29.loc[[a,b],"max_overlap_iou"]=np.maximum(base29.loc[[a,b],"max_overlap_iou"],overlap)
            base29.loc[[a,b],"overlap_count"]+=1
            if containment>=.9:base29.loc[[a,b],"containment_count"]+=1
            if scores[i]<scores[j]:base29.loc[a,"higher_score_overlap"]+=1
            elif scores[j]<scores[i]:base29.loc[b,"higher_score_overlap"]+=1

# Raster features use polygon interior and nearby ring in a bounded window.
# DSM is treated as a SURFACE (not a CHM); local contrasts are physically interpretable.
rgbcols29=["r_mean","g_mean","b_mean","r_std","g_std","b_std",
           "rgb_ring_contrast"]
dsmcols29=["dsm_median","dsm_std","dsm_p90_p10","dsm_ring_contrast",
           "dsm_local_relief"]
raster_columns29=rgbcols29+dsmcols29
raster_values29=np.full((len(P29),len(raster_columns29)),np.nan,dtype=np.float32)
with rasterio.open(RASTER) as rgb, rasterio.open(DSM29) as raw_dsm:
    aligned_dsm=WarpedVRT(raw_dsm,crs=rgb.crs,transform=rgb.transform,
                           width=rgb.width,height=rgb.height,
                           resampling=Resampling.bilinear)
    try:
        for i,g in enumerate(P29.geometry):
            try:
                outer=g.buffer(2.0)
                win=from_bounds(*outer.bounds,transform=rgb.transform).round_offsets().round_lengths()
                win=win.intersection(Window(0,0,rgb.width,rgb.height))
                if win.width<2 or win.height<2:continue
                step=max(1,int(np.ceil(max(win.width,win.height)/192)))
                out_w=max(1,int(np.ceil(win.width/step)));out_h=max(1,int(np.ceil(win.height/step)))
                out_shape=(out_h,out_w)
                rgb_array=rgb.read([1,2,3],window=win,out_shape=(3,out_h,out_w),
                                   resampling=Resampling.average,masked=True)
                dsm_array=aligned_dsm.read(1,window=win,out_shape=out_shape,
                                           resampling=Resampling.average,masked=True)
                tr=rgb.window_transform(win)*rasterio.Affine.scale(win.width/out_w,win.height/out_h)
                inside=geometry_mask([mapping(g)],out_shape=out_shape,transform=tr,invert=True)
                ring=geometry_mask([mapping(outer)],out_shape=out_shape,transform=tr,invert=True)&~inside
                if inside.sum()<3:continue
                c=np.asarray(rgb_array.filled(np.nan),dtype=float)
                z=np.asarray(dsm_array.filled(np.nan),dtype=float)
                means=[];stds=[]
                for channel in range(3):
                    v=c[channel,inside];v=v[np.isfinite(v)]
                    means.append(float(np.mean(v)) if len(v) else np.nan)
                    stds.append(float(np.std(v)) if len(v) else np.nan)
                gray=c.mean(axis=0)
                inside_gray=gray[inside];ring_gray=gray[ring]
                iv=inside_gray[np.isfinite(inside_gray)]
                rv=ring_gray[np.isfinite(ring_gray)]
                rgb_diff=float(np.nanmedian(iv)-np.nanmedian(rv)) if len(iv) and len(rv) else np.nan
                iz=z[inside];iz=iz[np.isfinite(iz)]
                rz=z[ring];rz=rz[np.isfinite(rz)]
                if len(iz)>=3:
                    z_median=float(np.median(iz));z_std=float(np.std(iz))
                    z_range=float(np.percentile(iz,90)-np.percentile(iz,10))
                    z_contrast=float(z_median-np.median(rz)) if len(rz)>=3 else np.nan
                    relief=float(np.percentile(iz,90)-np.percentile(rz,10)) if len(rz)>=3 else np.nan
                else:z_median=z_std=z_range=z_contrast=relief=np.nan
                raster_values29[i,:]=means+stds+[rgb_diff,z_median,z_std,z_range,z_contrast,relief]
            except Exception as error:
                if i<4:print("Raster feature warning",i,str(error))
            if (i+1)%1000==0:print(f"Raster features {i+1}/{len(P29)}")
    finally:aligned_dsm.close()
for k,col in enumerate(raster_columns29):base29[col]=raster_values29[:,k]
# Do not include absolute DSM elevation; no DTM or elevation normalization assumed.
rgb_features29=[c for c in base29.columns if c not in
                (["proposal_id","roi_id"]+dsmcols29)]
dsm_features29=rgb_features29+["dsm_std","dsm_p90_p10","dsm_ring_contrast","dsm_local_relief"]
base29.to_csv(DEST29/"candidate_features_all_rois.csv",index=False)

# Build labels separately for each ROI. Only TRAINING ROI labels can enter fitting.
# One positive proposal per reference crown; others remain unlabeled unless
# confidently identifiable as wrong crown geometry or mostly ground.
labels29=np.full(len(P29),-1,dtype=np.int8)
reference_matrices29={}
for roi_id in range(6):
    ids=np.flatnonzero(P29.roi_id.to_numpy()==roi_id)
    pp=P29.iloc[ids].reset_index(drop=True)
    rr=ref_by_roi29[roi_id]
    ref_index=rr.sindex
    iou=np.zeros((len(rr),len(pp)),dtype=np.float32)
    candidate_covered=np.zeros(len(pp),dtype=np.float32)
    for j,g in enumerate(pp.geometry):
        intersections=[]
        for k in ref_index.query(g,predicate="intersects"):
            k=int(k);h=rr.geometry.iloc[k]
            overlap=g.intersection(h).area
            if overlap<=0:continue
            iou[k,j]=overlap/max(g.area+h.area-overlap,1e-9)
            intersections.append(h.intersection(g))
        if intersections:
            candidate_covered[j]=min(1.,unary_union(intersections).area/max(g.area,1e-9))
    reference_matrices29[roi_id]=iou
    row,col=linear_sum_assignment((iou>=.5).astype(float)*2+iou,maximize=True)
    positive=np.array([j for i,j in zip(row,col) if iou[i,j]>=.5],dtype=int)
    local=np.full(len(pp),-1,dtype=np.int8)
    local[positive]=1
    best_iou=iou.max(axis=0)
    # Poor fit despite mostly annotated-canopy overlap -> conservative negative.
    # Mostly outside exhaustively annotated canopy -> background negative.
    negatives=((best_iou<.25)&(candidate_covered>=.80)) | (candidate_covered<=.20)
    local[negatives&(local!=1)]=0
    labels29[ids]=local
base29["supervision_label"]=labels29
base29.to_csv(DEST29/"candidate_features_and_training_labels.csv",index=False)
print("Label counts:",pd.Series(labels29).value_counts().to_dict())
print("DSM valid proportions:",base29[dsm_features29[-4:]].notna().mean().round(3).to_dict())

# Leave-one-ROI-out: same RF family / parameters, no held-out label fitting.
budgets29=[1.,.9,.8,.7,.6,.5,.4,.3]
iou_thresholds29=[.3,.5,.7]
methods29={"rgb_only":rgb_features29,"rgb_plus_dsm":dsm_features29}
comparison29=[];importance29=[];prediction_rows29=[];size_rows29=[]
sizes29=[(0,20,"under_20"),(20,40,"20_to_40"),(40,80,"40_to_80"),(80,np.inf,"over_80")]
for held_id in range(6):
    train_ids=np.flatnonzero((P29.roi_id.to_numpy()!=held_id)&(labels29>=0))
    test_ids=np.flatnonzero(P29.roi_id.to_numpy()==held_id)
    ytrain=labels29[train_ids]
    assert set(np.unique(ytrain))=={0,1},f"Training fold {held_id} has one label class"
    mat=reference_matrices29[held_id].astype(np.float64)
    nref=mat.shape[0];ntest=len(test_ids)
    test_score29=pd.to_numeric(P29.iloc[test_ids].score,errors="coerce").fillna(-1).to_numpy()
    rankings={"sam3_confidence":np.argsort(-test_score29,kind="stable")}
    for method,features in methods29.items():
        Xtrain=base29.iloc[train_ids][features].replace([np.inf,-np.inf],np.nan)
        Xtest=base29.iloc[test_ids][features].replace([np.inf,-np.inf],np.nan)
        model=make_pipeline(SimpleImputer(strategy="median"),
            RandomForestClassifier(n_estimators=250,min_samples_leaf=4,
            class_weight="balanced_subsample",n_jobs=-1,random_state=42))
        model.fit(Xtrain,ytrain)
        score=model.predict_proba(Xtest)[:,list(model.classes_).index(1)]
        rankings[method]=np.argsort(-score,kind="stable")
        for feature,value in zip(features,model[-1].feature_importances_):
            importance29.append(dict(heldout_roi=held_id,model=method,
                                     feature=feature,importance=float(value)))
        joblib.dump({"model":model,"features":features,"heldout_roi":held_id,
                     "training_rois":[j for j in range(6) if j!=held_id]},
                    DEST29/f"model_{method}_holdout_roi_{held_id}.joblib")
        prediction_rows29.extend(dict(heldout_roi=held_id,proposal_id=int(test_ids[j]),
                                      method=method,score=float(score[j])) for j in range(ntest))
    for method,rank in rankings.items():
        for budget in budgets29:
            k=max(1,int(np.ceil(ntest*budget)));chosen=rank[:k]
            sub=mat[:,chosen]
            for threshold in iou_thresholds29:
                ri,cj=linear_sum_assignment((sub>=threshold).astype(float)*2+sub,maximize=True)
                count=int(np.count_nonzero(sub[ri,cj]>=threshold))
                comparison29.append(dict(heldout_roi=held_id,method=method,
                    retained_fraction=budget,proposals_retained=k,
                    original_proposals=ntest,reduction_pct=100*(1-k/ntest),
                    iou_threshold=threshold,reference_crowns=nref,matched=count,
                    recall=count/nref,reference_matched_prediction_fraction=count/k))
                ref_areas=ref_by_roi29[held_id].geometry.area.to_numpy()
                for lo,hi,label in sizes29:
                    selected_ref=np.flatnonzero((ref_areas>=lo)&(ref_areas<hi))
                    if not len(selected_ref):continue
                    part=sub[selected_ref,:]
                    si,sj=linear_sum_assignment((part>=threshold).astype(float)*2+part,maximize=True)
                    cnt=int(np.count_nonzero(part[si,sj]>=threshold))
                    size_rows29.append(dict(heldout_roi=held_id,method=method,
                         retained_fraction=budget,iou_threshold=threshold,size_class=label,
                         reference_crowns=len(selected_ref),matched=cnt,recall=cnt/len(selected_ref)))
    print(f"ROI {held_id}: train {len(train_ids)} labeled candidates; held-out {ntest} proposals, {nref} crowns")
results29=pd.DataFrame(comparison29)
sizes29_df=pd.DataFrame(size_rows29)
importance29_df=pd.DataFrame(importance29)
results29.to_csv(DEST29/"leave_one_roi_out_budget_recall.csv",index=False)
sizes29_df.to_csv(DEST29/"leave_one_roi_out_crown_size.csv",index=False)
importance29_df.to_csv(DEST29/"feature_importance_by_fold.csv",index=False)
pd.DataFrame(prediction_rows29).to_csv(DEST29/"heldout_proposal_scores.csv",index=False)
pooled29=results29.groupby(["method","retained_fraction","iou_threshold"],as_index=False).agg(
    reference_crowns=("reference_crowns","sum"),matched=("matched","sum"),
    proposals_retained=("proposals_retained","sum"),original_proposals=("original_proposals","sum"))
pooled29["pooled_recall"]=pooled29.matched/pooled29.reference_crowns
pooled29["reduction_pct"]=100*(1-pooled29.proposals_retained/pooled29.original_proposals)
pooled29.to_csv(DEST29/"pooled_leave_one_roi_out_recall.csv",index=False)
print("Pooled IoU >= 0.5 (fixed proposal budgets):")
display(pooled29[(pooled29.iou_threshold==.5)&pooled29.retained_fraction.isin([1.,.7,.6,.5])].round(3))
print("ROI 4 at 40% reduction:")
display(results29[(results29.heldout_roi==4)&(results29.iou_threshold==.5)&
                  (results29.retained_fraction==.6)].round(3))
print("Mean feature importance:")
display(importance29_df.groupby(["model","feature"],as_index=False).importance.mean()
        .sort_values(["model","importance"],ascending=[True,False]).groupby("model").head(10))
print("Saved outputs:",DEST29)
print("CAUTION: Evaluation uses exhaustively digitized visible crowns; unmatched proposals may still be partial or merged canopy objects.")
print("CAUTION: DSM is a surface model, not canopy height above ground without a DTM.")


## 30 — SAM3 crown fine-tuning pilot: export masks and verify official training setup

**Fresh Colab GPU runtime:** run **0R → 2 → 14 → 30**. Unlike the CPU classifier experiments, SAM3 training requires Block 2 and a GPU.

This block creates a **reproducible COCO instance-segmentation dataset** from the manually delineated `Crown_Kristof.shp` polygons and existing Block 22 tile manifests. It reserves whole ROIs **0–3 for training, 4 for validation, and 5 for untouched testing**, avoiding random-image leakage. Each tile is annotated with all intersecting manual crowns, including clipped border instances; pixels outside the annotated ROI are blacked out in exported training imagery to avoid treating unannotated surroundings as background. Dataset includes category `tree crown`, COCO RLE masks, a dataset manifest, and checks. Original imagery and previous outputs are not modified.

The official SAM3 repository supports custom-data training via `sam3/train/train.py` and `Sam3ImageDataset`; its example Roboflow training YAML **defaults to box-only loss**. This block creates a conservative segmentation-enabled pilot configuration by enabling mask targets and loss, and reports the training command; the launcher is **disabled by default** until you inspect the exported samples and configuration. The official training setup is primarily concept/detection oriented; successful training does **not** automatically imply that a geometric-prompt decoder has been fine-tuned. Later we must verify which weights are updated and separately evaluate automatic geometric-grid inference versus prompted masks.

**No DSM used in this first pilot:** we keep the original RGB SAM3 input interface. Fine-tuning outputs, once trained, will need evaluation with ROI 5 held out.


In [ ]:
# BLOCK 30 — Fine-tuning DATASET and official trainer pilot configuration
# Run 0R -> 2 -> 14 -> 30 on GPU runtime. Does NOT launch training by default.
from pathlib import Path
import json, subprocess, sys, shutil
import numpy as np, pandas as pd, geopandas as gpd, rasterio
from rasterio.windows import Window
from rasterio.features import rasterize, geometry_mask
from shapely.geometry import box, mapping
from PIL import Image
try:
    from pycocotools import mask as coco_mask
except ImportError:
    subprocess.run([sys.executable,"-m","pip","-q","install","pycocotools"],check=True)
    from pycocotools import mask as coco_mask
import yaml

DEST30=OUT/"experiments_v30_sam3_finetune"
DATA30=DEST30/"dataset"/"musanga"
SPLIT30={0:"train",1:"train",2:"train",3:"train",4:"valid",5:"test"}
DEST30.mkdir(parents=True,exist_ok=True)
assert "reference_v2" in globals() and "roi_v2" in globals(), "Run Block 14."
assert Path("/content/sam3/sam3/train/train.py").exists(), (
    "Official SAM3 train script missing; check the cloned repository in Block 2.")
with rasterio.open(RASTER) as src:
    crs30=src.crs
ref30=reference_v2.to_crs(crs30)
roi30=roi_v2.to_crs(crs30)
records30=[];summaries30=[]
combined30={split:{'images':[],'annotations':[]} for split in ('train','valid','test')}
image_id30=0;ann_id30=0
for roi_id30,split30 in SPLIT30.items():
    manifest30=OUT/f"experiments_v22_full_roi_8m_{roi_id30}"/"full"/"prompt_manifest.csv"
    assert manifest30.exists(),f"Missing Block 22 tile manifest ROI {roi_id30}"
    tiles30=pd.read_csv(manifest30).drop_duplicates("tile_id").sort_values("tile_id")
    folder30=DATA30/split30
    folder30.mkdir(parents=True,exist_ok=True)
    polygon30=roi30.geometry.iloc[roi_id30]
    crown30=ref30[ref30.geometry.intersects(polygon30)].copy()
    images30=[];annotations30=[]
    with rasterio.open(RASTER) as src:
        for t in tiles30.itertuples():
            win=Window(int(t.col_off),int(t.row_off),int(t.width),int(t.height))
            transform=src.window_transform(win)
            footprint=box(*rasterio.windows.bounds(win,src.transform))
            if not footprint.intersects(polygon30):continue
            rgb=src.read([1,2,3],window=win,boundless=True,fill_value=0).transpose(1,2,0)
            h,w=rgb.shape[:2]
            roi_mask=geometry_mask([mapping(polygon30)],out_shape=(h,w),transform=transform,invert=True)
            rgb[~roi_mask]=0
            filename=f"roi{roi_id30}_tile{int(t.tile_id):04d}.png"
            Image.fromarray(np.clip(rgb,0,255).astype(np.uint8)).save(folder30/filename)
            image_id30+=1
            images30.append(dict(id=image_id30,file_name=filename,width=w,height=h,roi_id=roi_id30))
            crown_intersections=crown30[crown30.geometry.intersects(footprint)]
            for crown in crown_intersections.geometry:
                clipped=crown.intersection(footprint).intersection(polygon30)
                if clipped.is_empty or clipped.area<2:continue
                mask=rasterize([(mapping(clipped),1)],out_shape=(h,w),transform=transform,
                               fill=0,dtype="uint8")
                pixels=int(mask.sum())
                if pixels<16:continue
                yy,xx=np.where(mask>0)
                x0,y0=int(xx.min()),int(yy.min())
                x1,y1=int(xx.max())+1,int(yy.max())+1
                encoded=coco_mask.encode(np.asfortranarray(mask))
                encoded["counts"]=encoded["counts"].decode("ascii")
                ann_id30+=1
                annotations30.append(dict(id=ann_id30,image_id=image_id30,category_id=1,
                    bbox=[x0,y0,x1-x0,y1-y0],area=pixels,
                    segmentation=encoded,iscrowd=0,noun_phrase="tree crown"))
    coco30=dict(info={"description":"Musanga Baego exhaustive visible tree crowns; ROI-separated"},
                licenses=[],images=images30,annotations=annotations30,
                categories=[{"id":1,"name":"tree crown","supercategory":"canopy"}])
    # Accumulate all ROIs belonging to each split before writing annotation JSON.
    combined30[split30]["images"].extend(images30)
    combined30[split30]["annotations"].extend(annotations30)
    summaries30.append(dict(roi_id=roi_id30,split=split30,
                            images=len(images30),mask_instances=len(annotations30)))
    records30.extend(dict(roi_id=roi_id30,split=split30,
                          image=x["file_name"]) for x in images30)
# Write a single complete COCO JSON for each split, not once for each ROI.
for split_name30,items30 in combined30.items():
    image_list30=items30["images"]; ann_list30=items30["annotations"]
    image_ids30={im["id"] for im in image_list30}
    assert len(image_ids30)==len(image_list30), "Duplicate image IDs"
    assert len({a["id"] for a in ann_list30})==len(ann_list30), "Duplicate annotation IDs"
    assert all(a["image_id"] in image_ids30 for a in ann_list30)
    coco_split30=dict(info={"description":"Musanga Baego crowns, ROI-separated"},
                      licenses=[],images=image_list30,annotations=ann_list30,
                      categories=[{"id":1,"name":"tree crown","supercategory":"canopy"}])
    (DATA30/split_name30/"_annotations.coco.json").write_text(json.dumps(coco_split30))
    print("COMBINED",split_name30,len(image_list30),"tiles",len(ann_list30),"masks")
summary30=pd.DataFrame(summaries30)
summary30.to_csv(DEST30/"dataset_manifest.csv",index=False)
display(summary30)
assert summary30.groupby("split").images.sum().min()>0
assert summary30.groupby("split").mask_instances.sum().min()>0
print("COCO mask dataset ready:",DATA30)
# Use official example only as a starting point; specifically enable MASK training.
example30=Path("/content/sam3/sam3/train/configs/roboflow_v100/roboflow_v100_full_ft_100_images.yaml")
if example30.exists():
    config30=yaml.safe_load(example30.read_text())
    config30["paths"]["roboflow_vl_100_root"]=str(DEST30/"dataset")
    config30["paths"]["experiment_log_dir"]=str(DEST30/"training_logs")
    config30["paths"]["bpe_path"]="/content/sam3/sam3/assets/bpe_simple_vocab_16e6.txt.gz"
    config30["roboflow_train"]["supercategory"]="musanga"
    config30["roboflow_train"]["num_images"]=None
    config30["scratch"]["enable_segmentation"]=True
    config30["scratch"]["num_train_workers"]=2
    config30["scratch"]["num_val_workers"]=0
    config30["scratch"]["max_data_epochs"]=3
    config30["scratch"]["target_epoch_size"]=100
    config30["trainer"]["max_epochs"]=3
    config30["trainer"]["skip_saving_ckpts"]=False
    config30["trainer"]["val_epoch_freq"]=1
    config30["launcher"]["gpus_per_node"]=1
    loss30=config30["roboflow_train"]["loss"]["loss_fns_find"]
    if not any(str(item.get("_target_","")).endswith(".Masks") for item in loss30):
        loss30.append({"_target_":"sam3.train.loss.loss_fns.Masks",
                       "focal_alpha":0.25,"focal_gamma":2.,
                       "weight_dict":{"loss_mask":200.,"loss_dice":10.},
                       "compute_aux":False})
    # Validation expects the example's /test annotation folder; use reserved ROI 4 for validation.
    config30["trainer"]["data"]["val"]["dataset"]["img_folder"]=str(DATA30/"valid")+"/"
    config30["trainer"]["data"]["val"]["dataset"]["ann_file"]=str(DATA30/"valid"/"_annotations.coco.json")
    config30["trainer"]["data"]["train"]["dataset"]["img_folder"]=str(DATA30/"train")+"/"
    config30["trainer"]["data"]["train"]["dataset"]["ann_file"]=str(DATA30/"train"/"_annotations.coco.json")
    # Default Roboflow "supercategory" interpolation no longer needed for local paths.
    config30["trainer"]["model"]["enable_segmentation"]=True
    config_file30=DEST30/"sam3_crown_segmentation_pilot.yaml"
    config_file30.write_text('# @package _global_\n'+yaml.safe_dump(config30,sort_keys=False))
    print("Pilot trainer config:",config_file30)
    print("Official trainer command (after checking configuration and sample images):")
    print(f"{SAMPY} /content/sam3/sam3/train/train.py -c '{config_file30}' --use-cluster 0 --num-gpus 1")
else:
    print("Official example configuration not found; dataset exported, trainer config not written.")
# Explicit launch gate. DO NOT run training automatically.
RUN_FINE_TUNING_30=False
if RUN_FINE_TUNING_30:
    assert example30.exists()
    print("Launching official SAM3 trainer with segmentation losses enabled.")
    subprocess.run([SAMPY,"/content/sam3/sam3/train/train.py","-c",str(config_file30),
                    "--use-cluster","0","--num-gpus","1"],check=True)
else:
    print("TRAINING NOT STARTED. Inspect data/config first; set RUN_FINE_TUNING_30=True to launch.")
print("IMPORTANT: ROI 5 is never used for training or configuration selection.")


## 30B — SAM3 fine-tuning smoke test (COCO QA and short GPU run)

**Run 0R → 2 → 14 → 30 → 30B** in a fresh GPU Colab, or just 30B after a successful Block 30 in the same session. This checks COCO masks, creates an isolated training configuration, and runs at most two training batches through the official SAM3 trainer. Logs and possible checkpoints are stored separately under `experiments_v30b_training_smoke`. ROI 5 is not trained on. **Do not start full training until this passes.**

The official SAM3 training example is originally box-detection-oriented. We explicitly check that mask segmentation is enabled and that the loss configuration includes masks. This smoke test does not yet prove that geometric-prompt mask-decoder weights are being updated; assess that separately before drawing scientific conclusions.


In [ ]:
# BLOCK 30B-1 — COCO RLE integrity + tiny segmentation-trainer configuration
from pathlib import Path
import json, shutil, yaml, numpy as np
from PIL import Image
from IPython.display import display
from pycocotools import mask as coco_mask
D30B=OUT/"experiments_v30b_training_smoke"
D30B.mkdir(parents=True,exist_ok=True)
DATA30B=OUT/"experiments_v30_sam3_finetune"/"dataset"/"musanga"
source30b=OUT/"experiments_v30_sam3_finetune"/"sam3_crown_segmentation_pilot.yaml"
assert source30b.exists(), "Run Block 30 first."
for part in ("train","valid","test"):
    folder=DATA30B/part
    obj=json.loads((folder/"_annotations.coco.json").read_text())
    assert obj["images"] and obj["annotations"]
    expected30b={"train":(105,1883),"valid":(33,549),"test":(30,497)}
    nimg30b,nmask30b=expected30b[part]
    assert (len(obj["images"]),len(obj["annotations"]))==(nimg30b,nmask30b), (
        f"{part} got {len(obj['images'])} tiles and {len(obj['annotations'])} masks, "
        f"expected {nimg30b} tiles and {nmask30b} masks. Re-run Block 30."
    )
    assert len({im["id"] for im in obj["images"]})==len(obj["images"])
    assert all(a["image_id"] in {im["id"] for im in obj["images"]} for a in obj["annotations"])
    imgs={im["id"]:im for im in obj["images"]}
    for a in obj["annotations"][:20]:
        im=imgs[a["image_id"]]
        enc=dict(a["segmentation"]);enc["counts"]=enc["counts"].encode("ascii")
        mask=coco_mask.decode(enc)
        assert mask.shape==(im["height"],im["width"])
        assert mask.sum()==int(a["area"]) and mask.any()
    first=obj["images"][0]
    pic=np.array(Image.open(folder/first["file_name"]).convert("RGB"))
    mask_all=np.zeros(pic.shape[:2],dtype=bool)
    for a in obj["annotations"]:
        if a["image_id"]==first["id"]:
            enc=dict(a["segmentation"]);enc["counts"]=enc["counts"].encode("ascii")
            mask_all|=coco_mask.decode(enc).astype(bool)
    pic[mask_all]=(0.5*pic[mask_all]+0.5*np.array([255,0,255])).astype("uint8")
    print(part, len(obj["images"]),"tiles,",len(obj["annotations"]),"instances")
    display(Image.fromarray(pic).resize((450,450)))
cfg30b=yaml.safe_load(source30b.read_text())
assert cfg30b["scratch"]["enable_segmentation"] is True
assert any(str(x.get("_target_","")).endswith(".Masks")
           for x in cfg30b["roboflow_train"]["loss"]["loss_fns_find"])
cfg30b["paths"]["experiment_log_dir"]=str(D30B/"training_logs")
cfg30b["trainer"]["max_epochs"]=1
cfg30b["trainer"]["skip_first_val"]=True
cfg30b["trainer"]["val_epoch_freq"]=999
cfg30b["trainer"]["skip_saving_ckpts"]=False
cfg30b["scratch"]["target_epoch_size"]=2
cfg30b["scratch"]["max_data_epochs"]=1
cfg30b["scratch"]["num_train_workers"]=0
cfg30b["scratch"]["num_val_workers"]=0
# Retain official 1008 resolution: SAM3 ViT RoPE has fixed positional frequencies.
cfg30b["scratch"]["resolution"]=1008
cfg30b["roboflow_train"]["num_images"]=2
# Official trainer requires a top-level launcher; create it if the pilot omitted it.
cfg30b.setdefault("launcher",{})
cfg30b["launcher"].update({
    "num_nodes":1,
    "gpus_per_node":1,
    "experiment_log_dir":str(D30B/"training_logs"),
    "multiprocessing_context":"forkserver",
})
cfg30b.setdefault("submitit",{})
cfg30b["submitit"].setdefault("port_range",[10000,65000])
cfg30b["submitit"]["use_cluster"]=False
cfg30b["roboflow_train"]["supercategory"]="musanga"
cfg30b["submitit"].pop("job_array",None)
cfg30b["trainer"]["logging"]["log_freq"]=1
cfg30b["trainer"]["data"]["train"]["num_workers"]=0
cfg30b["trainer"]["data"]["val"]["num_workers"]=0
# Minimal smoke: freeze the pretrained vision trunk before Hydra creates the optimizer.
# Local wrapper leaves upstream SAM3 code unchanged and avoids gradients through
# sam3.perflib.fused.addmm_act (which explicitly requires no-grad).
builder_path30b=Path("/content/sam3/sam3/train/musanga_frozen_builder.py")
builder_path30b.write_text("""
from sam3.model_builder import build_sam3_image_model

def build_frozen_vision_sam3(**kwargs):
    model = build_sam3_image_model(**kwargs)
    for param in model.backbone.vision_backbone.parameters():
        param.requires_grad_(False)
    # SAM3 fused ViT MLP requires torch.is_grad_enabled() == False.
    # Parameter freezing alone does NOT switch off grad mode during forward.
    import types
    import torch
    original_forward_image = model.backbone.forward_image
    def frozen_forward_image(self, samples):
        with torch.no_grad():
            return original_forward_image(samples)
    model.backbone.forward_image = types.MethodType(frozen_forward_image, model.backbone)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert 0 < trainable < total, (trainable, total)
    assert any(p.requires_grad for p in model.segmentation_head.parameters()), "Segmentation head frozen!"
    print(f"MUSANGA FROZEN VISION (NO_GRAD): {trainable:,} / {total:,} trainable parameters", flush=True)
    return model
""")
cfg30b["trainer"]["model"]["_target_"]="sam3.train.musanga_frozen_builder.build_frozen_vision_sam3"
print("Frozen-vision model builder:",builder_path30b)
# Official Hydra configuration is registered as part of sam3.train.
official_config30b=Path("/content/sam3/sam3/train/configs/musanga_kvo30b_smoke.yaml")
assert official_config30b.parent.exists(),"SAM3 official config folder not found"
official_config30b.write_text('# @package _global_\n'+yaml.safe_dump(cfg30b,sort_keys=False))
# Verify the actual YAML on disk, then test Hydra composition with the SAM3
# interpreter. This catches missing keys BEFORE launching a GPU trainer.
written30b=yaml.safe_load(official_config30b.read_text())
assert set(("launcher","trainer","submitit","scratch")).issubset(written30b), list(written30b)
assert written30b["launcher"]["experiment_log_dir"]
import subprocess
preflight30b=subprocess.run([str(SAMPY),"-c",'''
from hydra import compose, initialize_config_module
from omegaconf import OmegaConf
with initialize_config_module("sam3.train",version_base="1.2"):
    cfg=compose(config_name="configs/musanga_kvo30b_smoke.yaml")
    assert cfg.launcher.gpus_per_node == 1
    assert cfg.launcher.num_nodes == 1
    assert cfg.scratch.resolution == 1008
    assert cfg.launcher.experiment_log_dir
    print("Hydra config PASS; sections:",list(cfg.keys()))
    print("Training output:",cfg.launcher.experiment_log_dir)
'''],capture_output=True,text=True,cwd="/content/sam3")
print(preflight30b.stdout)
if preflight30b.returncode:
    print(preflight30b.stderr[-3000:])
    raise RuntimeError("SAM3 Hydra config preflight failed; do not launch training.")
print("COCO QA PASS; registered Hydra config:",official_config30b)


In [ ]:
# BLOCK 30B-2 — One-epoch / two-batch GPU smoke
import os, subprocess, torch
# The inference-only SAM3 environment may lack trainer dependencies.
check_submitit30b=subprocess.run([str(SAMPY),"-c","import submitit"],capture_output=True)
if check_submitit30b.returncode:
    print("Installing missing official trainer dependency: submitit")
    subprocess.run([str(SAMPY),"-m","pip","install","submitit"],check=True)
assert torch.cuda.is_available(), "Select a Colab GPU runtime"
assert official_config30b.exists(), "Run Block 30B-1 first"
log30b=D30B/"trainer_stdout.log"
cmd30b=[str(SAMPY),"/content/sam3/sam3/train/train.py",
        "-c","configs/musanga_kvo30b_smoke.yaml","--use-cluster","0","--num-gpus","1"]
print("GPU:",torch.cuda.get_device_name(0))
print("Starting short official trainer smoke. Log:",log30b)
with log30b.open("w") as fd:
    proc30b=subprocess.run(cmd30b,cwd="/content/sam3",stdout=fd,stderr=subprocess.STDOUT,check=False)
print("\n".join(log30b.read_text(errors="replace").splitlines()[-100:]))
checkpoints30b=list((D30B/"training_logs").rglob("*.pt"))+list((D30B/"training_logs").rglob("*.pth"))
print("Exit code:",proc30b.returncode)
print("Checkpoint files:",[str(x) for x in checkpoints30b])
assert proc30b.returncode==0, f"SMOKE FAILED: inspect {log30b} before launching longer training"
print("Smoke trainer exited normally. Inspect loss/optimizer steps and checkpoint output.")
